# MemWeave Stage B/C Controlled Reconciliation

**Purpose:** isolate three known implementation differences under one pinned dataset, one deterministic DEV/TEST split, one classifier, one overlap-exclusion rule, and one memory budget.

This notebook does not overwrite historical Stage B or Stage C results. It performs a new controlled reconstruction. It is a confirmatory analysis only if the protocol is frozen before TEST is inspected. The eight combinations are a predeclared $2^3$ factorial: feature scaling (raw/z-score), TF-IDF vocabulary cap (20,000/unlimited), and message filtering (all messages/valid non-empty user+assistant messages).

**Important:** a successful run can identify which switches alter the regenerated result. It cannot prove which historical saved output was produced by which executed notebook state unless original execution artifacts are available.

In [5]:
# Kaggle setup. Internet must be enabled in Notebook Settings.
import os
from pathlib import Path

KAGGLE = Path('/kaggle/working')
KAGGLE.mkdir(parents=True, exist_ok=True)
os.chdir(KAGGLE)
os.environ['CONFIRM_LONGMEMEVAL_TERMS'] = 'YES'   # Change to 'yes' only after reviewing LongMemEval terms.
os.environ['CONFIRM_LOCOMO_TERMS'] = 'YES'        # Change to 'yes' only after reviewing LoCoMo terms.
os.environ['MEMWEAVE_OUT'] = str(KAGGLE / 'memweave_followup_outputs')
print('Working directory:', Path.cwd())
print('Output directory:', os.environ['MEMWEAVE_OUT'])


Working directory: /kaggle/working
Output directory: /kaggle/working/memweave_followup_outputs


In [6]:
import numpy
import pandas
import scipy
import sklearn
import requests
import huggingface_hub

print("All packages imported successfully!")

All packages imported successfully!


In [7]:
%pip install -q huggingface_hub requests numpy pandas scipy scikit-learn
print('Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.')


Note: you may need to restart the kernel to use updated packages.
Dependencies are ready. If Kaggle displays a package-restart notice, use Session → Restart session, then run all cells from the top.


## 1. Resolve and hash immutable datasets

LongMemEval is resolved to an immutable Hugging Face commit before download. LoCoMo is resolved to the Git commit containing `data/locomo10.json`. If you supply a local LongMemEval file, set `LONGMEMEVAL_REVISION` and `LONGMEMEVAL_SOURCE_URL` yourself; the file SHA-256 will still be recorded, but the revision remains explicitly unknown if not supplied.

In [8]:
import os, re, json, hashlib, time, datetime, platform, sys, subprocess, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def resolve_longmemeval():
    local = os.environ.get("LONGMEMEVAL_PATH")
    if local:
        return os.path.abspath(local), os.environ.get("LONGMEMEVAL_REVISION", "LOCAL_REVISION_NOT_PROVIDED"), os.environ.get("LONGMEMEVAL_SOURCE_URL", "local file; source URL not provided")
    repo_id = "xiaowu0162/longmemeval-cleaned"
    requested_revision = os.environ.get("LONGMEMEVAL_REVISION", "main")
    info = HfApi().dataset_info(repo_id, revision=requested_revision)
    commit_sha = info.sha
    path = hf_hub_download(
        repo_id=repo_id,
        filename="longmemeval_s_cleaned.json",
        repo_type="dataset",
        revision=commit_sha
    )
    url = f"https://huggingface.co/datasets/{repo_id}/resolve/{commit_sha}/longmemeval_s_cleaned.json"
    return path, commit_sha, url

LME_PATH, LME_REVISION, LME_URL = resolve_longmemeval()
LME_SHA256 = sha256_file(LME_PATH)

api_url = "https://api.github.com/repos/snap-research/locomo/commits?path=data/locomo10.json&per_page=1"
api_response = requests.get(api_url, timeout=60)
api_response.raise_for_status()
LOCOMO_COMMIT = api_response.json()[0]["sha"]
LOCOMO_URL = f"https://raw.githubusercontent.com/snap-research/locomo/{LOCOMO_COMMIT}/data/locomo10.json"
os.makedirs("data", exist_ok=True)
LOCOMO_PATH = os.path.join("data", "locomo10.json")
urllib.request.urlretrieve(LOCOMO_URL, LOCOMO_PATH)
LOCOMO_SHA256 = sha256_file(LOCOMO_PATH)

print("LongMemEval revision:", LME_REVISION)
print("LongMemEval SHA-256:", LME_SHA256)
print("LoCoMo commit:", LOCOMO_COMMIT)
print("LoCoMo SHA-256:", LOCOMO_SHA256)

if os.environ.get("CONFIRM_LONGMEMEVAL_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LongMemEval terms, then set CONFIRM_LONGMEMEVAL_TERMS=yes.")
if os.environ.get("CONFIRM_LOCOMO_TERMS", "").lower() != "yes":
    raise RuntimeError("Review LoCoMo terms, then set CONFIRM_LOCOMO_TERMS=yes.")

DATA_PROVENANCE = {
    "longmemeval": {"source_url": LME_URL, "revision": LME_REVISION, "sha256": LME_SHA256},
    "locomo": {"source_url": LOCOMO_URL, "revision": LOCOMO_COMMIT, "sha256": LOCOMO_SHA256}
}


longmemeval_s_cleaned.json: reconstructing file:   0%|          |  0.00B /  277MB            

longmemeval_s_cleaned.json: downloading bytes:           |  0.00B            

LongMemEval revision: 98d7416c24c778c2fee6e6f3006e7a073259d48f
LongMemEval SHA-256: d6f21ea9d60a0d56f34a05b609c79c88a451d2ae03597821ea3d5a9678c3a442
LoCoMo commit: cbfbc1dba6bc53d00625212a0f22d55ffee7c1fc
LoCoMo SHA-256: 79fa87e90f04081343b8c8debecb80a9a6842b76a7aa537dc9fdf651ea698ff4


## 2. Rebuild the canonical LongMemEval representation

This loads the core preprocessing and feature-building cells from the repository's final executed notebook at a pinned Git commit. It reuses the original parser, TF-IDF representation, `same_pipeline` pool, and stratified 235/235 split. The original raw JSON remains in memory only so the six role-aware features can be reconstructed.

In [9]:
# Pinned source notebook commit: the canonical executed notebook at the time this
# follow-up was authored. Update only after auditing a deliberate source change.
SOURCE_NOTEBOOK_COMMIT = "d642215226fa2a9204bb6d64c4ff0f6447920274"
SOURCE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/tishaaaaaaaaa5-sketch/memweavee/"
    + SOURCE_NOTEBOOK_COMMIT
    + "/MemWeave/notebooks/04_MemWeave_Final_Executed.ipynb"
)
source_bytes = requests.get(SOURCE_NOTEBOOK_URL, timeout=60)
source_bytes.raise_for_status()
source_nb = source_bytes.json()

def source_cell(index):
    src = source_nb["cells"][index]["source"]
    return "".join(src) if isinstance(src, list) else str(src)

# Original setup/configuration.
exec(compile(source_cell(1), "canonical_cell_1", "exec"), globals())

# Point the original data-preparation cell at the pinned local file.
LME_PATH = LME_PATH
prep_source = source_cell(4).replace("del raw\n", "")
exec(compile(prep_source, "canonical_cell_4", "exec"), globals())

# Original base features, pools and deterministic DEV/TEST split.
exec(compile(source_cell(6), "canonical_cell_6", "exec"), globals())

POOL = "same_pipeline"
BASE_IDX = [0, 1, 7, 8, 9]
ROLE_IDX = list(range(10, 16))
FULL_IDX = BASE_IDX + ROLE_IDX
C_FROZEN = 0.3
BUDGET_FRACTION = 0.10

assert len(DEV) == 235 and len(TEST) == 235, (
    f"Expected canonical 235/235 split; got {len(DEV)}/{len(TEST)}. "
    "Do not compare this run directly with the historical aggregate."
)
assert not set(DEV).intersection(TEST)
print("Split verified:", len(DEV), "DEV /", len(TEST), "TEST")

config: {'budget': 0.1, 'budget_sweep': [0.05, 0.1, 0.2, 0.3], 'test_frac': 0.5, 'cv_folds': 5, 'C_grid': [0.03, 0.3, 3.0], 'equiv_margin': 0.03, 'alpha': 0.05, 'n_boot': 4000, 'topk': 3, 'n_random_reps': 50, 'primary_pool': 'same_pipeline', 'max_questions': None}
questions loaded: 500
unique sessions: 18464  | built in 6s
tf-idf matrix: (18464, 524288) | 22s elapsed
{'same_pipeline': 470, 'all': 470, 'filler': 470} | valid in all pools: 470
DEV: 235  TEST (locked): 235
mean pool size: {'same_pipeline': np.float64(23.4), 'all': np.float64(47.7), 'filler': np.float64(26.2)}
Split verified: 235 DEV / 235 TEST


## Controlled feature reconstruction

The two historical endpoints are:
- Stage B-like: raw features, 20,000-feature cap, all messages.
- Stage C-like: within-question z-score, uncapped vocabulary, filtered non-empty user/assistant messages.

All other pipeline settings are held constant. The message-filter switch also uses the historical role handling: Stage B compares literal role labels; Stage C lowercases role labels.

In [10]:
# Reconciliation configuration — freeze before TEST evaluation.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest
import hashlib, json, itertools, platform
from pathlib import Path

ROLE_FEATURE_NAMES = [
    "user_max_question_similarity",
    "user_top3_mean_question_similarity",
    "assistant_max_question_similarity",
    "assistant_top3_mean_question_similarity",
    "log_user_message_count",
    "log_assistant_message_count",
]
ROW_MAPS = {
    qi: {int(row["h"]): row for row in qrecs[qi]["rows"]}
    for qi in DEV + TEST
}

def construct_role_features(qi, *, scale, vocab_cap, filter_messages):
    q = qrecs[qi]
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))
    hashes = Q[POOL][qi]["h"]
    n = len(hashes)
    texts, roles, candidate_ids = [], [], []
    user_counts = np.zeros(n, dtype=float)
    assistant_counts = np.zeros(n, dtype=float)

    for ci, h in enumerate(hashes):
        row = ROW_MAPS[qi][int(h)]
        session = sessions[str(row["sid"])]
        for message in session:
            raw_role = str(message.get("role", ""))
            content = str(message.get("content", ""))
            role = raw_role.lower() if filter_messages else raw_role
            if filter_messages and (role not in ("user", "assistant") or not content.strip()):
                continue
            texts.append(content.strip() if filter_messages else content)
            roles.append(role)
            candidate_ids.append(ci)
            if role == "user":
                user_counts[ci] += 1
            elif role == "assistant":
                assistant_counts[ci] += 1

    features = np.zeros((n, 6), dtype=np.float64)
    if texts:
        kwargs = dict(lowercase=True, stop_words="english",
                      ngram_range=(1, 2), sublinear_tf=True)
        if vocab_cap is not None:
            kwargs["max_features"] = int(vocab_cap)
        vectorizer = TfidfVectorizer(**kwargs)
        matrix = vectorizer.fit_transform([str(q["question"])] + texts)
        similarities = (matrix[1:] @ matrix[0].T).toarray().ravel()
        grouped = {(i, role): [] for i in range(n) for role in ("user", "assistant")}
        for similarity, ci, role in zip(similarities, candidate_ids, roles):
            if role in ("user", "assistant"):
                grouped[(ci, role)].append(float(similarity))
        for ci in range(n):
            user = sorted(grouped[(ci, "user")], reverse=True)
            assistant = sorted(grouped[(ci, "assistant")], reverse=True)
            features[ci, 0] = max(user) if user else 0.0
            features[ci, 1] = np.mean(user[:3]) if user else 0.0
            features[ci, 2] = max(assistant) if assistant else 0.0
            features[ci, 3] = np.mean(assistant[:3]) if assistant else 0.0

    features[:, 4] = np.log1p(user_counts)
    features[:, 5] = np.log1p(assistant_counts)
    if scale == "zscore":
        features = (features - features.mean(0, keepdims=True)) / (
            features.std(0, keepdims=True) + 1e-6
        )
    elif scale != "raw":
        raise ValueError("scale must be raw or zscore")
    return features.astype(np.float32)

# Build the eight factorial conditions. No TEST labels are used to construct features.
FACTORIAL = list(itertools.product(
    ["raw", "zscore"], [20000, None], [False, True]
))
VARIANT_CONFIGS = {
    f"scale={scale}|cap={cap if cap is not None else 'none'}|filtered={filtered}": {
        "scale": scale, "vocab_cap": cap, "filter_messages": filtered
    }
    for scale, cap, filtered in FACTORIAL
}
ROLE_FEATURES = {}
FEATURE_DIAGNOSTICS = []
for name, cfg in VARIANT_CONFIGS.items():
    ROLE_FEATURES[name] = {}
    for qi in DEV + TEST:
        values = construct_role_features(qi, **cfg)
        ROLE_FEATURES[name][qi] = values
        FEATURE_DIAGNOSTICS.append({
            "variant": name, "question_id_sha256_20": hashlib.sha256(str(qi).encode("utf-8")).hexdigest()[:20],
            "n_candidates": int(values.shape[0]),
            "feature_mean_abs": float(np.abs(values).mean()),
            "feature_std_mean": float(values.std(axis=0).mean()),
            "nonzero_fraction": float(np.count_nonzero(values) / values.size)
        })
print("Built", len(VARIANT_CONFIGS), "conditions for", len(DEV)+len(TEST), "questions.")
display(pd.DataFrame(FEATURE_DIAGNOSTICS).groupby("variant").agg(
    mean_abs=("feature_mean_abs","mean"),
    mean_feature_std=("feature_std_mean","mean"),
    mean_nonzero_fraction=("nonzero_fraction","mean")
).reset_index())

Built 8 conditions for 470 questions.


,variant,mean_abs,mean_feature_std,mean_nonzero_fraction
0,scale=raw|cap=20000|filtered=False,0.657202,0.029371,0.662308
1,scale=raw|cap=20000|filtered=True,0.657202,0.029371,0.662308
2,scale=raw|cap=none|filtered=False,0.655620,0.026335,0.662308
3,scale=raw|cap=none|filtered=True,0.655620,0.026335,0.662308
4,scale=zscore|cap=20000|filtered=False,0.490121,0.821238,0.959220
5,scale=zscore|cap=20000|filtered=True,0.490121,0.821238,0.959220
6,scale=zscore|cap=none|filtered=False,0.489931,0.821231,0.959220
7,scale=zscore|cap=none|filtered=True,0.489931,0.821231,0.959220


## Fit and evaluate under identical conditions

The classifier is trained on DEV only. Candidate hashes appearing in TEST are excluded from DEV training exactly as in the Stage C pipeline. Every method uses the same candidate order, stable score sorting, 10% word-count proxy budget, and held-out questions. The budget remains a word-count proxy—not exact tokenizer accounting.

In [11]:
# Freeze all common pipeline settings.
BASE_IDX = [0, 1, 7, 8, 9]
C_FROZEN = 0.3
BUDGET_FRACTION = 0.10
TEST_HASHES = set().union(*[qrecs[q]["all_hashes"] for q in TEST])
BASE_MATRICES = {q: Q[POOL][q]["F"][:, BASE_IDX].copy() for q in DEV + TEST}

def train_and_score(feature_map):
    xs, ys, hashes = [], [], []
    for qi in DEV:
        xs.append(np.column_stack([BASE_MATRICES[qi], feature_map[qi]]))
        ys.append(np.asarray(Q[POOL][qi]["gold"], dtype=int))
        hashes.append(np.asarray(Q[POOL][qi]["h"], dtype=np.int64))
    x_train, y_train, h_train = np.vstack(xs), np.concatenate(ys), np.concatenate(hashes)
    keep = ~np.isin(h_train, np.fromiter(TEST_HASHES, dtype=np.int64))
    if len(np.unique(y_train[keep])) != 2:
        raise RuntimeError("Training set after overlap exclusion does not contain both classes.")
    model = LogisticRegression(C=C_FROZEN, class_weight="balanced", max_iter=500)
    model.fit(x_train[keep], y_train[keep])
    scores = {}
    for qi in TEST:
        x = np.column_stack([BASE_MATRICES[qi], feature_map[qi]])
        scores[qi] = model.predict_proba(x)[:, 1]
    return model, scores, int(keep.sum()), int(len(keep))

def train_baseline():
    xs, ys, hashes = [], [], []
    for qi in DEV:
        xs.append(BASE_MATRICES[qi])
        ys.append(np.asarray(Q[POOL][qi]["gold"], dtype=int))
        hashes.append(np.asarray(Q[POOL][qi]["h"], dtype=np.int64))
    x_train, y_train, h_train = np.vstack(xs), np.concatenate(ys), np.concatenate(hashes)
    keep = ~np.isin(h_train, np.fromiter(TEST_HASHES, dtype=np.int64))
    model = LogisticRegression(C=C_FROZEN, class_weight="balanced", max_iter=500)
    model.fit(x_train[keep], y_train[keep])
    scores = {qi: model.predict_proba(BASE_MATRICES[qi])[:, 1] for qi in TEST}
    return model, scores, int(keep.sum()), int(len(keep))

def select_budgeted(scores, costs, budget=BUDGET_FRACTION):
    costs = np.asarray(costs, dtype=float)
    cap = budget * float(costs.sum())
    chosen, used = [], 0.0
    for ix in np.argsort(-np.asarray(scores), kind="stable"):
        cost = float(costs[ix])
        if used + cost <= cap:
            chosen.append(int(ix))
            used += cost
    return np.asarray(chosen, dtype=int), used, cap

def evaluate_score_map(score_map):
    per_question = {}
    for qi in TEST:
        d = Q[POOL][qi]
        scores = np.asarray(score_map[qi], dtype=float)
        gold = np.asarray(d["gold"], dtype=bool)
        if not gold.any():
            raise RuntimeError(f"No gold candidates for test question {qi}")
        order = np.argsort(-scores, kind="stable")
        rank_lookup = np.empty(len(order), dtype=int)
        rank_lookup[order] = np.arange(1, len(order)+1)
        rank = int(rank_lookup[np.flatnonzero(gold)].min())
        selected, used, cap = select_budgeted(scores, d["cost"])
        per_question[qi] = {
            "rank": rank,
            "recall": float(gold[selected].sum()/gold.sum()),
            "used_word_cost": float(used),
            "budget_word_cap": float(cap),
            "budget_utilization": float(used/cap) if cap else 0.0,
            "selected_candidate_indices": selected.tolist()
        }
    return per_question

baseline_model, baseline_scores, baseline_kept, baseline_total = train_baseline()
baseline_metrics = evaluate_score_map(baseline_scores)
RUNS = {}
FIT_ROWS = []
for name, feature_map in ROLE_FEATURES.items():
    model, scores, kept, total = train_and_score(feature_map)
    metrics = evaluate_score_map(scores)
    RUNS[name] = {"model": model, "scores": scores, "metrics": metrics}
    FIT_ROWS.append({"variant": name, "train_rows_kept": kept, "train_rows_total": total})
print("Baseline train rows:", baseline_kept, "/", baseline_total)
display(pd.DataFrame(FIT_ROWS))

Baseline train rows: 4216 / 5523


,variant,train_rows_kept,train_rows_total
0,scale=raw|cap=20000|filtered=False,4216,5523
1,scale=raw|cap=20000|filtered=True,4216,5523
2,scale=raw|cap=none|filtered=False,4216,5523
3,scale=raw|cap=none|filtered=True,4216,5523
4,scale=zscore|cap=20000|filtered=False,4216,5523
5,scale=zscore|cap=20000|filtered=True,4216,5523
6,scale=zscore|cap=none|filtered=False,4216,5523
7,scale=zscore|cap=none|filtered=True,4216,5523


## Paired inference and factorial effects

The primary outcome is per-question budgeted evidence recall. The bootstrap below resamples questions, matching the historical evaluation's paired question-level unit. If multiple questions share an independent conversation/user cluster, replace this with a cluster bootstrap before confirmatory claims. Holm adjustment is applied to the three main-effect tests. The eight-way comparisons are descriptive unless multiplicity is addressed.

In [12]:
# Full factorial contrasts computed on paired per-question outcomes.
# Effects are average high-minus-low contrasts (or interaction contrasts) over all other factors.
variant_names = list(VARIANT_CONFIGS)
question_outcomes = np.vstack([
    np.array([RUNS[name]["metrics"][q]["recall"] for q in TEST], dtype=float)
    for name in variant_names
])
factor_codes = []
for name in variant_names:
    cfg = VARIANT_CONFIGS[name]
    factor_codes.append([
        1 if cfg["scale"] == "zscore" else -1,
        1 if cfg["vocab_cap"] is None else -1,
        1 if cfg["filter_messages"] else -1
    ])
factor_codes = np.asarray(factor_codes, dtype=float)

terms = {
    "scale": factor_codes[:,0],
    "vocab_cap": factor_codes[:,1],
    "filter_messages": factor_codes[:,2],
    "scale_x_vocab_cap": factor_codes[:,0]*factor_codes[:,1],
    "scale_x_filter_messages": factor_codes[:,0]*factor_codes[:,2],
    "vocab_cap_x_filter_messages": factor_codes[:,1]*factor_codes[:,2],
    "scale_x_vocab_cap_x_filter_messages": factor_codes[:,0]*factor_codes[:,1]*factor_codes[:,2],
}
def factorial_contrast(outcomes, code):
    # Conventional 2^k factorial effect: twice the coded mean contrast.
    return float(2.0 * np.mean(code[:,None] * outcomes))

rng = np.random.default_rng(20261128)
N_BOOT = 20000
effect_rows = []
for term, code in terms.items():
    observed_by_q = 2.0 * np.mean(code[:,None] * question_outcomes, axis=0)
    point = float(observed_by_q.mean())
    draws = np.empty(N_BOOT)
    for start in range(0, N_BOOT, 1000):
        end = min(start+1000, N_BOOT)
        ix = rng.integers(0, len(TEST), size=(end-start, len(TEST)))
        draws[start:end] = observed_by_q[ix].mean(axis=1)
    effect_rows.append({
        "factorial_term": term,
        "effect_recall_points": point,
        "ci95_low": float(np.quantile(draws,0.025)),
        "ci95_high": float(np.quantile(draws,0.975))
    })
factorial_effects = pd.DataFrame(effect_rows)
display(factorial_effects)


,factorial_term,effect_recall_points,ci95_low,ci95_high
0,scale,0.126809,0.086596,0.169007
1,vocab_cap,-0.015603,-0.026241,-0.006738
2,filter_messages,0.000000,0.000000,0.000000
3,scale_x_vocab_cap,0.009220,-0.000355,0.019504
4,scale_x_filter_messages,0.000000,0.000000,0.000000
5,vocab_cap_x_filter_messages,0.000000,0.000000,0.000000
6,scale_x_vocab_cap_x_filter_messages,0.000000,0.000000,0.000000


## Export reproducible artifacts

Exports contain hashed question identifiers and candidate indices, not cleartext question text, answers, or conversation content. Candidate indices are local to each question and only meaningful alongside the exact pinned input and split. Review privacy and dataset terms before publishing outputs.

In [13]:
# Save a run manifest and privacy-minimised result tables.
OUT = Path(os.environ.get("MEMWEAVE_OUT", "memweave_reconciliation_outputs"))
OUT.mkdir(parents=True, exist_ok=True)

def q_hash(qi):
    return hashlib.sha256(str(qi).encode("utf-8")).hexdigest()[:20]

per_question_rows = []
selection_rows = []
for qi in TEST:
    qid = q_hash(qi)
    b = baseline_metrics[qi]
    per_question_rows.append({
        "question_id_sha256_20": qid,
        "baseline_rank": b["rank"],
        "baseline_budgeted_evidence_recall": b["recall"],
        "baseline_budget_utilization": b["budget_utilization"]
    })
    for name, run in RUNS.items():
        m = run["metrics"][qi]
        per_question_rows.append({
            "question_id_sha256_20": qid,
            "variant": name,
            "rank": m["rank"],
            "budgeted_evidence_recall": m["recall"],
            "budget_utilization": m["budget_utilization"]
        })
        selection_rows.append({
            "question_id_sha256_20": qid,
            "variant": name,
            "selected_candidate_indices_json": json.dumps(m["selected_candidate_indices"]),
            "used_word_cost": m["used_word_cost"],
            "budget_word_cap": m["budget_word_cap"]
        })


# Candidate-level, privacy-minimised trace for offline MFAR adapter.
# This records the benchmark's retention decision only; it is NOT a retrieval or answer trace.
candidate_trace_rows = []
for qi in TEST:
    qid = q_hash(qi)
    question_text = str(qrecs[qi]["question"])
    query_content_sha256 = hashlib.sha256(question_text.encode("utf-8")).hexdigest()
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))
    d = Q[POOL][qi]
    candidate_ids = list(map(int, d["h"]))
    variant_selections = {
        "baseline": set(baseline_metrics[qi]["selected_candidate_indices"])
    }
    for variant_name, run in RUNS.items():
        variant_selections[variant_name] = set(
            run["metrics"][qi]["selected_candidate_indices"]
        )

    for variant_name, selected_indices in variant_selections.items():
        for candidate_index, candidate_hash in enumerate(candidate_ids):
            row = ROW_MAPS[qi][candidate_hash]
            session_id = str(row["sid"])
            session = sessions[session_id]
            canonical_session = json.dumps(
                session, sort_keys=True, ensure_ascii=False, separators=(",", ":")
            )
            source_id = "src_" + hashlib.sha256(
                f"{LME_REVISION}:{session_id}".encode("utf-8")
            ).hexdigest()[:32]
            candidate_trace_rows.append({
                "question_id_sha256_20": qid,
                "query_content_sha256": query_content_sha256,
                "variant": variant_name,
                "candidate_index": int(candidate_index),
                "source_id": source_id,
                "content_sha256": hashlib.sha256(
                    canonical_session.encode("utf-8")
                ).hexdigest(),
                "relevance_grade": (
                    "gold" if bool(d["gold"][candidate_index]) else "irrelevant"
                ),
                "word_cost": float(d["cost"][candidate_index]),
                "selected": bool(candidate_index in selected_indices)
            })

candidate_trace_path = OUT / "stage_bc_candidate_trace.csv"
pd.DataFrame(candidate_trace_rows).to_csv(candidate_trace_path, index=False)
print("Saved privacy-minimised candidate trace:", candidate_trace_path)

summary.to_csv(OUT/"stage_bc_factorial_summary.csv", index=False)
pd.DataFrame(per_question_rows).to_csv(OUT/"stage_bc_per_question_results.csv", index=False)
pd.DataFrame(paired_rows).assign(question_id_sha256_20=lambda d: d["qi"].map(q_hash)).drop(columns=["qi"]).to_csv(
    OUT/"stage_bc_paired_question_results.csv", index=False)
pd.DataFrame(selection_rows).to_csv(OUT/"stage_bc_selection_trace.csv", index=False)
pd.DataFrame(FEATURE_DIAGNOSTICS).to_csv(OUT/"stage_bc_feature_diagnostics.csv", index=False)
factorial_effects.to_csv(OUT/"stage_bc_factorial_effects.csv", index=False)

manifest = {
    "experiment": "MemWeave Stage B/C controlled factorial reconstruction",
    "status": "new controlled reconstruction; not a retroactive proof of historical execution",
    "candidate_trace_semantics": "Session-level candidate and budgeted-retention selection only; not a retrieval trace, persistent-memory provenance log, or answer trace.",
    "source_notebook_commit": SOURCE_NOTEBOOK_COMMIT,
    "longmemeval_revision": LME_REVISION,
    "longmemeval_sha256": LME_SHA256,
    "pool": POOL, "dev_questions": len(DEV), "test_questions": len(TEST),
    "classifier": "sklearn LogisticRegression(C=0.3,class_weight=balanced,max_iter=500)",
    "test_hash_overlap_exclusion": True,
    "budget_fraction": BUDGET_FRACTION,
    "budget_cost_unit": "word-count proxy from canonical notebook; not exact tokenizer tokens",
    "factors": ["scale: raw/zscore", "vocab_cap: 20000/None", "filter_messages: False/True"],
    "variant_count": len(VARIANT_CONFIGS),
    "bootstrap_unit": "question; cluster bootstrap required if questions share independent conversation clusters",
    "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "outputs": ["stage_bc_factorial_summary.csv","stage_bc_per_question_results.csv","stage_bc_paired_question_results.csv",
                "stage_bc_selection_trace.csv","stage_bc_feature_diagnostics.csv",
                "stage_bc_factorial_effects.csv","stage_bc_candidate_trace.csv"]
}
with open(OUT/"stage_bc_reconciliation_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)
print("Saved reconciliation outputs to:", OUT.resolve())
print(json.dumps(manifest, indent=2))

Saved privacy-minimised candidate trace: /kaggle/working/memweave_followup_outputs/stage_bc_candidate_trace.csv


NameError: name 'summary' is not defined

## Interpretation and acceptance guardrails

- Stage B-like and Stage C-like corners are regenerated under the same controlled pipeline; they are not guaranteed to reproduce historical aggregates because saved historical execution lineage may be incomplete.
- Factorial main effects quantify sensitivity to implementation choices in this reconstruction. Interactions matter; inspect all eight cells.
- If the Stage B-like and Stage C-like endpoints remain different, the switches explain part of the controlled difference, not necessarily the entire historical gap.
- Do not select the variant with the highest TEST score as the new production method. Use DEV/CV for selection, freeze the method, and reserve a genuinely untouched external test set for confirmation.
- The current budget is the canonical word-count proxy. Exact tokenizer-based budgets and independently checked evidence mappings remain separate required work.
- This notebook evaluates evidence selection/recall, not downstream answer accuracy.

In [14]:
!pip install jsonschema

In [18]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    if "assemble_mfar_retention_traces.py" in files:
        print("SCRIPT FOUND:", os.path.join(root, "assemble_mfar_retention_traces.py"))

In [19]:
!ls -lah /kaggle/working

total 20K
drwxr-xr-x 5 root root 4.0K Nov 26 15:34 .
drwxr-xr-x 5 root root 4.0K Nov 26 15:28 ..
drwxr-xr-x 2 root root 4.0K Nov 26 15:33 data
drwxr-xr-x 2 root root 4.0K Nov 26 15:40 memweave_followup_outputs
drwxr-xr-x 2 root root 4.0K Nov 26 15:29 .virtual_documents


In [20]:
!find /kaggle/input -name "assemble_mfar_retention_traces.py" -o -name "memweavee.zip"

In [21]:
!unzip -l /kaggle/working/memweave_followup_outputs.zip | head -40

unzip:  cannot find or open /kaggle/working/memweave_followup_outputs.zip, /kaggle/working/memweave_followup_outputs.zip.zip or /kaggle/working/memweave_followup_outputs.zip.ZIP.


In [ ]:
!find /kaggle -name "stage_bc_candidate_trace.csv"

In [22]:
!git clone --branch research/memweave-followup-release https://github.com/tishaaaaaaaaa5-sketch/memweavee.git /kaggle/working/memweavee

Cloning into '/kaggle/working/memweavee'...
remote: Enumerating objects: 1121, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 1121 (delta 47), reused 0 (delta 0), pack-reused 960 (from 2)
Receiving objects: 100% (1121/1121), 1.75 MiB | 2.25 MiB/s, done.
Resolving deltas: 100% (629/629), done.


In [23]:
!ls /kaggle/working/memweavee/MemWeave/scripts/

assemble_mfar_retention_traces.py  validate_mfar_runtime_events.py
audit_mfar_retention_traces.py	   validate_mfar_trace_integrity.py
mfar_runtime_event_writer.py	   validate_mfar_trace.py
README.md			   validate_release.py


In [24]:
!python /kaggle/working/memweavee/MemWeave/scripts/assemble_mfar_retention_traces.py \
    --candidates /kaggle/working/memweave_followup_outputs/stage_bc_candidate_trace.csv \
    --manifest /kaggle/working/memweavee/MemWeave/results/stage_bc_2026-11-26/stage_bc_reconciliation_manifest_audited.json \
    --output /kaggle/working/mfar_retention_traces.jsonl

Assembled 2115 traces across 235 test questions.
All traces passed JSON Schema Draft 2020-12 validation.
Observability boundary: retention selection is represented; retrieval and answer stages remain unobserved.
Output: /kaggle/working/mfar_retention_traces.jsonl


In [25]:
with open("/kaggle/working/mfar_retention_traces.jsonl", encoding="utf-8") as f:
    print("Total traces:", sum(1 for _ in f))

Total traces: 2115


In [27]:
import os
os.environ["MFAR_TRACE_ENABLED"] = "1"

In [28]:
from pathlib import Path

print("Current directory:", Path.cwd())

for base in [Path.cwd(), Path("/kaggle/working")]:
    if base.exists():
        print(f"\nSearching: {base}")
        for pattern in [
            "mfar_runtime_events_*.jsonl",
            "mfar_runtime_trace_manifest_*.json"
        ]:
            for f in base.rglob(pattern):
                print(f)

Current directory: /kaggle/working

Searching: /kaggle/working

Searching: /kaggle/working


In [29]:
import os
from pathlib import Path

print("Tracing setting:", os.environ.get("MFAR_TRACE_ENABLED"))
print("Notebook working directory:", Path.cwd())

for base in [Path.cwd(), Path("/kaggle/working")]:
    if base.exists():
        for pattern in [
            "mfar_runtime_events_*.jsonl",
            "mfar_runtime_trace_manifest_*.json"
        ]:
            for f in base.rglob(pattern):
                print("FOUND:", f)

print("Diagnostic complete.")

Tracing setting: 1
Notebook working directory: /kaggle/working
Diagnostic complete.


In [31]:
from pathlib import Path

print("=== MFAR objects loaded in notebook ===")
matches = [
    name for name in globals()
    if "mfar" in name.lower() or "trace" in name.lower()
]
print(matches if matches else "No MFAR/trace objects found")

print("\n=== Relevant output directories ===")
for path in [
    Path("/kaggle/working/v2_1"),
    Path.cwd() / "v2_1",
    Path("/kaggle/working")
]:
    print(f"\n{path} | exists: {path.exists()}")
    if path.exists() and path.is_dir():
        print("Contents:", [p.name for p in list(path.iterdir())[:30]])

print("\nDiagnostic complete.")

=== MFAR objects loaded in notebook ===
['candidate_trace_rows', 'candidate_trace_path']

=== Relevant output directories ===

/kaggle/working/v2_1 | exists: False

/kaggle/working/v2_1 | exists: False

/kaggle/working | exists: True
Contents: ['memweave_followup_outputs', '.virtual_documents', 'mfar_retention_traces.jsonl', 'memweavee', 'data']

Diagnostic complete.


In [32]:
from pathlib import Path
import json
from collections import Counter

trace_path = Path("/kaggle/working/mfar_retention_traces.jsonl")

print("=== RETENTION TRACE VALIDATION ===")
print("File exists:", trace_path.exists())

if trace_path.exists():
    print("File size (MB):", round(trace_path.stat().st_size / (1024**2), 2))

    total = 0
    malformed = 0
    event_counts = Counter()
    sample_records = []

    with trace_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
                total += 1

                if len(sample_records) < 3:
                    sample_records.append(record)

                for key in ("event", "event_type", "trace_type", "method"):
                    if key in record:
                        event_counts[f"{key}={record[key]}"] += 1

            except json.JSONDecodeError:
                malformed += 1

    print("Valid JSON records:", total)
    print("Malformed records:", malformed)
    print("Event/method counts:")
    for key, count in event_counts.most_common(20):
        print(f"  {key}: {count}")

    print("\nSample record structure:")
    for record in sample_records:
        print({
            key: (list(value.keys()) if isinstance(value, dict) else type(value).__name__)
            for key, value in record.items()
        })

print("\n=== NOTEBOOK GLOBALS ===")
print("candidate_trace_rows exists:", "candidate_trace_rows" in globals())
print("candidate_trace_path:", globals().get("candidate_trace_path", "Not defined"))

if "candidate_trace_rows" in globals():
    print("In-memory trace rows:", len(candidate_trace_rows))

=== RETENTION TRACE VALIDATION ===
File exists: True
File size (MB): 31.09
Valid JSON records: 2115
Malformed records: 0
Event/method counts:

Sample record structure:
{'schema_version': 'str', 'run_id': 'str', 'example_id': 'str', 'created_at': 'str', 'dataset': ['name', 'revision', 'checksum_sha256', 'split', 'conversation_cluster_id'], 'query': ['query_id', 'content_sha256', 'question_type', 'requirements'], 'observability': ['source_archive_available', 'precompression_trace_available', 'memory_provenance_available', 'retrieval_trace_available', 'answer_trace_available', 'unavailable_reasons'], 'pipeline': ['source_evidence', 'candidate_memory', 'stored_memory', 'retrieved_evidence', 'transformations'], 'diagnosis': ['mode', 'probabilities', 'uncertainty', 'model_version', 'feature_snapshot'], 'oracle_labels': ['construction', 'compression', 'retrieval', 'answering', 'annotation_version', 'annotator_agreement'], 'controller': ['selected_action', 'policy_version', 'reason_codes', 'es

In [33]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

trace_path = Path("/kaggle/working/mfar_retention_traces.jsonl")
candidate_path = Path(candidate_trace_path)

# Load retention records
with trace_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

# Load candidate trace CSV
candidates = pd.read_csv(candidate_path)

print("=== RETENTION LEDGER ===")
print("Unique run IDs:", len(set(r.get("run_id") for r in records)))
print("Unique example IDs:", len(set(r.get("example_id") for r in records)))
print("Schema versions:", Counter(r.get("schema_version") for r in records))
print("Dataset names:", Counter(r.get("dataset", {}).get("name") for r in records))
print("Dataset splits:", Counter(r.get("dataset", {}).get("split") for r in records))
print("Outcome statuses:", Counter(r.get("outcome", {}).get("status") for r in records))

print("\n=== OBSERVABILITY ===")
for field in [
    "source_archive_available",
    "precompression_trace_available",
    "memory_provenance_available",
    "retrieval_trace_available",
    "answer_trace_available"
]:
    print(field, Counter(
        r.get("observability", {}).get(field) for r in records
    ))

print("\n=== CANDIDATE TRACE CSV ===")
print("Rows:", len(candidates))
print("Columns:", candidates.columns.tolist())
print("Duplicate rows:", candidates.duplicated().sum())

print("\n=== ID OVERLAP ===")
retention_ids = {str(r.get("example_id")) for r in records}
print("Candidate columns containing ID:", [
    c for c in candidates.columns
    if "id" in c.lower()
])

for col in candidates.columns:
    if "example" in col.lower() and "id" in col.lower():
        candidate_ids = set(candidates[col].dropna().astype(str))
        print("Candidate ID column:", col)
        print("Shared IDs:", len(retention_ids & candidate_ids))
        print("Retention IDs:", len(retention_ids))
        print("Candidate IDs:", len(candidate_ids))

print("\nInspection complete. No experiment was rerun.")

=== RETENTION LEDGER ===
Unique run IDs: 9
Unique example IDs: 2115
Schema versions: Counter({'0.2.0': 2115})
Dataset names: Counter({'LongMemEval-S': 2115})
Dataset splits: Counter({'test': 2115})
Outcome statuses: Counter({'not_evaluated': 2115})

=== OBSERVABILITY ===
source_archive_available Counter({True: 2115})
precompression_trace_available Counter({False: 2115})
memory_provenance_available Counter({True: 2115})
retrieval_trace_available Counter({False: 2115})
answer_trace_available Counter({False: 2115})

=== CANDIDATE TRACE CSV ===
Rows: 49140
Columns: ['question_id_sha256_20', 'query_content_sha256', 'variant', 'candidate_index', 'source_id', 'content_sha256', 'relevance_grade', 'word_cost', 'selected']
Duplicate rows: 0

=== ID OVERLAP ===
Candidate columns containing ID: ['question_id_sha256_20', 'candidate_index', 'source_id']

Inspection complete. No experiment was rerun.


In [34]:
import pandas as pd
from collections import Counter

# Collect identifiers from the retention ledger
example_ids = {str(r.get("example_id")) for r in records}
query_ids = {str(r.get("query", {}).get("query_id")) for r in records}
query_hashes = {str(r.get("query", {}).get("content_sha256")) for r in records}

candidate_ids = set(candidates["question_id_sha256_20"].dropna().astype(str))

print("=== IDENTIFIER MATCHING ===")
print("Candidate question IDs:", len(candidate_ids))
print("Retention example IDs:", len(example_ids))
print("Retention query IDs:", len(query_ids))
print("Retention query hashes:", len(query_hashes))

for name, ids in [
    ("example_id", example_ids),
    ("query_id", query_ids),
    ("query.content_sha256", query_hashes)
]:
    print(f"Exact overlap with {name}:", len(candidate_ids & ids))

# Check whether candidate IDs are prefixes of full SHA-256 hashes
print("\n=== HASH PREFIX CHECK ===")
for name, hashes in [
    ("query.content_sha256", query_hashes),
    ("example_id", example_ids)
]:
    prefixes = {h[:20] for h in hashes if len(h) >= 20}
    print(f"Candidate IDs matching {name} prefixes:",
          len(candidate_ids & prefixes))

print("\n=== VARIANT DISTRIBUTION ===")
print(candidates["variant"].value_counts(dropna=False).to_string())

print("\n=== SELECTION COUNTS ===")
print(candidates["selected"].value_counts(dropna=False).to_string())

print("\n=== CANDIDATES PER QUESTION ===")
per_question = candidates.groupby("question_id_sha256_20").size()
print(per_question.describe().to_string())

print("\n=== RETENTION RUN DISTRIBUTION ===")
print(Counter(r["run_id"] for r in records))

print("\n=== PIPELINE FIELD STRUCTURE ===")
for field in ["source_evidence", "candidate_memory",
              "stored_memory", "retrieved_evidence", "transformations"]:
    values = [r.get("pipeline", {}).get(field) for r in records]
    print(field, "example type:", type(values[0]).__name__,
          "| non-empty records:", sum(bool(v) for v in values))

print("\nInspection complete. No experiment was rerun.")

=== IDENTIFIER MATCHING ===
Candidate question IDs: 235
Retention example IDs: 2115
Retention query IDs: 235
Retention query hashes: 235
Exact overlap with example_id: 0
Exact overlap with query_id: 0
Exact overlap with query.content_sha256: 0

=== HASH PREFIX CHECK ===
Candidate IDs matching query.content_sha256 prefixes: 0
Candidate IDs matching example_id prefixes: 0

=== VARIANT DISTRIBUTION ===
variant
baseline                                 5460
scale=raw|cap=20000|filtered=False       5460
scale=raw|cap=20000|filtered=True        5460
scale=raw|cap=none|filtered=False        5460
scale=raw|cap=none|filtered=True         5460
scale=zscore|cap=20000|filtered=False    5460
scale=zscore|cap=20000|filtered=True     5460
scale=zscore|cap=none|filtered=False     5460
scale=zscore|cap=none|filtered=True      5460

=== SELECTION COUNTS ===
selected
False    44287
True      4853

=== CANDIDATES PER QUESTION ===
count    235.000000
mean     209.106383
std       20.245749
min      153.0000

In [35]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

# Load the existing artifacts
trace_path = Path("/kaggle/working/mfar_retention_traces.jsonl")
candidate_path = Path(candidate_trace_path)

with trace_path.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

candidates = pd.read_csv(candidate_path)

# Compare content hashes, not question IDs
retention_hashes = {
    str(r["query"].get("content_sha256", "")).lower()
    for r in records
    if r.get("query", {}).get("content_sha256")
}

candidate_hashes = {
    str(x).lower()
    for x in candidates["query_content_sha256"].dropna()
}

print("=== CONTENT HASH MATCHING ===")
print("Unique retention content hashes:", len(retention_hashes))
print("Unique candidate content hashes:", len(candidate_hashes))
print("Exact shared hashes:", len(retention_hashes & candidate_hashes))

# Check whether every run has the same question population
print("\n=== RETENTION RUN CONSISTENCY ===")
for run_id, group in pd.DataFrame(records).groupby("run_id"):
    print(run_id, "| records:", len(group),
          "| unique queries:", group["query"].apply(
              lambda x: x.get("query_id")
          ).nunique())

# Locate existing reconciliation outputs
print("\n=== EXISTING RECONCILIATION OUTPUT FILES ===")
output_root = Path("/kaggle/working/memweave_followup_outputs")

if output_root.exists():
    for p in sorted(output_root.rglob("*")):
        if p.is_file() and (
            p.suffix.lower() in [".csv", ".json", ".jsonl", ".txt"]
            or "manifest" in p.name.lower()
        ):
            print(p, "|", round(p.stat().st_size / 1024, 1), "KB")
else:
    print("Output directory not found:", output_root)

print("\nInspection complete. No experiment was rerun.")

=== CONTENT HASH MATCHING ===
Unique retention content hashes: 235
Unique candidate content hashes: 235
Exact shared hashes: 235

=== RETENTION RUN CONSISTENCY ===
run_0a3490eb2811006fe2e0_33025479b8 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_48301af320 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_5f07fcc98d | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_6eb8131247 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_8ba8496a25 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_95e9a8534d | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_c3ecaffab4 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_df2b90d201 | records: 235 | unique queries: 235
run_0a3490eb2811006fe2e0_f27b826d17 | records: 235 | unique queries: 235

=== EXISTING RECONCILIATION OUTPUT FILES ===
/kaggle/working/memweave_followup_outputs/stage_bc_candidate_trace.csv | 11830.2 KB

Inspection complete. No experiment was rerun.


In [37]:
import pandas as pd
import numpy as np

df = pd.read_csv(candidate_trace_path)

# Normalize selection flags
df["selected"] = (
    df["selected"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes"])
)

# Ensure word cost is numeric
df["word_cost"] = pd.to_numeric(df["word_cost"], errors="coerce")

df["relevance_grade"] = (
    df["relevance_grade"].astype(str).str.strip().str.lower()
)

# 1. Variant-level summary
summary_rows = []

for variant, group in df.groupby("variant"):
    chosen = group[group["selected"]]
    grade_counts = chosen["relevance_grade"].value_counts()
    total_selected = len(chosen)

    summary_rows.append({
        "variant": variant,
        "questions": group["query_content_sha256"].nunique(),
        "candidate_rows": len(group),
        "selected_memories": total_selected,
        "selected_per_question": (
            total_selected / group["query_content_sha256"].nunique()
        ),
        "selected_word_cost": chosen["word_cost"].sum(),
        "gold_count": grade_counts.get("gold", 0),
        "gold_fraction": (
            grade_counts.get("gold", 0) / total_selected
            if total_selected else np.nan
        ),
        "irrelevant_count": grade_counts.get("irrelevant", 0),
        "other_grades": {
            k: int(v) for k, v in grade_counts.items()
            if k not in ["gold", "irrelevant"]
        }
    })

summary = pd.DataFrame(summary_rows)

print("=== VARIANT-LEVEL RESULTS ===")
display(summary.round(4))

# 2. Selection-set comparison against baseline
def selected_sets(frame):
    return {
        query: set(
            group.loc[group["selected"], "source_id"]
            .dropna().astype(str)
        )
        for query, group in frame.groupby("query_content_sha256")
    }

baseline = df[df["variant"] == "baseline"]
baseline_sets = selected_sets(baseline)

comparison_rows = []

for variant in sorted(df["variant"].unique()):
    if variant == "baseline":
        continue

    variant_sets = selected_sets(df[df["variant"] == variant])

    changed = 0
    jaccards = []

    for query, baseline_set in baseline_sets.items():
        current_set = variant_sets.get(query, set())

        if baseline_set != current_set:
            changed += 1

        union = baseline_set | current_set
        intersection = baseline_set & current_set

        jaccards.append(
            len(intersection) / len(union) if union else 1.0
        )

    comparison_rows.append({
        "variant": variant,
        "questions_changed": changed,
        "fraction_changed": changed / len(baseline_sets),
        "mean_jaccard": np.mean(jaccards)
    })

comparison = pd.DataFrame(comparison_rows)

print("\n=== SELECTION CHANGES VS BASELINE ===")
display(comparison.round(4))

# 3. Categorical relevance distribution
print("\n=== SELECTED RELEVANCE DISTRIBUTION ===")

grade_distribution = pd.crosstab(
    df[df["selected"]]["variant"],
    df[df["selected"]]["relevance_grade"],
    normalize="index"
).fillna(0)

display((grade_distribution * 100).round(2))

print("\nAnalysis complete. No experiment was rerun.")

=== VARIANT-LEVEL RESULTS ===


,variant,questions,candidate_rows,selected_memories,selected_per_question,selected_word_cost,gold_count,gold_fraction,irrelevant_count,other_grades
0,baseline,235,5460,541,2.3021,1112478.0,192,0.3549,349,{}
1,scale=raw|cap=20000|filtered=False,235,5460,539,2.2936,1141342.0,306,0.5677,233,{}
2,scale=raw|cap=20000|filtered=True,235,5460,539,2.2936,1141342.0,306,0.5677,233,{}
3,scale=raw|cap=none|filtered=False,235,5460,537,2.2851,1138431.0,297,0.5531,240,{}
4,scale=raw|cap=none|filtered=True,235,5460,537,2.2851,1138431.0,297,0.5531,240,{}
5,scale=zscore|cap=20000|filtered=False,235,5460,539,2.2936,1145491.0,340,0.6308,199,{}
6,scale=zscore|cap=20000|filtered=True,235,5460,539,2.2936,1145491.0,340,0.6308,199,{}
7,scale=zscore|cap=none|filtered=False,235,5460,541,2.3021,1146564.0,338,0.6248,203,{}
8,scale=zscore|cap=none|filtered=True,235,5460,541,2.3021,1146564.0,338,0.6248,203,{}



=== SELECTION CHANGES VS BASELINE ===


,variant,questions_changed,fraction_changed,mean_jaccard
0,scale=raw|cap=20000|filtered=False,144,0.6128,0.5342
1,scale=raw|cap=20000|filtered=True,144,0.6128,0.5342
2,scale=raw|cap=none|filtered=False,134,0.5702,0.5682
3,scale=raw|cap=none|filtered=True,134,0.5702,0.5682
4,scale=zscore|cap=20000|filtered=False,182,0.7745,0.3835
5,scale=zscore|cap=20000|filtered=True,182,0.7745,0.3835
6,scale=zscore|cap=none|filtered=False,181,0.7702,0.3862
7,scale=zscore|cap=none|filtered=True,181,0.7702,0.3862



=== SELECTED RELEVANCE DISTRIBUTION ===


relevance_grade,gold,irrelevant
variant,,
baseline,35.49,64.51
scale=raw|cap=20000|filtered=False,56.77,43.23
scale=raw|cap=20000|filtered=True,56.77,43.23
scale=raw|cap=none|filtered=False,55.31,44.69
scale=raw|cap=none|filtered=True,55.31,44.69
scale=zscore|cap=20000|filtered=False,63.08,36.92
scale=zscore|cap=20000|filtered=True,63.08,36.92
scale=zscore|cap=none|filtered=False,62.48,37.52
scale=zscore|cap=none|filtered=True,62.48,37.52



Analysis complete. No experiment was rerun.


In [39]:
import pandas as pd
import numpy as np

# Reuse q if it already exists; otherwise reconstruct it.
if "q" not in globals():
    raise ValueError("The question-level dataframe q is missing. Re-run the previous data-preparation cell.")

rng = np.random.default_rng(42)
results = []

# Paired factorial main effects
for factor, low, high in [
    ("scale", "raw", "zscore"),
    ("cap", "none", "20000"),
    ("filtered", "False", "True")
]:
    # Average over the other two factors for each question
    factor_means = (
        q.groupby(["query", factor])["gold_fraction"]
        .mean()
        .unstack(factor)
    )

    # Paired question-level differences
    differences = (
        factor_means[high] - factor_means[low]
    ).dropna().to_numpy()

    boot = np.array([
        rng.choice(differences, size=len(differences), replace=True).mean()
        for _ in range(10000)
    ])

    results.append({
        "factor": factor,
        "contrast": f"{high} minus {low}",
        "questions": len(differences),
        "mean_effect_pp": differences.mean() * 100,
        "ci_95_low_pp": np.percentile(boot, 2.5) * 100,
        "ci_95_high_pp": np.percentile(boot, 97.5) * 100
    })

print("=== PAIRED FACTORIAL MAIN EFFECTS ===")
display(pd.DataFrame(results).round(3))

# Check whether filtering changes each question's results
print("\n=== FILTERING EQUIVALENCE ===")

filter_checks = []

for scale in ["raw", "zscore"]:
    for cap in ["20000", "none"]:
        subset = q[
            (q["scale"] == scale) &
            (q["cap"] == cap)
        ]

        wide = subset.pivot(
            index="query",
            columns="filtered",
            values=["gold_fraction", "selected_count", "word_cost"]
        )

        same = (
            np.isclose(
                wide[("gold_fraction", "False")],
                wide[("gold_fraction", "True")],
                equal_nan=True
            )
            &
            (wide[("selected_count", "False")] ==
             wide[("selected_count", "True")])
            &
            (wide[("word_cost", "False")] ==
             wide[("word_cost", "True")])
        )

        filter_checks.append({
            "scale": scale,
            "cap": cap,
            "identical_questions": int(same.sum()),
            "total_questions": len(same)
        })

display(pd.DataFrame(filter_checks))

print("\nStatistical analysis complete. No experiment was rerun.")

=== PAIRED FACTORIAL MAIN EFFECTS ===


,factor,contrast,questions,mean_effect_pp,ci_95_low_pp,ci_95_high_pp
0,scale,zscore minus raw,235,7.482,4.504,10.532
1,cap,20000 minus none,235,1.064,0.496,1.702
2,filtered,True minus False,235,0.000,0.000,0.000



=== FILTERING EQUIVALENCE ===


,scale,cap,identical_questions,total_questions
0,raw,20000,235,235
1,raw,none,235,235
2,zscore,20000,235,235
3,zscore,none,235,235



Statistical analysis complete. No experiment was rerun.


In [40]:

from pathlib import Path
from collections import Counter
import pandas as pd

# --------------------------------------------------
# 1. Retrieve the EXISTING primary factorial results
# --------------------------------------------------
print("=== PRIMARY FACTORIAL RESULTS: BUDGETED EVIDENCE RECALL ===")

if "factorial_effects" in globals():
    display(factorial_effects)
else:
    output_dir = Path(
        os.environ.get("MEMWEAVE_OUT", "memweave_reconciliation_outputs")
    )
    effects_path = output_dir / "stage_bc_factorial_effects.csv"

    if effects_path.exists():
        factorial_effects = pd.read_csv(effects_path)
        display(factorial_effects)
    else:
        print("Existing factorial_effects variable and CSV not found.")
        print("No experiment has been rerun.")

# --------------------------------------------------
# 2. Audit the actual message-filtering difference
# --------------------------------------------------
print("\n=== MESSAGE FILTERING POPULATION AUDIT ===")

counts = Counter()
questions_audited = 0
sessions_audited = 0

for qi in TEST:
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))

    # Only inspect sessions in this question's candidate pool.
    session_ids = {
        str(ROW_MAPS[qi][int(h)]["sid"])
        for h in Q[POOL][qi]["h"]
    }

    questions_audited += 1
    sessions_audited += len(session_ids)

    for sid in session_ids:
        for message in sessions[sid]:
            raw_role = str(message.get("role", ""))
            content = str(message.get("content", ""))
            normalized_role = raw_role.lower()

            counts["all_messages"] += 1

            if normalized_role in ("user", "assistant"):
                counts["user_or_assistant"] += 1

                if raw_role != normalized_role:
                    counts["role_case_changes"] += 1

                if not content.strip():
                    counts["empty_user_assistant"] += 1
                else:
                    counts["valid_filtered_messages"] += 1

            else:
                counts["other_roles"] += 1

            if content != content.strip():
                counts["messages_with_outer_whitespace"] += 1

            # Messages grouped by the original unfiltered implementation.
            if raw_role in ("user", "assistant"):
                counts["raw_grouped_role_messages"] += 1

            # Messages grouped by the filtered implementation.
            if normalized_role in ("user", "assistant") and content.strip():
                counts["filtered_grouped_role_messages"] += 1

print("Questions audited:", questions_audited)
print("Candidate sessions audited:", sessions_audited)

display(pd.DataFrame([
    {"measurement": key, "count": value}
    for key, value in counts.items()
]))

print("\nAudit complete. No model or experiment was rerun.")


=== PRIMARY FACTORIAL RESULTS: BUDGETED EVIDENCE RECALL ===


,factorial_term,effect_recall_points,ci95_low,ci95_high
0,scale,0.126809,0.086596,0.169007
1,vocab_cap,-0.015603,-0.026241,-0.006738
2,filter_messages,0.000000,0.000000,0.000000
3,scale_x_vocab_cap,0.009220,-0.000355,0.019504
4,scale_x_filter_messages,0.000000,0.000000,0.000000
5,vocab_cap_x_filter_messages,0.000000,0.000000,0.000000
6,scale_x_vocab_cap_x_filter_messages,0.000000,0.000000,0.000000



=== MESSAGE FILTERING POPULATION AUDIT ===
Questions audited: 235
Candidate sessions audited: 5458


,measurement,count
0,all_messages,65288
1,user_or_assistant,65288
2,valid_filtered_messages,65288
3,raw_grouped_role_messages,65288
4,filtered_grouped_role_messages,65288
5,messages_with_outer_whitespace,37



Audit complete. No model or experiment was rerun.


In [41]:

import numpy as np
import pandas as pd
from scipy.stats import binomtest

stage_b = "scale=raw|cap=20000|filtered=False"
stage_c = "scale=zscore|cap=none|filtered=True"

b = np.array([
    RUNS[stage_b]["metrics"][qi]["recall"] for qi in TEST
])
c = np.array([
    RUNS[stage_c]["metrics"][qi]["recall"] for qi in TEST
])

diff = c - b

rng = np.random.default_rng(20261126)
n_boot = 20000

boot = np.array([
    rng.choice(diff, size=len(diff), replace=True).mean()
    for _ in range(n_boot)
])

positive = int((diff > 0).sum())
negative = int((diff < 0).sum())
ties = int((diff == 0).sum())

non_ties = positive + negative
sign_test = (
    binomtest(positive, non_ties, 0.5).pvalue
    if non_ties else 1.0
)

print("=== CONTROLLED STAGE B-LIKE vs STAGE C-LIKE ===")
print("Questions:", len(diff))
print("Stage B-like mean recall:", round(b.mean(), 6))
print("Stage C-like mean recall:", round(c.mean(), 6))
print("Paired improvement:", round(diff.mean(), 6))
print("Improvement (percentage points):",
      round(diff.mean() * 100, 3))
print("95% paired bootstrap CI (pp):",
      np.round(np.percentile(boot, [2.5, 97.5]) * 100, 3))

print("\n=== QUESTION-LEVEL DIRECTION ===")
print("Stage C-like better:", positive)
print("Stage B-like better:", negative)
print("Ties:", ties)
print("Exact paired sign-test p-value:", sign_test)

print("\nNo model training or experiment rerun.")


=== CONTROLLED STAGE B-LIKE vs STAGE C-LIKE ===
Questions: 235
Stage B-like mean recall: 0.709433
Stage C-like mean recall: 0.820638
Paired improvement: 0.111206
Improvement (percentage points): 11.121
95% paired bootstrap CI (pp): [ 7.099 15.411]

=== QUESTION-LEVEL DIRECTION ===
Stage C-like better: 39
Stage B-like better: 8
Ties: 188
Exact paired sign-test p-value: 5.539621596994947e-06

No model training or experiment rerun.


In [44]:

from collections import Counter, defaultdict

session_questions = defaultdict(set)

for q in DEV:
    for sid in set(map(str, raw[q]["haystack_session_ids"])):
        session_questions[sid].add(q)

degrees = Counter()
for questions in session_questions.values():
    for q in questions:
        degrees[q] += len(questions) - 1

print("Unique source sessions:", len(session_questions))
print("Sessions shared by multiple questions:",
      sum(len(v) > 1 for v in session_questions.values()))

print("\nQuestions with most shared-session links:")
for q, degree in degrees.most_common(15):
    print(str(q)[:30], degree)

print("\nSession reuse distribution:")
print(Counter(len(v) for v in session_questions.values()))


Unique source sessions: 10140
Sessions shared by multiple questions: 1015

Questions with most shared-session links:
291 21
201 21
35 19
357 19
215 18
460 18
213 18
7 17
171 17
23 17
468 17
52 16
374 16
122 16
101 16

Session reuse distribution:
Counter({1: 9125, 2: 920, 3: 89, 4: 6})


In [46]:

import numpy as np
from collections import defaultdict

rng = np.random.default_rng(20261126)

questions = list(DEV)
rng.shuffle(questions)

folds = np.array_split(questions, 5)

sessions = {
    q: set(map(str, raw[q]["haystack_session_ids"]))
    for q in DEV
}

print("PURGED 5-FOLD FEASIBILITY AUDIT")
print("-" * 45)

for fold_id, validation in enumerate(folds, 1):
    validation = set(validation)
    training = set(DEV) - validation

    validation_sessions = set().union(
        *(sessions[q] for q in validation)
    )

    purged = {
        q for q in training
        if sessions[q] & validation_sessions
    }

    retained = training - purged

    print(f"\nFold {fold_id}")
    print("Validation questions:", len(validation))
    print("Training before purge:", len(training))
    print("Training questions purged:", len(purged))
    print("Training retained:", len(retained))
    print("Retention percentage:",
          round(100 * len(retained) / len(training), 2))

    assert all(
        not (sessions[q] & validation_sessions)
        for q in retained
    )

print("\nAll retained training questions are session-disjoint.")


PURGED 5-FOLD FEASIBILITY AUDIT
---------------------------------------------

Fold 1
Validation questions: 47
Training before purge: 188
Training questions purged: 171
Training retained: 17
Retention percentage: 9.04

Fold 2
Validation questions: 47
Training before purge: 188
Training questions purged: 160
Training retained: 28
Retention percentage: 14.89

Fold 3
Validation questions: 47
Training before purge: 188
Training questions purged: 164
Training retained: 24
Retention percentage: 12.77

Fold 4
Validation questions: 47
Training before purge: 188
Training questions purged: 163
Training retained: 25
Retention percentage: 13.3

Fold 5
Validation questions: 47
Training before purge: 188
Training questions purged: 166
Training retained: 22
Retention percentage: 11.7

All retained training questions are session-disjoint.


In [49]:

keywords = (
    "conversation", "source", "person",
    "user_id", "task", "instance",
    "dialogue", "group"
)

print("POTENTIAL GROUPING FIELDS")

for key in sorted(raw[DEV[0]].keys()):
    if any(word in key.lower() for word in keywords):
        values = [raw[q].get(key) for q in DEV]
        present = [str(v) for v in values if v is not None]

        print(
            key,
            "| present:", len(present),
            "| unique:", len(set(present))
        )


POTENTIAL GROUPING FIELDS


In [50]:
print("Available record keys:")
print(list(raw[DEV[0]].keys()))

Available record keys:
['question_id', 'question_type', 'question', 'question_date', 'answer', 'answer_session_ids', 'haystack_dates', 'haystack_session_ids', 'haystack_sessions']


In [51]:
from collections import Counter

answer_sessions = {
    q: set(map(str, raw[q]["answer_session_ids"]))
    for q in DEV
}

session_counts = Counter(
    sid
    for sessions in answer_sessions.values()
    for sid in sessions
)

print("Unique answer sessions:", len(session_counts))
print("Sessions shared across questions:",
      sum(count > 1 for count in session_counts.values()))
print("Maximum questions per answer session:",
      max(session_counts.values(), default=0))
print("Reuse distribution:", Counter(session_counts.values()))

Unique answer sessions: 434
Sessions shared across questions: 3
Maximum questions per answer session: 2
Reuse distribution: Counter({1: 431, 2: 3})


In [55]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path(".")
]

seen = set()

for root in roots:
    if not root.exists():
        continue

    print(f"\n{'=' * 25} {root} {'=' * 25}")

    for path in sorted(root.rglob("*")):
        if path.is_file():
            full_path = str(path.resolve())
            if full_path not in seen:
                seen.add(full_path)
                print(full_path)

print(f"\nTOTAL FILES FOUND: {len(seen)}")


========================= /kaggle/working =========================
/kaggle/working/.virtual_documents/__notebook_source__.ipynb
/kaggle/working/data/locomo10.json
/kaggle/working/memweave_followup_outputs/stage_bc_candidate_trace.csv
/kaggle/working/memweavee/.git/HEAD
/kaggle/working/memweavee/.git/config
/kaggle/working/memweavee/.git/description
/kaggle/working/memweavee/.git/hooks/applypatch-msg.sample
/kaggle/working/memweavee/.git/hooks/commit-msg.sample
/kaggle/working/memweavee/.git/hooks/fsmonitor-watchman.sample
/kaggle/working/memweavee/.git/hooks/post-update.sample
/kaggle/working/memweavee/.git/hooks/pre-applypatch.sample
/kaggle/working/memweavee/.git/hooks/pre-commit.sample
/kaggle/working/memweavee/.git/hooks/pre-merge-commit.sample
/kaggle/working/memweavee/.git/hooks/pre-push.sample
/kaggle/working/memweavee/.git/hooks/pre-rebase.sample
/kaggle/working/memweavee/.git/hooks/pre-receive.sample
/kaggle/working/memweavee/.git/hooks/prepare-commit-msg.sample
/kaggle/work

In [56]:
from pathlib import Path
import pandas as pd

folders = [
    Path("/kaggle/working/memweave_followup_outputs"),
    Path("/kaggle/working/memweavee/MemWeave/results/tables"),
    Path("/kaggle/working/memweavee/MemWeave/results/stage_bc_2026-11-26")
]

for folder in folders:
    if not folder.exists():
        continue

    for file in sorted(folder.glob("*.csv")):
        print("\n" + "=" * 80)
        print("FILE:", file.name)
        try:
            df = pd.read_csv(file)
            print("SHAPE:", df.shape)
            print("COLUMNS:", df.columns.tolist())
            display(df.head())
        except Exception as e:
            print("Could not read:", e)


FILE: stage_bc_candidate_trace.csv
SHAPE: (49140, 9)
COLUMNS: ['question_id_sha256_20', 'query_content_sha256', 'variant', 'candidate_index', 'source_id', 'content_sha256', 'relevance_grade', 'word_cost', 'selected']


,question_id_sha256_20,query_content_sha256,variant,candidate_index,source_id,content_sha256,relevance_grade,word_cost,selected
0,5feceb66ffc86f38d952,2ca9a0d5f1876fa9e2cca5d2f38a6d2b5082703ddfb8ab...,baseline,0,src_4436eb4ffa623d1b1cbe0c066bdf24a1,d9fde8e9793d90c0b165c154fd843c97962d1b48799964...,irrelevant,2476.0,False
1,5feceb66ffc86f38d952,2ca9a0d5f1876fa9e2cca5d2f38a6d2b5082703ddfb8ab...,baseline,1,src_80d294d77490f73e215a465b08be4b77,67b8d027f22f288fa0d86abf93aa7a4cd8c69fe8817408...,irrelevant,1737.0,False
2,5feceb66ffc86f38d952,2ca9a0d5f1876fa9e2cca5d2f38a6d2b5082703ddfb8ab...,baseline,2,src_2c8d3e184637250a747c9821cf7f3879,ad53f52cb077cf634457956cbcfd8ac33605fc41174a7c...,irrelevant,2208.0,False
3,5feceb66ffc86f38d952,2ca9a0d5f1876fa9e2cca5d2f38a6d2b5082703ddfb8ab...,baseline,3,src_8a435685a9fe52d078960288707713c7,0e497b357805f466aeb131947203ff162943dcfe402f24...,irrelevant,2629.0,False
4,5feceb66ffc86f38d952,2ca9a0d5f1876fa9e2cca5d2f38a6d2b5082703ddfb8ab...,baseline,4,src_8f5ad5f78e072092ef8f05fac2081d0e,11a88732572752eb5035e993f5fb95339ae31849e4ddf0...,irrelevant,1898.0,False



FILE: table_abstention_sensitivity.csv
SHAPE: (3, 8)
COLUMNS: ['comparison', 'subset', 'mean_diff', 'ci_low', 'ci_high', 'p', 'n', 'holm_p']


,comparison,subset,mean_diff,ci_low,ci_high,p,n,holm_p
0,MemWeave-LR (primary) - low_novelty,excluding _abs,0.024764,0.000142,0.049612,7.744311e-02,470,7.744311e-02
1,MemWeave-LR (primary) - random,excluding _abs,0.307270,0.267351,0.346856,1.967716e-37,470,5.903148e-37
2,MemWeave-LR (primary) - recency,excluding _abs,0.292813,0.248994,0.336691,3.353363e-26,470,6.706727e-26



FILE: table_artifact_facts.csv
SHAPE: (5, 2)
COLUMNS: ['statement', 'value']


,statement,value
0,gold sessions whose ID starts with 'answer_',948 / 948
1,non-gold sessions whose ID starts with 'answer_',0 / 22919
2,abstention (_abs) questions that still list ev...,30 / 30 (58 sessions)
3,questions with 2+ evidence sessions,324 / 500
4,2+ evidence: nearest neighbour of an evidence ...,0.820 vs 0.030


... (18 outputs left out)


FILE: table_related_work_ledger.csv
SHAPE: (5, 6)
COLUMNS: ['work', 'where', 'covers', 'overlap', 'gap_vs_yours', 'full_text_read']


,work,where,covers,overlap,gap_vs_yours,full_text_read
0,Auditing Long-Term Memory Evaluation: Repeated...,arXiv 2609.38021,"judge noise, reader variation and answer-key d...",audit framing and LongMemEval-S,answer-level; no retention-budget or ID/cluste...,False
1,UTILMEM,arXiv 2608.30508,"new benchmark with strong, semantically adjace...","motivates harder distractors, i.e. same concer...",builds a benchmark; excerpt does not report an...,False
2,TIAP: scoring-target audit,arXiv 2605.24060,re-scores saved retrievals under raw/source/ca...,audit of evaluation design on both of your dat...,"scoring targets, not retention or distractor s...",False
3,ReMe_longmemeval_clean_s_v2 (reviewed labels),huggingface.co/datasets/agentscope-ai/ReMe_lon...,reviewed answers and evidence IDs; evidence re...,DIRECTLY affects your labels (your file has 94...,"label correction, not an artifact analysis",False
4,LoCoMo audit (Penfield Labs blog),penfieldlabs.substack.com,answer-key errors in LoCoMo; argues LongMemEva...,critique of both benchmarks,"blog, not peer reviewed; answer key and contex...",False



FILE: table_retention_vs_qa.csv
SHAPE: (4, 4)
COLUMNS: ['method', 'retention_recall', 'qa_accuracy', 'share_with_any_gold']


,method,retention_recall,qa_accuracy,share_with_any_gold
0,MemWeave-LR (primary),0.440000,0.11,0.56
1,low_novelty,0.401000,0.09,0.49
2,random,0.071667,0.05,0.12
3,recency,0.161667,0.14,0.26


In [58]:
import json
from pathlib import Path

p = Path(
    "/kaggle/working/memweavee/MemWeave/notebooks/"
    "06_Stage_BC_Controlled_Reconciliation.ipynb"
)

print("File exists:", p.exists())
print("File size:", p.stat().st_size if p.exists() else "N/A")

nb = json.loads(p.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    outputs = cell.get("outputs", [])

    if any(x in source.lower() for x in [
        "groupkfold", "grouped", "foldsplits",
        "cross-validation", "cross validation"
    ]):
        print(f"\n{'='*70}")
        print("CELL:", i)
        print(source[:3500])
        print("SAVED OUTPUTS:")
        for out in outputs:
            print("".join(out.get("text", []))[:3000])
            data = out.get("data", {})
            if "text/plain" in data:
                print("".join(data["text/plain"])[:3000])

File exists: True
File size: 35492

CELL: 9
# Reconciliation configuration — freeze before TEST evaluation.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest
import hashlib, json, itertools, platform
from pathlib import Path

ROLE_FEATURE_NAMES = [
    "user_max_question_similarity",
    "user_top3_mean_question_similarity",
    "assistant_max_question_similarity",
    "assistant_top3_mean_question_similarity",
    "log_user_message_count",
    "log_assistant_message_count",
]
ROW_MAPS = {
    qi: {int(row["h"]): row for row in qrecs[qi]["rows"]}
    for qi in DEV + TEST
}

def construct_role_features(qi, *, scale, vocab_cap, filter_messages):
    q = qrecs[qi]
    sessions = dict(zip(
        map(str, raw[qi]["haystack_session_ids"]),
        raw[qi]["haystack_sessions"]
    ))
    hashes = Q[POOL][qi]["h"]
    n = len(hashes)
    texts, roles, candidate_ids = [], [], []
    user_counts = n

In [59]:
import json
from pathlib import Path

root = Path("/kaggle/working/memweavee")

terms = [
    "GroupKFold",
    "grouped validation",
    "grouped cv",
    "fold assignments",
    "fold bootstrap",
    "DEV_CV_SUMMARY",
    "FOLD_SPLITS",
    "answer-session",
]

for path in root.rglob("*.ipynb"):
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        outputs = []

        for out in cell.get("outputs", []):
            outputs.extend([
                "".join(out.get("text", [])),
                "".join(out.get("data", {}).get("text/plain", []))
            ])

        output_text = "\n".join(outputs)
        matches = [
            term for term in terms
            if term.lower() in (source + "\n" + output_text).lower()
        ]

        if matches:
            print("\n" + "=" * 80)
            print("NOTEBOOK:", path.name)
            print("CELL:", i)
            print("MATCHES:", matches)
            print("CODE:", source[:1800])
            print("OUTPUT:", output_text[:3500])


NOTEBOOK: 03_memweave_v2_3_addendum.ipynb
CELL: 3
MATCHES: ['GroupKFold', 'grouped cv']
CODE: # ===== Section 23. Corrected inference layer (reads saved results only; changes no model, no tuning) =====
# Fixes: (a) H2 verdicts must use Holm; (b) duplicate rule counted twice; (c) "SUPPORTED" for components that were never active;
# (d) abstention questions scored as if they had evidence; (e) session-reuse leak path, via an exposure-gradient test.
import json, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
RES_DIR   = globals().get("RES_DIR", Path("memweave2_results/v2_2"))
DATA_PATH = globals().get("DATA_PATH", Path("data/longmemeval/longmemeval_s_cleaned.json"))
PF_, PRIM = 0.10, "MemWeave-LR (primary)"
R23 = globals().get("R", None)
if R23 is None: R23 = pd.read_csv(RES_DIR / "all_results_raw.csv")
raw23 = globals().get("raw", None)
if raw23 is None: raw23 = json.load(open(DATA_PATH, encoding="utf-8"))
rng23 = np.random.default_rng(23)

def holm23(ps):
   

In [60]:
names = [
    "DEV_CV_SUMMARY",
    "FOLD_SPLITS",
    "DEV",
    "TEST",
    "POOL",
    "Q",
    "BASE_IDX",
    "C_FROZEN",
    "VARIANT_CONFIGS",
    "DEV_ROLE_FEATURES",
    "SEED_GROUP_PROTOCOL",
    "model",
    "eval_fn",
]

print("CURRENT KERNEL STATE")
print("=" * 55)

for name in names:
    exists = name in globals()
    value = globals().get(name)

    if exists and hasattr(value, "shape"):
        detail = f"shape={value.shape}"
    elif exists and hasattr(value, "__len__"):
        try:
            detail = f"length={len(value)}"
        except Exception:
            detail = type(value).__name__
    elif exists:
        detail = type(value).__name__
    else:
        detail = "NOT PRESENT"

    print(f"{name:25} {detail}")

CURRENT KERNEL STATE
DEV_CV_SUMMARY            NOT PRESENT
FOLD_SPLITS               NOT PRESENT
DEV                       length=235
TEST                      length=235
POOL                      length=13
Q                         length=3
BASE_IDX                  length=5
C_FROZEN                  float
VARIANT_CONFIGS           length=8
DEV_ROLE_FEATURES         NOT PRESENT
SEED_GROUP_PROTOCOL       NOT PRESENT
model                     LogisticRegression
eval_fn                   NOT PRESENT


In [61]:
import subprocess
import json
from pathlib import Path

repo = Path("/kaggle/working/memweavee")
branch = "research/memweave-followup-release"

# Update the existing repository clone.
subprocess.run(
    ["git", "-C", str(repo), "fetch", "origin", branch],
    check=True
)
subprocess.run(
    ["git", "-C", str(repo), "checkout", branch],
    check=True
)
subprocess.run(
    ["git", "-C", str(repo), "pull", "--ff-only", "origin", branch],
    check=True
)

# Load the newly committed self-contained experiment.
notebook_path = (
    repo / "MemWeave/notebooks/"
    "06_Stage_BC_Controlled_Reconciliation.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

matches = [
    cell for cell in nb["cells"]
    if cell["cell_type"] == "code"
    and "Self-contained DEV-only novelty-aware retention experiment."
    in "".join(cell["source"])
]

assert len(matches) == 1, f"Expected one experiment cell; found {len(matches)}"

source = "".join(matches[0]["source"])

print("Updated notebook loaded.")
print("Starting novelty-aware DEV experiment...")

exec(compile(source, "memweave_novelty_experiment.py", "exec"), globals())

From https://github.com/tishaaaaaaaaa5-sketch/memweavee
 * branch            research/memweave-followup-release -> FETCH_HEAD
   d4a4213..7011ebc  research/memweave-followup-release -> origin/research/memweave-followup-release
Already on 'research/memweave-followup-release'


Your branch is behind 'origin/research/memweave-followup-release' by 25 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)


From https://github.com/tishaaaaaaaaa5-sketch/memweavee
 * branch            research/memweave-followup-release -> FETCH_HEAD


Updating d4a4213..7011ebc
Fast-forward
 MemWeave/README.md                                 |  15 +-
 MemWeave/docs/CANONICAL_PIPELINE_FREEZE.md         | 182 ++++++++
 .../06_Stage_BC_Controlled_Reconciliation.ipynb    | 486 ++++++++++++++++++++-
 MemWeave/notebooks/README.md                       |   8 +-
 MemWeave/results/README.md                         |  16 +
 MemWeave/results/stage_bc_reconciliation/README.md | 111 +++++
 6 files changed, 814 insertions(+), 4 deletions(-)
 create mode 100644 MemWeave/docs/CANONICAL_PIPELINE_FREEZE.md
 create mode 100644 MemWeave/results/stage_bc_reconciliation/README.md
Updated notebook loaded.
Starting novelty-aware DEV experiment...
NOVELTY-AWARE DEV EXPERIMENT COMPLETE
DEV questions: 235
Answer-session groups: 233
Fold sizes: [47, 47, 47, 47, 47]


,alpha_relevance,alpha_novelty,mean_oof_budgeted_evidence_recall,group_bootstrap_se,group_bootstrap_ci95_low,group_bootstrap_ci95_high,paired_delta_vs_relevance_only,paired_delta_ci95_low,paired_delta_ci95_high,mean_selected_evidence_precision,mean_budget_utilization,mean_selected_novelty_percentile
0,1.00,0.00,0.4049,0.0279,0.3520,0.4607,0.0000,0.0000,0.0000,0.3351,0.8856,0.1654
1,0.90,0.10,0.4060,0.0275,0.3531,0.4604,0.0011,-0.0159,0.0191,0.3277,0.8869,0.1815
2,0.75,0.25,0.3727,0.0253,0.3242,0.4227,-0.0322,-0.0643,-0.0004,0.2851,0.8821,0.2701
3,0.50,0.50,0.1833,0.0237,0.1376,0.2315,-0.2216,-0.2879,-0.1554,0.0837,0.8770,0.7729


Saved outputs: /kaggle/working/memweave_followup_outputs/novelty_aware_dev_experiment


In [62]:
%cd /kaggle/working/memweavee
!git pull origin research/memweave-followup-release

/kaggle/working/memweavee
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 37 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 10.93 KiB | 799.00 KiB/s, done.
From https://github.com/tishaaaaaaaaa5-sketch/memweavee
 * branch            research/memweave-followup-release -> FETCH_HEAD
   7011ebc..47e67a1  research/memweave-followup-release -> origin/research/memweave-followup-release
Updating 7011ebc..47e67a1
Fast-forward
 .../06_Stage_BC_Controlled_Reconciliation.ipynb    | 14 +++++-
 MemWeave/notebooks/README.md                       | 12 +++++
 MemWeave/results/README.md                         | 10 +++++
 .../CONSTRAINED_NOVELTY_TIEBREAKER_PROTOCOL.md     | 48 ++++++++++++++++++++
 .../NOVELTY_AWARE_DEV_EXPERIMENT.md                | 51 ++++++++++++++++++++++
 5 files changed, 134 insertions(+), 1 deletion(-)
 create mode 100644 MemWeave/results/s

In [63]:
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/memweavee/"
    "MemWeave/notebooks/06_Stage_BC_Controlled_Reconciliation.ipynb"
)

assert notebook_path.exists(), (
    "Notebook not found. Make sure you pulled the latest GitHub branch."
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cell 27 contains the constrained novelty tie-breaker.
cell = notebook["cells"][27]

assert cell["cell_type"] == "code", "Expected experiment code in cell 27."

code = cell["source"]
if isinstance(code, list):
    code = "".join(code)

exec(compile(code, "constrained_novelty_tiebreaker.py", "exec"), globals())

CONSTRAINED NOVELTY TIE-BREAKER COMPLETE
DEV questions: 235
Answer-session groups: 233
Fold sizes: [47, 47, 47, 47, 47]


,method,mean_oof_budgeted_evidence_recall,group_bootstrap_se,group_bootstrap_ci95_low,group_bootstrap_ci95_high,mean_selected_evidence_precision,mean_budget_utilization,mean_selected_novelty_percentile,mean_selected_relevance_percentile
0,relevance_only,0.4049,0.0273,0.3509,0.4578,0.3351,0.8856,0.1654,0.9500
1,constrained_novelty_tiebreak_5pp,0.4028,0.0271,0.3495,0.4556,0.3344,0.8846,0.1648,0.9503


,comparison,mean_paired_delta_recall,paired_group_bootstrap_ci95_low,paired_group_bootstrap_ci95_high,n_dev_questions,n_answer_session_groups,relevance_band_width
0,constrained_novelty_tiebreak_5pp - relevance_only,-0.0021,-0.0086,0.0043,235,233,0.05


Saved outputs: /kaggle/working/memweave_followup_outputs/constrained_novelty_tiebreaker_dev


In [64]:
%cd /kaggle/working/memweavee
!git pull origin research/memweave-followup-release

/kaggle/working/memweavee
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 48 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (48/48), 15.31 KiB | 922.00 KiB/s, done.
From https://github.com/tishaaaaaaaaa5-sketch/memweavee
 * branch            research/memweave-followup-release -> FETCH_HEAD
   47e67a1..8966d8c  research/memweave-followup-release -> origin/research/memweave-followup-release
Updating 47e67a1..8966d8c
Fast-forward
 MemWeave/README.md                                 |  5 +++
 .../notebooks/05_MemWeave_Kaggle_Followup.ipynb    | 19 +++++++++
 MemWeave/notebooks/README.md                       |  9 ++++-
 MemWeave/results/README.md                         |  2 +-
 .../followup_2026/END_TO_END_READER_QA_PROTOCOL.md | 46 ++++++++++++++++++++++
 MemWeave/results/followup_2026/README.md           |  7 ++++
 .../CONSTRAINED_NOVELTY_TIEBREAKER_PROTOCOL

In [67]:
%pip install -q transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [69]:
# ============================================================
# LOAD LOCOMO DIRECTLY — NO DATASET REDOWNLOAD
# ============================================================

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

LOCOMO_PATH = Path("/kaggle/working/data/locomo10.json")

if not LOCOMO_PATH.exists():
    raise FileNotFoundError(
        f"LoCoMo dataset not found: {LOCOMO_PATH}"
    )

with open(LOCOMO_PATH, "r", encoding="utf-8") as f:
    locomo = json.load(f)

if isinstance(locomo, dict):
    locomo = locomo.get("data", [locomo])

assert isinstance(locomo, list), "Unexpected LoCoMo JSON structure."

print("LoCoMo loaded:", len(locomo), "conversations")
print("Dataset size:", f"{LOCOMO_PATH.stat().st_size:,} bytes")

# Restore output directory
OUT_DIR = Path(
    "/kaggle/working/memweave_followup_outputs"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Check remaining Reader QA prerequisites
required = [
    "FROZEN_BASELINE",
    "FROZEN_ROLE_AWARE",
    "DATA_PROVENANCE",
    "locomo_gold_indices",
    "extract_session_messages",
    "session_stats",
    "role_features_for_sessions",
    "build_features",
    "tfidf",
    "hv",
    "BASE_IDX",
    "FULL_IDX"
]

missing = [
    name for name in required
    if name not in globals()
]

print("\nReader QA prerequisite check:")
print("Missing:", missing if missing else "None")

if missing:
    print(
        "\nThe dataset is loaded, but the missing selector/"
        "feature definitions must be restored from Notebook 05."
    )
else:
    print("\nREADY: You can execute the Reader QA evaluation cell.")

LoCoMo loaded: 10 conversations
Dataset size: 2,805,274 bytes

Reader QA prerequisite check:
Missing: ['FROZEN_BASELINE', 'FROZEN_ROLE_AWARE', 'locomo_gold_indices', 'extract_session_messages', 'role_features_for_sessions']

The dataset is loaded, but the missing selector/feature definitions must be restored from Notebook 05.


In [70]:

# ============================================================
# MEMWEAVE: END-TO-END READER QA
# Exact token-budgeted memory -> retrieval -> generated answer
# ============================================================

import os, re, json, time, string, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# 1. Verify prerequisites
# ------------------------------------------------------------

assert "locomo" in globals() and len(locomo) == 10, \
    "Run the LoCoMo loading cell first."

assert "FROZEN_BASELINE" in globals(), \
    "Frozen baseline selector is missing."

assert "FROZEN_ROLE_AWARE" in globals(), \
    "Frozen role-aware selector is missing."

assert "DATA_PROVENANCE" in globals(), \
    "Dataset provenance is missing."

required_functions = [
    "locomo_gold_indices",
    "extract_session_messages",
    "session_stats",
    "role_features_for_sessions",
    "build_features"
]

for name in required_functions:
    assert name in globals(), f"Missing required function: {name}"

assert "tfidf" in globals() and "hv" in globals(), \
    "The fitted TF-IDF representation is missing."

assert "BASE_IDX" in globals() and "FULL_IDX" in globals(), \
    "Frozen feature indices are missing."

assert "OUT_DIR" in globals(), \
    "Output directory OUT_DIR is missing."

# ------------------------------------------------------------
# 2. Fixed experiment configuration
# ------------------------------------------------------------

READER_MODEL_ID = "google/flan-t5-small"

MAX_QA_PER_CONVERSATION = 20
READER_INPUT_MAX_TOKENS = 512
READER_MAX_NEW_TOKENS = 64

RETRIEVAL_TOP_K = 3
MEMORY_BUDGET_FRACTION = 0.10

SAMPLE_SEED = 20261126
BOOTSTRAP_REPS = 20000

QA_OUT_DIR = Path(OUT_DIR) / "locomo_reader_qa"
QA_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Load and pin reader model
# ------------------------------------------------------------

reader_revision = HfApi().model_info(READER_MODEL_ID).sha

print("Reader:", READER_MODEL_ID)
print("Revision:", reader_revision)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

reader_tokenizer = AutoTokenizer.from_pretrained(
    READER_MODEL_ID,
    revision=reader_revision
)

reader_model = AutoModelForSeq2SeqLM.from_pretrained(
    READER_MODEL_ID,
    revision=reader_revision
).to(device)

reader_model.eval()

print("Device:", device)

# ------------------------------------------------------------
# 4. Answer evaluation functions
# ------------------------------------------------------------

def qa_normalize(text):
    text = str(text or "").lower()

    text = "".join(
        ch for ch in text
        if ch not in string.punctuation
    )

    text = re.sub(r"\b(a|an|the)\b", " ", text)

    return " ".join(text.split())


def qa_exact_match(prediction, reference):
    return float(
        qa_normalize(prediction) ==
        qa_normalize(reference)
    )


def qa_token_f1(prediction, reference):

    pred = qa_normalize(prediction).split()
    gold = qa_normalize(reference).split()

    if not pred and not gold:
        return 1.0

    if not pred or not gold:
        return 0.0

    counts = {}

    for token in pred:
        counts[token] = counts.get(token, 0) + 1

    overlap = 0

    for token in gold:
        if counts.get(token, 0) > 0:
            overlap += 1
            counts[token] -= 1

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred)
    recall = overlap / len(gold)

    return (
        2 * precision * recall /
        (precision + recall)
    )


def qa_references(answer):

    if isinstance(answer, (list, tuple)):
        refs = [
            str(x) for x in answer
            if str(x).strip()
        ]

        return refs or [""]

    return [str(answer or "")]


def qa_metrics(prediction, answer):

    refs = qa_references(answer)

    em = max(
        qa_exact_match(prediction, ref)
        for ref in refs
    )

    f1 = max(
        qa_token_f1(prediction, ref)
        for ref in refs
    )

    return em, f1


# ------------------------------------------------------------
# 5. Session rendering and token-budget selection
# ------------------------------------------------------------

def render_locomo_session(turns, speaker_a, speaker_b):

    lines = []

    for role, message in turns:

        speaker = (
            speaker_a if role == "user"
            else speaker_b
        )

        lines.append(f"{speaker}: {message}")

    return "\n".join(lines)


def select_exact_token_budget(
    scores,
    token_costs,
    fraction=MEMORY_BUDGET_FRACTION
):

    scores = np.asarray(scores, dtype=float)
    costs = np.asarray(token_costs, dtype=int)

    cap = max(
        1,
        int(np.floor(float(costs.sum()) * fraction))
    )

    chosen = []
    used = 0

    for idx in np.argsort(-scores, kind="stable"):

        cost = int(costs[idx])

        if used + cost <= cap:
            chosen.append(int(idx))
            used += cost

    return chosen, used, cap


def build_reader_prompt(question, context):

    if context:

        return (
            "Answer the question using only the conversation context. "
            "Give a concise answer. If the context does not contain "
            "the answer, say: I don't know.\n\n"
            f"Conversation context:\n{context}\n\n"
            f"Question: {question}\nAnswer:"
        )

    return (
        "Answer the question. No conversation context is available. "
        "If you cannot determine the answer, say: I don't know.\n\n"
        f"Question: {question}\nAnswer:"
    )


# ------------------------------------------------------------
# 6. Deterministic question sampling
# ------------------------------------------------------------

sampled = []

for conv_idx, item in enumerate(locomo):

    conv = item.get("conversation", {})

    session_numbers = sorted(
        int(k.split("_")[1])
        for k in conv
        if re.fullmatch(r"session_\d+", k)
    )

    valid_qas = []

    for qa_idx, qa in enumerate(item.get("qa", []) or []):

        if not isinstance(qa, dict):
            continue

        if qa.get("category") == 5:
            continue

        question = str(
            qa.get("question", "")
        ).strip()

        answer = qa.get("answer", "")

        if not question or answer is None:
            continue

        if not str(answer).strip():
            continue

        gold = locomo_gold_indices(
            qa,
            session_numbers
        )

        if gold:
            valid_qas.append(
                (qa_idx, qa, gold)
            )

    rng = np.random.default_rng(
        SAMPLE_SEED + conv_idx
    )

    if len(valid_qas) > MAX_QA_PER_CONVERSATION:

        chosen_idx = np.sort(
            rng.choice(
                len(valid_qas),
                size=MAX_QA_PER_CONVERSATION,
                replace=False
            )
        )

        valid_qas = [
            valid_qas[int(i)]
            for i in chosen_idx
        ]

    for qa_idx, qa, gold in valid_qas:

        sampled.append(
            (
                conv_idx,
                qa_idx,
                item,
                qa,
                gold,
                session_numbers
            )
        )


if not sampled:
    raise RuntimeError(
        "No answerable LoCoMo questions with valid evidence annotations."
    )


print("Sampled questions:", len(sampled))

print(
    "Conversation clusters:",
    len(set(x[0] for x in sampled))
)

# ------------------------------------------------------------
# 7. Construct selector scores and reader prompts
# ------------------------------------------------------------

prompt_rows = []
question_meta = []

for sample_id, (
    conv_idx,
    qa_idx,
    item,
    qa,
    gold,
    session_numbers
) in enumerate(sampled):

    conv = item["conversation"]

    speaker_a = str(
        conv.get("speaker_a", "Speaker A")
    )

    speaker_b = str(
        conv.get("speaker_b", "Speaker B")
    )

    sessions = []
    rendered_sessions = []

    word_costs = []
    user_counts = []
    turn_counts = []
    fp_values = []
    digit_values = []

    for number in session_numbers:

        turns = extract_session_messages(
            conv,
            number
        )

        if not turns:
            continue

        full, words, user_words, nturns, fpv, dgv = (
            session_stats(turns, "user")
        )

        rendered = render_locomo_session(
            turns,
            speaker_a,
            speaker_b
        )

        if not rendered.strip():
            continue

        sessions.append(turns)
        rendered_sessions.append(rendered)

        word_costs.append(words)
        user_counts.append(user_words)
        turn_counts.append(nturns)
        fp_values.append(fpv)
        digit_values.append(dgv)

    if not sessions:
        continue

    question = str(
        qa["question"]
    ).strip()

    answers = qa_references(
        qa.get("answer", "")
    )

    # Reuse fitted LongMemEval representation.
    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    days = np.arange(
        len(sessions),
        dtype=float
    )

    Fbase, _ = build_features(
        V,
        np.asarray(word_costs, dtype=float),
        np.asarray(user_counts, dtype=float),
        np.asarray(turn_counts, dtype=float),
        np.asarray(fp_values, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    Frole = role_features_for_sessions(
        question,
        sessions
    )

    Ffull = np.column_stack(
        [Fbase, Frole]
    ).astype(np.float32)

    score_map = {

        "baseline":
            FROZEN_BASELINE.predict_proba(
                Fbase[:, BASE_IDX]
            )[:, 1],

        "role_aware":
            FROZEN_ROLE_AWARE.predict_proba(
                Ffull[:, FULL_IDX]
            )[:, 1],

        "recency":
            Fbase[:, 2]
    }

    # Exact FLAN-T5 tokenizer costs.
    token_costs = np.asarray(
        [
            len(
                reader_tokenizer.encode(
                    text,
                    add_special_tokens=False
                )
            )
            for text in rendered_sessions
        ],
        dtype=int
    )

    qvec = tfidf.transform(
        hv.transform([question])
    )

    similarities = (
        V @ qvec.T
    ).toarray().ravel()

    question_hash = hashlib.sha256(
        f"{conv_idx}|{qa_idx}|{question}".encode("utf-8")
    ).hexdigest()[:20]

    question_meta.append({

        "sample_id": sample_id,

        "conversation_index": int(conv_idx),

        "question_hash": question_hash,

        "question": question,

        "answer": answers,

        "category": str(
            qa.get("category", "unknown")
        ),

        "gold_session_count": int(len(gold))
    })

    # Three selector methods.
    for method, scores in score_map.items():

        selected, memory_used, memory_cap = (
            select_exact_token_budget(
                scores,
                token_costs
            )
        )

        retrieved = sorted(
            selected,
            key=lambda i: (
                -float(similarities[i]),
                i
            )
        )[:RETRIEVAL_TOP_K]

        context = "\n\n".join(
            rendered_sessions[i]
            for i in retrieved
        )

        prompt_rows.append({

            "sample_id": sample_id,

            "method": method,

            "prompt": build_reader_prompt(
                question,
                context
            ),

            "memory_tokens_selected": int(memory_used),

            "memory_token_budget": int(memory_cap),

            "retrieved_context_tokens": int(
                len(
                    reader_tokenizer.encode(
                        context,
                        add_special_tokens=False
                    )
                )
            ),

            "selected_session_count": int(
                len(selected)
            ),

            "retrieved_session_count": int(
                len(retrieved)
            )
        })

    # Question-only control.
    prompt_rows.append({

        "sample_id": sample_id,

        "method": "question_only",

        "prompt": build_reader_prompt(
            question,
            ""
        ),

        "memory_tokens_selected": 0,

        "memory_token_budget": 0,

        "retrieved_context_tokens": 0,

        "selected_session_count": 0,

        "retrieved_session_count": 0
    })


if not prompt_rows:
    raise RuntimeError(
        "No reader prompts were constructed."
    )


print("Reader prompts:", len(prompt_rows))

# ------------------------------------------------------------
# 8. Generate answers
# ------------------------------------------------------------

batch_size = (
    8 if device.type == "cuda"
    else 4
)

generated = []

for start in range(
    0,
    len(prompt_rows),
    batch_size
):

    batch = prompt_rows[
        start:start + batch_size
    ]

    encoded = reader_tokenizer(
        [x["prompt"] for x in batch],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=READER_INPUT_MAX_TOKENS
    ).to(device)

    t0 = time.time()

    with torch.inference_mode():

        output_ids = reader_model.generate(
            **encoded,
            max_new_tokens=READER_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1
        )

    elapsed = time.time() - t0

    decoded = reader_tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )

    per_item_seconds = (
        elapsed / max(1, len(batch))
    )

    for row, prediction in zip(batch, decoded):

        generated.append({

            **{
                k: v
                for k, v in row.items()
                if k != "prompt"
            },

            "prediction": str(
                prediction
            ).strip(),

            "generation_seconds_est": float(
                per_item_seconds
            )
        })

    if (start // batch_size + 1) % 20 == 0:

        print(
            f"Generated {min(start + batch_size, len(prompt_rows))}"
            f"/{len(prompt_rows)} answers"
        )


# ------------------------------------------------------------
# 9. Score generated answers
# ------------------------------------------------------------

pred_by_key = {

    (int(row["sample_id"]), row["method"]): row

    for row in generated
}

meta_by_id = {

    int(x["sample_id"]): x

    for x in question_meta
}

scored_rows = []

for (sample_id, method), row in pred_by_key.items():

    meta = meta_by_id[sample_id]

    em, f1 = qa_metrics(
        row["prediction"],
        meta["answer"]
    )

    scored_rows.append({

        "conversation_index":
            meta["conversation_index"],

        "question_hash":
            meta["question_hash"],

        "category":
            meta["category"],

        "method":
            method,

        "exact_match":
            em,

        "token_f1":
            f1,

        "memory_tokens_selected":
            row["memory_tokens_selected"],

        "memory_token_budget":
            row["memory_token_budget"],

        "retrieved_context_tokens":
            row["retrieved_context_tokens"],

        "selected_session_count":
            row["selected_session_count"],

        "retrieved_session_count":
            row["retrieved_session_count"],

        "generation_seconds_est":
            row["generation_seconds_est"]
    })


qa_results = pd.DataFrame(scored_rows)

# ------------------------------------------------------------
# 10. Aggregate method-level metrics
# ------------------------------------------------------------

method_summary = (

    qa_results.groupby("method")

    .agg(

        n=("token_f1", "size"),

        exact_match=("exact_match", "mean"),

        token_f1=("token_f1", "mean"),

        mean_memory_tokens=(
            "memory_tokens_selected",
            "mean"
        ),

        mean_reader_context_tokens=(
            "retrieved_context_tokens",
            "mean"
        ),

        mean_selected_sessions=(
            "selected_session_count",
            "mean"
        )
    )

    .reset_index()
)

# ------------------------------------------------------------
# 11. Paired conversation-cluster bootstrap
# ------------------------------------------------------------

def cluster_bootstrap_delta(
    df,
    metric,
    method_a="role_aware",
    method_b="baseline",
    n_boot=BOOTSTRAP_REPS,
    seed=20261127
):

    wide = df.pivot_table(

        index="conversation_index",

        columns="method",

        values=metric,

        aggfunc="mean"
    )

    if method_a not in wide or method_b not in wide:

        return {

            "delta": np.nan,

            "ci95_low": np.nan,

            "ci95_high": np.nan,

            "clusters": int(len(wide))
        }

    paired = (
        wide[method_a] - wide[method_b]
    ).dropna().to_numpy(float)

    if len(paired) < 2:

        return {

            "delta": (
                float(np.mean(paired))
                if len(paired) else np.nan
            ),

            "ci95_low": np.nan,

            "ci95_high": np.nan,

            "clusters": int(len(paired))
        }

    rng = np.random.default_rng(seed)

    draws = np.empty(
        n_boot,
        dtype=float
    )

    for start in range(0, n_boot, 1000):

        end = min(
            start + 1000,
            n_boot
        )

        ix = rng.integers(
            0,
            len(paired),
            size=(end - start, len(paired))
        )

        draws[start:end] = paired[ix].mean(axis=1)

    return {

        "delta": float(paired.mean()),

        "ci95_low": float(
            np.quantile(draws, 0.025)
        ),

        "ci95_high": float(
            np.quantile(draws, 0.975)
        ),

        "clusters": int(len(paired))
    }


paired_results = {

    "role_aware_minus_baseline_token_f1":
        cluster_bootstrap_delta(
            qa_results,
            "token_f1",
            seed=20261127
        ),

    "role_aware_minus_baseline_exact_match":
        cluster_bootstrap_delta(
            qa_results,
            "exact_match",
            seed=20261128
        ),

    "role_aware_minus_recency_token_f1":
        cluster_bootstrap_delta(
            qa_results,
            "token_f1",
            method_a="role_aware",
            method_b="recency",
            seed=20261129
        ),

    "baseline_minus_question_only_token_f1":
        cluster_bootstrap_delta(
            qa_results,
            "token_f1",
            method_a="baseline",
            method_b="question_only",
            seed=20261130
        )
}

# ------------------------------------------------------------
# 12. Save all four experiment artifacts
# ------------------------------------------------------------

per_question_path = (
    QA_OUT_DIR /
    "locomo_reader_qa_per_question.csv"
)

summary_path = (
    QA_OUT_DIR /
    "locomo_reader_qa_summary.csv"
)

paired_path = (
    QA_OUT_DIR /
    "locomo_reader_qa_paired_cluster_bootstrap.json"
)

manifest_path = (
    QA_OUT_DIR /
    "locomo_reader_qa_manifest.json"
)

# Privacy-safe exports only.
# No questions, references, prompts, contexts or predictions.
qa_results.to_csv(
    per_question_path,
    index=False
)

method_summary.to_csv(
    summary_path,
    index=False
)

with open(
    paired_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        paired_results,
        f,
        indent=2
    )


manifest = {

    "experiment":
        "MemWeave end-to-end reader QA on LoCoMo",

    "status":
        "exploratory; not independent confirmatory replication",

    "dataset":
        DATA_PROVENANCE.get("locomo", {}),

    "reader_model":
        READER_MODEL_ID,

    "reader_model_revision":
        reader_revision,

    "reader_device":
        str(device),

    "sampling": {

        "seed":
            SAMPLE_SEED,

        "max_questions_per_conversation":
            MAX_QA_PER_CONVERSATION,

        "questions_scored":
            int(qa_results["question_hash"].nunique()),

        "conversation_clusters":
            int(qa_results["conversation_index"].nunique())
    },

    "protocol": {

        "primary_comparison":
            "role_aware vs baseline",

        "controls":
            ["recency", "question_only"],

        "storage_budget_fraction":
            MEMORY_BUDGET_FRACTION,

        "storage_budget_unit":
            "FLAN-T5 tokenizer tokens",

        "retrieval_top_k":
            RETRIEVAL_TOP_K,

        "reader_input_max_tokens":
            READER_INPUT_MAX_TOKENS,

        "reader_max_new_tokens":
            READER_MAX_NEW_TOKENS,

        "generation":
            "greedy deterministic decoding"
    },

    "paired_cluster_bootstrap":
        paired_results,

    "limitations": [

        "LoCoMo has ten conversation clusters.",

        "LoCoMo was previously used for selector-transfer analysis.",

        "FLAN-T5-small is one fixed reader.",

        "Exact match and token F1 do not fully measure factual support.",

        "No LoCoMo fitting or tuning is performed.",

        "Reader input is capped at 512 tokens."
    ]
}

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 13. Print final results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("END-TO-END READER QA COMPLETE")
print("=" * 75)

print("Questions scored:",
      manifest["sampling"]["questions_scored"])

print("Conversation clusters:",
      manifest["sampling"]["conversation_clusters"])

print("\nMETHOD SUMMARY")

display(
    method_summary.round(4)
)

print("\nPAIRED CLUSTER BOOTSTRAP")

print(
    json.dumps(
        paired_results,
        indent=2
    )
)

print("\nSAVED OUTPUT FILES")

for path in [
    per_question_path,
    summary_path,
    paired_path,
    manifest_path
]:

    print(
        path,
        "|",
        f"{path.stat().st_size:,} bytes"
    )

print("\nOutput directory:", QA_OUT_DIR)


AssertionError: Frozen baseline selector is missing.

In [72]:
# ============================================================
# RECOVER FROZEN MEMWEAVE SELECTORS
# ============================================================

required = [
    "DEV",
    "stack_aug",
    "C_FROZEN",
    "BASE_IDX",
    "FULL_IDX"
]

missing = [x for x in required if x not in globals()]

if missing:
    print("Cannot fit frozen selectors yet.")
    print("Missing prerequisites:", missing)
    print(
        "\nExecute Notebook 05 from the top through "
        "the cell titled 'FREEZE MODELS + INDEPENDENT LOCOMO EVALUATION'."
    )
else:
    from sklearn.linear_model import LogisticRegression

    FROZEN_BASELINE = LogisticRegression(
        C=C_FROZEN,
        class_weight="balanced",
        max_iter=500
    )

    Xb, yb, _ = stack_aug(DEV, BASE_IDX)
    FROZEN_BASELINE.fit(Xb, yb)

    FROZEN_ROLE_AWARE = LogisticRegression(
        C=C_FROZEN,
        class_weight="balanced",
        max_iter=500
    )

    Xa, ya, _ = stack_aug(DEV, FULL_IDX)
    FROZEN_ROLE_AWARE.fit(Xa, ya)

    print("Both frozen selectors fitted successfully.")
    print("Baseline:", type(FROZEN_BASELINE).__name__)
    print("Role-aware:", type(FROZEN_ROLE_AWARE).__name__)

Cannot fit frozen selectors yet.
Missing prerequisites: ['stack_aug']

Execute Notebook 05 from the top through the cell titled 'FREEZE MODELS + INDEPENDENT LOCOMO EVALUATION'.


In [73]:
required = [
    "DEV", "TEST", "Q", "qrecs",
    "POOL", "BASE_IDX", "FULL_IDX", "C_FROZEN",
    "stack_aug", "LOCOMO_PATH", "DATA_PROVENANCE",
    "OUT_DIR", "tfidf", "hv", "build_features",
    "session_stats"
]

missing = [name for name in required if name not in globals()]

print("Available:", len(required) - len(missing), "/", len(required))
print("Missing:", missing)

if not missing:
    print("Ready to execute Cell 17.")
else:
    print("Run the original Notebook 05 cells from the beginning through Cell 15.")

Available: 15 / 16
Missing: ['stack_aug']
Run the original Notebook 05 cells from the beginning through Cell 15.


In [77]:
# Verify only the active pool and the question IDs used by the experiment.
print("Active pool:", POOL)
print("DEV questions:", len(DEV))
print("TEST questions:", len(TEST))

active_questions = Q[POOL]

missing_f_aug = [
    qi for qi in list(DEV) + list(TEST)
    if qi not in active_questions or "F_aug" not in active_questions[qi]
]

assert not missing_f_aug, (
    f"F_aug missing for {len(missing_f_aug)} active-pool questions. "
    f"Examples: {missing_f_aug[:10]}"
)

print("SUCCESS: F_aug exists for all active DEV/TEST questions.")

# Now define stack_aug and the related functions.
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/memweavee/MemWeave/notebooks/05_MemWeave_Kaggle_Followup.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

src = nb["cells"][12]["source"]
src = "".join(src) if isinstance(src, list) else src

exec(compile(src, "Notebook05_Cell12", "exec"), globals())

assert callable(stack_aug)
print("SUCCESS: stack_aug is ready.")

Active pool: same_pipeline
DEV questions: 235
TEST questions: 235
SUCCESS: F_aug exists for all active DEV/TEST questions.
baseline 4216 / 5523
full_role_aware 4216 / 5523
leave_out__user_max_question_similarity 4216 / 5523
leave_out__user_top3_mean_question_similarity 4216 / 5523
leave_out__assistant_max_question_similarity 4216 / 5523
leave_out__assistant_top3_mean_question_similarity 4216 / 5523
leave_out__log_user_message_count 4216 / 5523
leave_out__log_assistant_message_count 4216 / 5523
leave_out__user_message_group 4216 / 5523
leave_out__assistant_message_group 4216 / 5523
leave_out__message_similarity_group 4216 / 5523
leave_out__message_count_group 4216 / 5523
SUCCESS: stack_aug is ready.


In [78]:
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/memweavee/MemWeave/notebooks/05_MemWeave_Kaggle_Followup.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

src = nb["cells"][17]["source"]
src = "".join(src) if isinstance(src, list) else src

exec(compile(src, "Notebook05_Cell17", "exec"), globals())

required = [
    "FROZEN_BASELINE",
    "FROZEN_ROLE_AWARE",
    "locomo",
    "role_features_for_sessions",
    "locomo_gold_indices",
    "extract_session_messages",
    "evaluate_locomo_orientation",
]

missing = [name for name in required if name not in globals()]
assert not missing, f"Cell 17 incomplete; missing: {missing}"

print("SUCCESS: frozen models and LoCoMo evaluation helpers are ready.")

Frozen LongMemEval models fitted successfully.
LoCoMo conversations loaded: 10

Running orientation A...

Evaluation diagnostics
----------------------
Processed conversations: 10
Skipped empty conversations: 0
Skipped invalid conversations: 0

Running orientation B...

Evaluation diagnostics
----------------------
Processed conversations: 10
Skipped empty conversations: 0
Skipped invalid conversations: 0

LoCoMo evaluation completed.
Orientation A conversation clusters: 10
Orientation B conversation clusters: 10

Orientation A results:


,conversation_index,n_questions,frozen_baseline,frozen_role_aware,recency,low_novelty,random
0,0,150,0.061667,0.468000,0.110000,0.073889,0.115222
1,1,81,0.024691,0.570988,0.061728,0.024691,0.009259
2,2,152,0.134868,0.576206,0.100329,0.098136,0.093202
3,3,199,0.086404,0.596151,0.129564,0.114824,0.065466
4,4,178,0.089513,0.736102,0.120787,0.082638,0.081661
5,5,123,0.096883,0.620267,0.136334,0.073171,0.107995
6,6,150,0.141556,0.750000,0.111333,0.076667,0.107111
7,7,191,0.061394,0.678322,0.064011,0.051795,0.126839
8,8,156,0.097505,0.564911,0.068429,0.089919,0.111806
9,9,156,0.140491,0.608440,0.093483,0.140491,0.072650



Orientation B results:


,conversation_index,n_questions,frozen_baseline,frozen_role_aware,recency,low_novelty,random
0,0,150,0.061667,0.488000,0.110000,0.073889,0.115222
1,1,81,0.024691,0.549383,0.061728,0.024691,0.009259
2,2,152,0.134868,0.623904,0.100329,0.098136,0.093202
3,3,199,0.086404,0.596151,0.129564,0.114824,0.065466
4,4,178,0.089513,0.723930,0.120787,0.082638,0.081661
5,5,123,0.096883,0.591812,0.136334,0.073171,0.107995
6,6,150,0.141556,0.746667,0.111333,0.076667,0.107111
7,7,191,0.061394,0.678322,0.064011,0.051795,0.126839
8,8,156,0.097505,0.563736,0.068429,0.089919,0.111806
9,9,156,0.140491,0.603098,0.093483,0.140491,0.072650


SUCCESS: frozen models and LoCoMo evaluation helpers are ready.


In [79]:
%pip install -q transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [80]:
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/memweavee/MemWeave/notebooks/05_MemWeave_Kaggle_Followup.ipynb"
)

assert notebook_path.exists(), "Notebook 05 not found."

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

src = nb["cells"][23]["source"]
src = "".join(src) if isinstance(src, list) else src

exec(compile(src, "Notebook05_Cell23_Reader_QA", "exec"), globals())

print("Reader QA cell execution finished.")

Reader: google/flan-t5-small | revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (7 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (680 > 512). Running this sequence through the model will result in indexing errors


Generated 160/800 answers
Generated 320/800 answers
Generated 480/800 answers
Generated 640/800 answers
Generated 800/800 answers

END-TO-END READER QA COMPLETE
Questions scored: 200
Conversation clusters: 10


,method,n,exact_match,token_f1,mean_memory_tokens,mean_reader_context_tokens,mean_selected_sessions
0,baseline,200,0.000,0.0264,1851.90,1470.925,3.500
1,question_only,200,0.005,0.0467,0.00,0.000,0.000
2,recency,200,0.000,0.0273,1814.40,1814.400,2.700
3,role_aware,200,0.000,0.0408,1783.93,1760.120,2.505



Paired conversation-cluster bootstrap:
{
  "role_aware_minus_baseline_token_f1": {
    "delta": 0.01440017263880206,
    "ci95_low": 0.004269774500017758,
    "ci95_high": 0.025263337241165865,
    "clusters": 10
  },
  "role_aware_minus_baseline_exact_match": {
    "delta": 0.0,
    "ci95_low": 0.0,
    "ci95_high": 0.0,
    "clusters": 10
  },
  "role_aware_minus_recency_token_f1": {
    "delta": 0.013564506936455601,
    "ci95_low": 0.001614390596684154,
    "ci95_high": 0.026583068384329916,
    "clusters": 10
  },
  "baseline_minus_question_only_token_f1": {
    "delta": -0.02024124168517962,
    "ci95_low": -0.044152861125584045,
    "ci95_high": 0.00040446115146408363,
    "clusters": 10
  }
}

Saved outputs: /kaggle/working/memweave_followup_outputs/locomo_reader_qa
Reader QA cell execution finished.


In [81]:
import pandas as pd

MAX_TOKENS = READER_INPUT_MAX_TOKENS

audit = []

for row in prompt_rows:
    prompt = row["prompt"]
    marker = "Question:"
    pos = prompt.rfind(marker)

    if pos < 0:
        continue

    prefix_tokens = len(
        reader_tokenizer.encode(
            prompt[:pos],
            add_special_tokens=True
        )
    )

    question_tokens = len(
        reader_tokenizer.encode(
            prompt[pos:],
            add_special_tokens=False
        )
    )

    available = max(0, MAX_TOKENS - prefix_tokens)
    retained = min(available, question_tokens)

    audit.append({
        "method": row.get("method", "unknown"),
        "prompt_tokens": len(
            reader_tokenizer.encode(prompt, add_special_tokens=True)
        ),
        "question_tokens": question_tokens,
        "question_tokens_retained": retained,
        "question_fully_retained": retained >= question_tokens,
        "question_completely_truncated": retained == 0
    })

audit_df = pd.DataFrame(audit)

print("Prompt audit rows:", len(audit_df))
print("Prompts exceeding 512:", (audit_df.prompt_tokens > MAX_TOKENS).sum())

display(
    audit_df.groupby("method").agg(
        prompts=("prompt_tokens", "count"),
        over_512=("prompt_tokens", lambda x: (x > MAX_TOKENS).sum()),
        question_fully_retained=("question_fully_retained", "sum"),
        question_completely_truncated=("question_completely_truncated", "sum"),
        mean_question_retention=(
            "question_tokens_retained",
            "mean"
        )
    )
)

Prompt audit rows: 800
Prompts exceeding 512: 600


,prompts,over_512,question_fully_retained,question_completely_truncated,mean_question_retention
method,,,,,
baseline,200,200,0,200,0.00
question_only,200,0,200,0,16.77
recency,200,200,0,200,0.00
role_aware,200,200,0,200,0.00


In [82]:
%cd /kaggle/working/memweavee
!git pull --ff-only origin research/memweave-followup-release

/kaggle/working/memweavee
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 30 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (30/30), 8.10 KiB | 518.00 KiB/s, done.
From https://github.com/tishaaaaaaaaa5-sketch/memweavee
 * branch            research/memweave-followup-release -> FETCH_HEAD
   8966d8c..fc6b32b  research/memweave-followup-release -> origin/research/memweave-followup-release
Updating 8966d8c..fc6b32b
Fast-forward
 .../notebooks/05_MemWeave_Kaggle_Followup.ipynb    | 22 +++++++-
 .../LOCOMO_END_TO_END_READER_QA_INITIAL_RESULTS.md | 60 ++++++++++++++++++++++
 MemWeave/results/README.md                         |  5 ++
 3 files changed, 85 insertions(+), 2 deletions(-)
 create mode 100644 MemWeave/results/LOCOMO_END_TO_END_READER_QA_INITIAL_RESULTS.md


In [84]:
import shutil
from pathlib import Path

old_dir = Path("/kaggle/working/memweave_followup_outputs/locomo_reader_qa")
archive_dir = old_dir.with_name("locomo_reader_qa_invalid_prompt_run")

if old_dir.exists() and not archive_dir.exists():
    shutil.copytree(old_dir, archive_dir)
    print("Archived previous run:", archive_dir)
else:
    print("Archive already exists or previous output directory is absent.")

Archive already exists or previous output directory is absent.


In [85]:
import json
from pathlib import Path

notebook_path = Path(
    "/kaggle/working/memweavee/MemWeave/notebooks/05_MemWeave_Kaggle_Followup.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

src = nb["cells"][23]["source"]
src = "".join(src) if isinstance(src, list) else src

exec(compile(src, "Notebook05_Cell23_Corrected_Reader_QA", "exec"), globals())

Reader: google/flan-t5-small | revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (680 > 512). Running this sequence through the model will result in indexing errors


Reader device: cuda
Sampled QA questions: 200
Conversation clusters: 10
Questions per conversation: {0: 20, 1: 20, 2: 20, 3: 20, 4: 20, 5: 20, 6: 20, 7: 20, 8: 20, 9: 20}
Generated 160/800 answers
Generated 320/800 answers
Generated 480/800 answers
Generated 640/800 answers
Generated 800/800 answers

END-TO-END READER QA COMPLETE
Questions scored: 200
Conversation clusters: 10


,method,n,exact_match,token_f1,mean_memory_tokens,mean_retrieved_context_tokens_before_pack,mean_reader_context_tokens_after_pack,context_truncation_rate,mean_reader_input_tokens,mean_selected_sessions
0,baseline,200,0.035,0.1015,1851.90,1470.925,459.220,1.0,511.990,3.500
1,question_only,200,0.005,0.0467,0.00,0.000,0.000,0.0,42.770,0.000
2,recency,200,0.010,0.0736,1814.40,1814.400,459.205,1.0,511.975,2.700
3,role_aware,200,0.080,0.2057,1783.93,1760.120,459.205,1.0,511.975,2.505



Paired conversation-cluster bootstrap:
{
  "role_aware_minus_baseline_token_f1": {
    "delta": 0.104149286501796,
    "ci95_low": 0.050744201761349396,
    "ci95_high": 0.15861393652562378,
    "clusters": 10
  },
  "role_aware_minus_baseline_exact_match": {
    "delta": 0.045000000000000005,
    "ci95_low": 0.009999999999999995,
    "ci95_high": 0.08499999999999999,
    "clusters": 10
  },
  "role_aware_minus_recency_token_f1": {
    "delta": 0.13208109127631248,
    "ci95_low": 0.08126264868619652,
    "ci95_high": 0.1765426513450707,
    "clusters": 10
  },
  "baseline_minus_question_only_token_f1": {
    "delta": 0.05482204509917956,
    "ci95_low": 0.017291327830292528,
    "ci95_high": 0.09916581127029984,
    "clusters": 10
  }
}

Saved outputs: /kaggle/working/memweave_followup_outputs/locomo_reader_qa


In [86]:
import pandas as pd

meta_by_id = {
    int(x["sample_id"]): x for x in question_meta
}

audit = []

for row in prompt_rows:
    meta = meta_by_id[int(row["sample_id"])]
    expected_suffix = f"Question: {meta['question']}\nAnswer:"

    prompt = row["prompt"]
    input_tokens = len(
        reader_tokenizer.encode(prompt, add_special_tokens=True)
    )

    audit.append({
        "method": row["method"],
        "question_preserved": prompt.endswith(expected_suffix),
        "input_tokens": input_tokens,
        "within_limit": input_tokens <= 512
    })

audit_df = pd.DataFrame(audit)

print("Total prompts:", len(audit_df))
print("Questions preserved:", audit_df["question_preserved"].sum())
print("Inputs within limit:", audit_df["within_limit"].sum())

display(
    audit_df.groupby("method").agg(
        prompts=("method", "size"),
        questions_preserved=("question_preserved", "sum"),
        within_limit=("within_limit", "sum"),
        max_input_tokens=("input_tokens", "max")
    )
)

assert len(audit_df) == 800
assert audit_df["question_preserved"].all()
assert audit_df["within_limit"].all()

print("PASS: all 800 prompts preserve their questions and fit the reader limit.")

Total prompts: 800
Questions preserved: 800
Inputs within limit: 800


,prompts,questions_preserved,within_limit,max_input_tokens
method,,,,
baseline,200,200,200,512
question_only,200,200,200,61
recency,200,200,200,512
role_aware,200,200,200,512


PASS: all 800 prompts preserve their questions and fit the reader limit.


In [87]:

# ============================================================
# GOLD-EVIDENCE RETENTION AFTER 512-TOKEN CONTEXT PACKING
# Diagnostic only: no model generation or selector retraining.
# ============================================================

import re
import numpy as np
import pandas as pd

assert "sampled" in globals() and len(sampled) == 200
assert "reader_tokenizer" in globals()
assert "FROZEN_BASELINE" in globals()
assert "FROZEN_ROLE_AWARE" in globals()

def _norm_evidence_text(text):
    return re.sub(r"\s+", " ", str(text)).strip().casefold()

audit_rows = []

for sample_id, (conv_idx, qa_idx, item, qa, gold, session_numbers) in enumerate(sampled):

    conv = item["conversation"]
    speaker_a = str(conv.get("speaker_a", "Speaker A"))
    speaker_b = str(conv.get("speaker_b", "Speaker B"))

    # Recover annotated gold session numbers directly.
    gold_numbers = set()
    for evidence in qa.get("evidence", []) or []:
        gold_numbers.update(
            int(x) for x in re.findall(r"D(\d+):", str(evidence))
        )

    if not gold_numbers:
        continue

    sessions = []
    rendered_sessions = []
    word_costs = []
    user_counts = []
    turn_counts = []
    fp_values = []
    digit_values = []
    kept_numbers = []

    for number in session_numbers:
        turns = extract_session_messages(conv, number)

        if not turns:
            continue

        full, words, user_words, nturns, fpv, dgv = session_stats(
            turns, "user"
        )

        rendered = _render_locomo_session(
            turns, speaker_a, speaker_b
        )

        if not rendered.strip():
            continue

        sessions.append(turns)
        rendered_sessions.append(rendered)
        word_costs.append(words)
        user_counts.append(user_words)
        turn_counts.append(nturns)
        fp_values.append(fpv)
        digit_values.append(dgv)
        kept_numbers.append(number)

    if not sessions:
        continue

    question = str(qa["question"]).strip()

    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    days = np.arange(len(sessions), dtype=float)

    Fbase, _ = build_features(
        V,
        np.asarray(word_costs, dtype=float),
        np.asarray(user_counts, dtype=float),
        np.asarray(turn_counts, dtype=float),
        np.asarray(fp_values, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    Frole = role_features_for_sessions(question, sessions)
    Ffull = np.column_stack([Fbase, Frole]).astype(np.float32)

    score_map = {
        "baseline": FROZEN_BASELINE.predict_proba(
            Fbase[:, BASE_IDX]
        )[:, 1],

        "role_aware": FROZEN_ROLE_AWARE.predict_proba(
            Ffull[:, FULL_IDX]
        )[:, 1],

        "recency": Fbase[:, 2]
    }

    token_costs = np.asarray([
        len(reader_tokenizer.encode(
            text, add_special_tokens=False
        ))
        for text in rendered_sessions
    ], dtype=int)

    qvec = tfidf.transform(hv.transform([question]))
    similarities = (V @ qvec.T).toarray().ravel()

    gold_available = gold_numbers.intersection(set(kept_numbers))

    for method, scores in score_map.items():

        selected, _, _ = _select_exact_token_budget(scores, token_costs)

        retrieved = sorted(
            selected,
            key=lambda i: (-float(similarities[i]), i)
        )[:RETRIEVAL_TOP_K]

        retrieved_numbers = {
            kept_numbers[i] for i in retrieved
        }

        context = "\n\n".join(
            rendered_sessions[i] for i in retrieved
        )

        prompt, packed_tokens = _build_reader_prompt(
            question, context
        )

        if "Conversation context:\n" in prompt:
            packed_context = prompt.split(
                "Conversation context:\n", 1
            )[1].rsplit("\n\nQuestion:", 1)[0]
        else:
            packed_context = ""

        packed_norm = _norm_evidence_text(packed_context)

        retrieved_gold = gold_available.intersection(
            retrieved_numbers
        )

        fully_retained = set()
        prefix_retained = set()

        for i in retrieved:
            number = kept_numbers[i]

            if number not in gold_available:
                continue

            session_norm = _norm_evidence_text(
                rendered_sessions[i]
            )

            if session_norm and session_norm in packed_norm:
                fully_retained.add(number)

            # Conservative partial-retention check:
            # does the opening portion of this gold session survive?
            prefix_length = min(80, len(session_norm))

            if prefix_length >= 20:
                prefix = session_norm[:prefix_length]
                if prefix in packed_norm:
                    prefix_retained.add(number)

        denominator = max(1, len(gold_available))

        audit_rows.append({
            "sample_id": sample_id,
            "conversation_index": conv_idx,
            "method": method,
            "gold_sessions_available": len(gold_available),
            "gold_sessions_retrieved": len(retrieved_gold),
            "gold_sessions_fully_packed": len(fully_retained),
            "gold_sessions_partially_packed": len(prefix_retained),
            "retrieved_gold_recall": len(retrieved_gold) / denominator,
            "full_packed_gold_recall": len(fully_retained) / denominator,
            "partial_packed_gold_recall": len(prefix_retained) / denominator,
            "any_gold_retrieved": bool(retrieved_gold),
            "any_gold_fully_packed": bool(fully_retained),
            "any_gold_partially_packed": bool(prefix_retained),
            "reader_context_tokens_after_pack": packed_tokens
        })

gold_audit = pd.DataFrame(audit_rows)

assert not gold_audit.empty
assert set(gold_audit["method"].unique()) == {
    "baseline", "role_aware", "recency"
}

gold_summary = (
    gold_audit.groupby("method")
    .agg(
        questions=("sample_id", "nunique"),
        mean_retrieved_gold_recall=("retrieved_gold_recall", "mean"),
        mean_full_packed_gold_recall=("full_packed_gold_recall", "mean"),
        mean_partial_packed_gold_recall=("partial_packed_gold_recall", "mean"),
        questions_with_any_gold_retrieved=("any_gold_retrieved", "mean"),
        questions_with_any_gold_fully_packed=("any_gold_fully_packed", "mean"),
        questions_with_any_gold_partially_packed=("any_gold_partially_packed", "mean"),
        mean_packed_context_tokens=("reader_context_tokens_after_pack", "mean")
    )
    .reset_index()
)

print("Gold-evidence retention after context packing")
display(gold_summary)

print("\nPer-question diagnostic rows:", len(gold_audit))
print("Conversation clusters:", gold_audit["conversation_index"].nunique())


Gold-evidence retention after context packing


,method,questions,mean_retrieved_gold_recall,mean_full_packed_gold_recall,mean_partial_packed_gold_recall,questions_with_any_gold_retrieved,questions_with_any_gold_fully_packed,questions_with_any_gold_partially_packed,mean_packed_context_tokens
0,baseline,200,0.100833,0.014167,0.086667,0.120,0.020,0.100,459.220
1,recency,200,0.073750,0.001667,0.072083,0.085,0.005,0.085,459.205
2,role_aware,200,0.672667,0.019167,0.511417,0.735,0.025,0.570,459.205



Per-question diagnostic rows: 600
Conversation clusters: 10


In [91]:

# ============================================================
# MEMWEAVE: CONSOLIDATED DOWNSTREAM VALIDATION
# Requires successful execution of Notebook 05 Cells 23 and 24.
#
# Adds:
# 1. Gold-evidence retention under whole-session packing
# 2. Paired prefix-vs-whole-session cluster bootstrap
# 3. Second reader model evaluation
# 4. Lexical evidence-support diagnostic (NOT semantic grounding)
# 5. Exact tokenizer-budget and reader-input audits
# 6. Full eligible-question inventory across available conversations
#
# Does NOT retrain selectors or repeat Cell 23 / Cell 24.
# ============================================================

import os, re, json, time, string, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# 0. REQUIRE EXISTING EXPERIMENTS
# ------------------------------------------------------------

required = [
    "sampled", "qa_results", "whole_qa_results",
    "whole_prompt_rows", "whole_generated",
    "whole_summary", "packing_deltas",
    "reader_tokenizer", "OUT_DIR",
    "_qa_metrics", "_qa_normalize"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        "Run Cells 23 and 24 successfully first. Missing: "
        + ", ".join(missing)
    )

assert len(qa_results) == 800, "Unexpected prefix QA result count."
assert len(whole_qa_results) == 800, "Unexpected whole-session result count."
assert len(whole_prompt_rows) == 800, "Unexpected whole-session prompt count."

FOLLOWUP_DIR = Path(OUT_DIR) / "consolidated_followup"
FOLLOWUP_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20261127
BOOTSTRAP_REPS = 20000
SECOND_READER_ID = "google/flan-t5-base"
MAX_INPUT = 512
MAX_NEW_TOKENS = 64

rng = np.random.default_rng(SEED)

print("Existing prefix questions:", qa_results["question_hash"].nunique())
print("Existing whole-session questions:", whole_qa_results["question_hash"].nunique())
print("Available conversation clusters:",
      whole_qa_results["conversation_index"].nunique())


# ------------------------------------------------------------
# 1. GOLD-EVIDENCE RETENTION
# ------------------------------------------------------------

gold_rows = pd.DataFrame([
    {
        "sample_id": r["sample_id"],
        "method": r["method"],
        "retrieved_gold_recall": r.get("retrieved_gold_recall", np.nan),
        "packed_gold_recall": r.get("packed_gold_recall", np.nan),
        "packed_sessions": r.get("packed_session_count", np.nan),
        "reader_input_tokens": r.get("reader_input_tokens", np.nan)
    }
    for r in whole_prompt_rows
    if r["method"] != "question_only"
])

gold_summary = (
    gold_rows.groupby("method")
    .agg(
        questions=("sample_id", "nunique"),
        mean_retrieved_gold_recall=("retrieved_gold_recall", "mean"),
        mean_packed_gold_recall=("packed_gold_recall", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_reader_input_tokens=("reader_input_tokens", "mean")
    )
    .reset_index()
)

print("\nWHOLE-SESSION GOLD RETENTION")
display(gold_summary.round(4))


# ------------------------------------------------------------
# 2. PAIRED PREFIX VS WHOLE-SESSION CLUSTER BOOTSTRAP
# ------------------------------------------------------------

prefix = qa_results[
    ["conversation_index", "question_hash", "method",
     "exact_match", "token_f1"]
].copy()

whole = whole_qa_results[
    ["conversation_index", "question_hash", "method",
     "exact_match", "token_f1"]
].copy()

paired = prefix.merge(
    whole,
    on=["conversation_index", "question_hash", "method"],
    suffixes=("_prefix", "_whole"),
    validate="one_to_one"
)

assert len(paired) == 800, (
    f"Expected 800 paired rows; got {len(paired)}"
)

paired["token_f1_delta"] = (
    paired["token_f1_whole"] - paired["token_f1_prefix"]
)
paired["exact_match_delta"] = (
    paired["exact_match_whole"] - paired["exact_match_prefix"]
)

def cluster_bootstrap_delta(
    frame, metric_delta, reps=BOOTSTRAP_REPS, seed=SEED
):
    clusters = np.sort(frame["conversation_index"].unique())
    cluster_values = {
        c: frame.loc[
            frame["conversation_index"] == c, metric_delta
        ].mean()
        for c in clusters
    }

    values = np.asarray([cluster_values[c] for c in clusters], dtype=float)
    local_rng = np.random.default_rng(seed)

    draws = local_rng.integers(
        0, len(clusters), size=(reps, len(clusters))
    )
    boot = values[draws].mean(axis=1)

    return {
        "mean_delta": float(values.mean()),
        "ci_2.5": float(np.quantile(boot, 0.025)),
        "ci_97.5": float(np.quantile(boot, 0.975)),
        "clusters": int(len(clusters)),
        "bootstrap_reps": int(reps)
    }

packing_bootstrap = []

for method in sorted(paired["method"].unique()):
    sub = paired[paired["method"] == method]

    for metric in ["token_f1_delta", "exact_match_delta"]:
        result = cluster_bootstrap_delta(sub, metric)
        packing_bootstrap.append({
            "method": method,
            "metric": metric,
            **result
        })

packing_bootstrap = pd.DataFrame(packing_bootstrap)

print("\nPAIRED PREFIX VS WHOLE-SESSION COMPARISON")
display(
    paired.groupby("method")
    .agg(
        n=("token_f1_delta", "size"),
        prefix_f1=("token_f1_prefix", "mean"),
        whole_f1=("token_f1_whole", "mean"),
        f1_delta=("token_f1_delta", "mean"),
        prefix_em=("exact_match_prefix", "mean"),
        whole_em=("exact_match_whole", "mean"),
        em_delta=("exact_match_delta", "mean")
    )
    .reset_index()
    .round(4)
)

print("\nCONVERSATION-CLUSTER BOOTSTRAP")
display(packing_bootstrap.round(5))


# ------------------------------------------------------------
# 3. SECOND READER: FLAN-T5-BASE
# ------------------------------------------------------------

# FLAN-T5-small and FLAN-T5-base use the T5 tokenizer family.
# Verify vocabulary compatibility before reusing the existing prompts.

print("\nLoading second reader:", SECOND_READER_ID)

second_tokenizer = AutoTokenizer.from_pretrained(SECOND_READER_ID)

small_vocab = reader_tokenizer.get_vocab()
second_vocab = second_tokenizer.get_vocab()

assert small_vocab == second_vocab, (
    "Tokenizer vocabularies differ. Do not reuse the existing "
    "512-token prompts without rebuilding and auditing them."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

second_model = AutoModelForSeq2SeqLM.from_pretrained(
    SECOND_READER_ID
).to(device)
second_model.eval()

second_revision = None
try:
    from huggingface_hub import HfApi
    second_revision = HfApi().model_info(SECOND_READER_ID).sha
except Exception as exc:
    print("WARNING: Could not resolve model revision:", str(exc))

# Evaluate the same 800 whole-session prompts:
# baseline, role-aware, recency and question-only.
second_generated = []

batch_size = 8 if device.type == "cuda" else 2

for start in range(0, len(whole_prompt_rows), batch_size):
    batch = whole_prompt_rows[start:start + batch_size]

    encoded = second_tokenizer(
        [r["prompt"] for r in batch],
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to(device)

    assert encoded["input_ids"].shape[1] <= MAX_INPUT, (
        "Second-reader input exceeded 512 tokens."
    )

    with torch.inference_mode():
        output_ids = second_model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1
        )

    predictions = second_tokenizer.batch_decode(
        output_ids, skip_special_tokens=True
    )

    for row, prediction in zip(batch, predictions):
        second_generated.append({
            "sample_id": int(row["sample_id"]),
            "method": row["method"],
            "prediction": str(prediction).strip(),
            "reader_input_tokens": int(
                encoded["input_ids"].shape[1]
            )
        })

    if (start // batch_size + 1) % 20 == 0:
        print(
            f"Second reader generated "
            f"{min(start + batch_size, len(whole_prompt_rows))}/"
            f"{len(whole_prompt_rows)}"
        )

meta_by_id = {
    int(x["sample_id"]): x for x in whole_meta
}

second_scored = []

for row in second_generated:
    meta = meta_by_id[row["sample_id"]]
    em, f1 = _qa_metrics(row["prediction"], meta["answer"])

    second_scored.append({
        "sample_id": row["sample_id"],
        "conversation_index": int(meta["conversation_index"]),
        "question_hash": meta["question_hash"],
        "method": row["method"],
        "exact_match": em,
        "token_f1": f1,
        "prediction": row["prediction"]
    })

second_qa_results = pd.DataFrame(second_scored)

second_summary = (
    second_qa_results.groupby("method")
    .agg(
        n=("token_f1", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean")
    )
    .reset_index()
)

print("\nSECOND READER RESULTS: FLAN-T5-BASE")
display(second_summary.round(4))


# ------------------------------------------------------------
# 4. LEXICAL EVIDENCE-SUPPORT PROXY
# ------------------------------------------------------------

# This is NOT an entailment or semantic-grounding score.
# It measures the fraction of non-stopword answer tokens that
# also appear in the packed conversation context.
# It must not be described as proof that an answer is supported.

STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was",
    "were", "be", "been", "being", "to", "of", "in", "on",
    "at", "for", "from", "with", "by", "as", "it", "this",
    "that", "he", "she", "they", "them", "his", "her", "their",
    "i", "we", "you", "me", "my", "our", "your", "what",
    "when", "where", "who", "why", "how", "did", "does",
    "do", "has", "have", "had", "not", "no", "yes"
}

def lexical_tokens(text):
    tokens = re.findall(r"[a-z0-9]+", str(text).lower())
    return [t for t in tokens if t not in STOPWORDS]

def extract_context(prompt):
    marker = "Conversation context:\n"
    end_marker = "\n\nQuestion:"
    if marker not in prompt or end_marker not in prompt:
        return ""
    return prompt.split(marker, 1)[1].split(end_marker, 1)[0]

prompt_by_id_method = {
    (int(r["sample_id"]), r["method"]): r["prompt"]
    for r in whole_prompt_rows
}

support_rows = []

for row in second_generated:
    prompt = prompt_by_id_method[
        (int(row["sample_id"]), row["method"])
    ]
    context_tokens = set(lexical_tokens(extract_context(prompt)))
    answer_tokens = lexical_tokens(row["prediction"])

    if answer_tokens and context_tokens:
        overlap = sum(t in context_tokens for t in answer_tokens)
        support_ratio = overlap / len(answer_tokens)
    else:
        support_ratio = np.nan

    support_rows.append({
        "sample_id": int(row["sample_id"]),
        "method": row["method"],
        "lexical_context_token_support": support_ratio,
        "answer_nonstopword_tokens": len(answer_tokens),
        "context_nonstopword_types": len(context_tokens)
    })

support_df = pd.DataFrame(support_rows)

support_summary = (
    support_df.groupby("method")
    .agg(
        n=("sample_id", "size"),
        mean_lexical_support=(
            "lexical_context_token_support", "mean"
        ),
        median_lexical_support=(
            "lexical_context_token_support", "median"
        ),
        mean_answer_content_tokens=(
            "answer_nonstopword_tokens", "mean"
        )
    )
    .reset_index()
)

print("\nLEXICAL EVIDENCE-SUPPORT PROXY")
print("WARNING: lexical overlap is not semantic entailment or factual grounding.")
display(support_summary.round(4))


# ------------------------------------------------------------
# 5. EXACT TOKEN-BUDGET AND INPUT-LIMIT AUDIT
# ------------------------------------------------------------

# The storage budget was already computed using exact FLAN-T5-small
# tokenizer counts in Cell 23/24. Check that it remains comparable.
# Question-only is a no-memory control, not a matched-budget method.

budget_rows = []

for row in whole_prompt_rows:
    if row["method"] == "question_only":
        continue

    budget_rows.append({
        "sample_id": int(row["sample_id"]),
        "method": row["method"],
        "selected_memory_tokens": int(row["memory_tokens_selected"]),
        "memory_token_budget": int(row["memory_token_budget"]),
        "reader_input_tokens": int(row["reader_input_tokens"]),
        "within_reader_limit": (
            int(row["reader_input_tokens"]) <= MAX_INPUT
        )
    })

budget_df = pd.DataFrame(budget_rows)

budget_summary = (
    budget_df.groupby("method")
    .agg(
        n=("sample_id", "size"),
        mean_selected_memory_tokens=(
            "selected_memory_tokens", "mean"
        ),
        mean_memory_budget=("memory_token_budget", "mean"),
        mean_reader_input_tokens=("reader_input_tokens", "mean"),
        max_reader_input_tokens=("reader_input_tokens", "max"),
        all_inputs_within_limit=("within_reader_limit", "all")
    )
    .reset_index()
)

print("\nEXACT TOKEN-BUDGET AUDIT")
display(budget_summary.round(3))

assert budget_df["within_reader_limit"].all()
assert len(budget_df) == 600

# Verify that memory-budget accounting matches the original
# prefix and whole-session experiments.
prefix_budget = qa_results[
    ["question_hash", "method", "memory_tokens_selected",
     "memory_token_budget"]
].copy()

whole_budget = pd.DataFrame([
    {
        "question_hash": whole_meta_by_id[int(r["sample_id"])]["question_hash"],
        "method": r["method"],
        "memory_tokens_selected": r["memory_tokens_selected"],
        "memory_token_budget": r["memory_token_budget"]
    }
    for r in whole_prompt_rows if r["method"] != "question_only"
])

budget_pair = prefix_budget.merge(
    whole_budget,
    on=["question_hash", "method"],
    suffixes=("_prefix", "_whole"),
    validate="one_to_one"
)

budget_pair["memory_tokens_equal"] = (
    budget_pair["memory_tokens_selected_prefix"]
    == budget_pair["memory_tokens_selected_whole"]
)

budget_pair["budget_equal"] = (
    budget_pair["memory_token_budget_prefix"]
    == budget_pair["memory_token_budget_whole"]
)

print("\nPREFIX VS WHOLE MEMORY-BUDGET CONSISTENCY")
print("Selected-token counts equal:",
      bool(budget_pair["memory_tokens_equal"].all()))
print("Budget caps equal:",
      bool(budget_pair["budget_equal"].all()))

assert budget_pair["memory_tokens_equal"].all()
assert budget_pair["budget_equal"].all()


# ------------------------------------------------------------
# 6. INVENTORY ALL AVAILABLE ELIGIBLE QUESTIONS
# ------------------------------------------------------------

# The current dataset has ten conversations. This inventories all
# answerable, evidence-annotated questions, including questions
# outside the original 20-per-conversation sample.
# It does NOT claim additional independent clusters.

eligible_inventory = []

for conv_idx, item in enumerate(locomo):
    conv = item.get("conversation", {})
    session_numbers = sorted(
        int(k.split("_")[1])
        for k in conv
        if re.fullmatch(r"session_\d+", k)
    )

    for qa_idx, qa in enumerate(item.get("qa", []) or []):
        if not isinstance(qa, dict) or qa.get("category") == 5:
            continue

        question = str(qa.get("question", "")).strip()
        answer = qa.get("answer", "")

        if not question or answer is None or not str(answer).strip():
            continue

        gold = locomo_gold_indices(qa, session_numbers)
        if not gold:
            continue

        eligible_inventory.append({
            "conversation_index": int(conv_idx),
            "qa_index": int(qa_idx),
            "question_hash": hashlib.sha256(
                f"{conv_idx}|{qa_idx}|{question}".encode()
            ).hexdigest()[:20],
            "category": str(qa.get("category", "unknown")),
            "gold_session_count": len(gold)
        })

inventory_df = pd.DataFrame(eligible_inventory)

sampled_hashes = set(qa_results["question_hash"].unique())
inventory_df["in_existing_sample"] = (
    inventory_df["question_hash"].isin(sampled_hashes)
)

print("\nAVAILABLE DATASET INVENTORY")
print("All eligible questions:", len(inventory_df))
print("Already in reader sample:", int(inventory_df["in_existing_sample"].sum()))
print("Additional eligible questions:",
      int((~inventory_df["in_existing_sample"]).sum()))
print("Independent conversation clusters:",
      inventory_df["conversation_index"].nunique())

print(
    "NOTE: Additional questions from the same ten conversations "
    "increase question coverage, not the number of independent clusters."
)


# ------------------------------------------------------------
# 7. SAVE ALL FOLLOW-UP ARTIFACTS
# ------------------------------------------------------------

gold_summary.to_csv(
    FOLLOWUP_DIR / "whole_session_gold_retention.csv", index=False
)

paired.to_csv(
    FOLLOWUP_DIR / "prefix_vs_whole_paired_questions.csv", index=False
)

packing_bootstrap.to_csv(
    FOLLOWUP_DIR / "prefix_vs_whole_cluster_bootstrap.csv", index=False
)

second_qa_results.to_csv(
    FOLLOWUP_DIR / "second_reader_per_question.csv", index=False
)

second_summary.to_csv(
    FOLLOWUP_DIR / "second_reader_summary.csv", index=False
)

support_df.to_csv(
    FOLLOWUP_DIR / "lexical_evidence_support_per_question.csv", index=False
)

support_summary.to_csv(
    FOLLOWUP_DIR / "lexical_evidence_support_summary.csv", index=False
)

budget_summary.to_csv(
    FOLLOWUP_DIR / "exact_token_budget_audit.csv", index=False
)

inventory_df.to_csv(
    FOLLOWUP_DIR / "eligible_question_inventory.csv", index=False
)

manifest = {
    "experiment": "MemWeave consolidated downstream validation",
    "status": "exploratory",
    "seed": SEED,
    "bootstrap_repetitions": BOOTSTRAP_REPS,
    "second_reader": SECOND_READER_ID,
    "second_reader_revision": second_revision,
    "second_reader_tokenizer_vocab_identical": True,
    "reader_input_limit": MAX_INPUT,
    "prefix_questions": int(qa_results["question_hash"].nunique()),
    "whole_session_questions": int(
        whole_qa_results["question_hash"].nunique()
    ),
    "conversation_clusters": int(
        whole_qa_results["conversation_index"].nunique()
    ),
    "eligible_questions_in_dataset": int(len(inventory_df)),
    "additional_questions_outside_original_sample": int(
        (~inventory_df["in_existing_sample"]).sum()
    ),
    "lexical_support_warning": (
        "Lexical token overlap is only a surface-form diagnostic; "
        "it is not semantic entailment, factual grounding, or a "
        "validated hallucination metric."
    ),
    "limitations": [
        "Only ten independent LoCoMo conversation clusters are available.",
        "More questions from the same conversations do not create independent clusters.",
        "The second reader uses the same prompt and whole-session packing policy.",
        "Lexical overlap cannot establish semantic evidence support.",
        "A separate NLI or human-adjudicated grounding evaluation is still needed.",
        "Results remain exploratory and do not establish cross-dataset generalization."
    ]
}

with open(FOLLOWUP_DIR / "consolidated_followup_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 65)
print("CONSOLIDATED FOLLOW-UP COMPLETE")
print("=" * 65)
print("Artifacts saved to:", FOLLOWUP_DIR)
print("Second reader:", SECOND_READER_ID)
print("Conversation clusters:", manifest["conversation_clusters"])
print("Eligible questions in dataset:", manifest["eligible_questions_in_dataset"])
print("Additional questions available:", manifest["additional_questions_outside_original_sample"])
print("\nImportant:")
print("- Cluster bootstrap is exploratory with only ten clusters.")
print("- Lexical support is not a semantic groundedness score.")
print("- A new independent dataset is still needed for stronger validation.")


Existing prefix questions: 200
Existing whole-session questions: 200
Available conversation clusters: 10

WHOLE-SESSION GOLD RETENTION


,method,questions,mean_retrieved_gold_recall,mean_packed_gold_recall,mean_packed_sessions,mean_reader_input_tokens
0,baseline,200,0.1008,0.0142,0.445,230.465
1,recency,200,0.0738,0.0017,0.385,196.565
2,role_aware,200,0.6727,0.0192,0.210,131.285



PAIRED PREFIX VS WHOLE-SESSION COMPARISON


,method,n,prefix_f1,whole_f1,f1_delta,prefix_em,whole_em,em_delta
0,baseline,200,0.1015,0.0424,-0.0591,0.035,0.005,-0.030
1,question_only,200,0.0467,0.0467,0.0000,0.005,0.005,0.000
2,recency,200,0.0736,0.0366,-0.0369,0.010,0.005,-0.005
3,role_aware,200,0.2057,0.0486,-0.1571,0.080,0.010,-0.070



CONVERSATION-CLUSTER BOOTSTRAP


,method,metric,mean_delta,ci_2.5,ci_97.5,clusters,bootstrap_reps
0,baseline,token_f1_delta,-0.05909,-0.09872,-0.03073,10,20000
1,baseline,exact_match_delta,-0.03000,-0.07000,0.00000,10,20000
2,question_only,token_f1_delta,0.00000,0.00000,0.00000,10,20000
3,question_only,exact_match_delta,0.00000,0.00000,0.00000,10,20000
4,recency,token_f1_delta,-0.03695,-0.06302,-0.01179,10,20000
5,recency,exact_match_delta,-0.00500,-0.02000,0.01000,10,20000
6,role_aware,token_f1_delta,-0.15709,-0.20037,-0.11048,10,20000
7,role_aware,exact_match_delta,-0.07000,-0.10000,-0.04000,10,20000


... (13 outputs left out)

,method,n,mean_lexical_support,median_lexical_support,mean_answer_content_tokens
0,baseline,200,0.6709,0.6667,2.820
1,question_only,200,NaN,NaN,2.970
2,recency,200,0.6765,0.6667,2.735
3,role_aware,200,0.6940,1.0000,2.965



EXACT TOKEN-BUDGET AUDIT


,method,n,mean_selected_memory_tokens,mean_memory_budget,mean_reader_input_tokens,max_reader_input_tokens,all_inputs_within_limit
0,baseline,200,1851.90,1955.3,230.465,512,True
1,recency,200,1814.40,1955.3,196.565,512,True
2,role_aware,200,1783.93,1955.3,131.285,512,True



PREFIX VS WHOLE MEMORY-BUDGET CONSISTENCY
Selected-token counts equal: True
Budget caps equal: True

AVAILABLE DATASET INVENTORY
All eligible questions: 1536
Already in reader sample: 200
Additional eligible questions: 1336
Independent conversation clusters: 10
NOTE: Additional questions from the same ten conversations increase question coverage, not the number of independent clusters.

CONSOLIDATED FOLLOW-UP COMPLETE
Artifacts saved to: /kaggle/working/memweave_followup_outputs/consolidated_followup
Second reader: google/flan-t5-base
Conversation clusters: 10
Eligible questions in dataset: 1536
Additional questions available: 1336

Important:
- Cluster bootstrap is exploratory with only ten clusters.
- Lexical support is not a semantic groundedness score.
- A new independent dataset is still needed for stronger validation.


In [89]:
required = [
    "sampled",
    "reader_model",
    "reader_tokenizer",
    "qa_results",
    "whole_qa_results",
    "whole_prompt_rows",
    "whole_generated",
    "whole_summary",
    "packing_deltas"
]

for name in required:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

sampled: AVAILABLE
reader_model: AVAILABLE
reader_tokenizer: AVAILABLE
qa_results: AVAILABLE
whole_qa_results: MISSING
whole_prompt_rows: MISSING
whole_generated: MISSING
whole_summary: MISSING
packing_deltas: MISSING


In [90]:

# ============================================================
# CELL 24: WHOLE-SESSION PACKING FOLLOW-UP
# Compare complete-session packing against prefix packing.
# Keeps sample, selector scores, reader, and decoding fixed.
# ============================================================

import json
import hashlib
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert "sampled" in globals() and len(sampled) == 200
assert "reader_model" in globals() and "reader_tokenizer" in globals()
assert "qa_results" in globals(), "Run Cell 23 first."
assert len(qa_results) == 800

WHOLE_MAX_INPUT = 512
WHOLE_TOP_K = 3

WHOLE_OUT_DIR = Path(OUT_DIR) / "locomo_reader_qa_whole_session"
WHOLE_OUT_DIR.mkdir(parents=True, exist_ok=True)


def _whole_prompt(question, context):
    if not context:
        return (
            "Answer the question. No conversation context is available. "
            "If you cannot determine the answer, say: I don't know.\n\n"
            f"Question: {question}\nAnswer:"
        )

    return (
        "Answer the question using only the conversation context. "
        "Give a concise answer. If the context does not contain the answer, "
        "say: I don't know.\n\n"
        f"Conversation context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )


def _pack_complete_sessions(question, ranked_session_texts):
    # Greedily add complete sessions in descending query-similarity order.
    # Skip sessions that do not fit; continue checking later sessions.
    chosen = []

    for session_text in ranked_session_texts:
        candidate_context = "\n\n".join(chosen + [session_text])
        candidate_prompt = _whole_prompt(question, candidate_context)

        candidate_len = len(
            reader_tokenizer.encode(
                candidate_prompt,
                add_special_tokens=True
            )
        )

        if candidate_len <= WHOLE_MAX_INPUT:
            chosen.append(session_text)

    packed_context = "\n\n".join(chosen)
    prompt = _whole_prompt(question, packed_context)

    input_tokens = len(
        reader_tokenizer.encode(
            prompt,
            add_special_tokens=True
        )
    )

    assert input_tokens <= WHOLE_MAX_INPUT
    assert prompt.endswith(f"Question: {question}\nAnswer:")

    return prompt, chosen, input_tokens


# ------------------------------------------------------------
# 1. Construct whole-session prompts
# ------------------------------------------------------------

whole_prompt_rows = []
whole_meta = []

for sample_id, (
    conv_idx, qa_idx, item, qa, gold, session_numbers
) in enumerate(sampled):

    conv = item["conversation"]

    speaker_a = str(conv.get("speaker_a", "Speaker A"))
    speaker_b = str(conv.get("speaker_b", "Speaker B"))

    sessions = []
    rendered_sessions = []

    word_costs = []
    user_counts = []
    turn_counts = []
    fp_values = []
    digit_values = []
    kept_numbers = []

    for number in session_numbers:
        turns = extract_session_messages(conv, number)

        if not turns:
            continue

        full, words, user_words, nturns, fpv, dgv = (
            session_stats(turns, "user")
        )

        rendered = _render_locomo_session(
            turns, speaker_a, speaker_b
        )

        if not rendered.strip():
            continue

        sessions.append(turns)
        rendered_sessions.append(rendered)

        word_costs.append(words)
        user_counts.append(user_words)
        turn_counts.append(nturns)
        fp_values.append(fpv)
        digit_values.append(dgv)
        kept_numbers.append(number)

    if not sessions:
        continue

    question = str(qa["question"]).strip()
    answers = _qa_refs(qa.get("answer", ""))

    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    days = np.arange(len(sessions), dtype=float)

    Fbase, _ = build_features(
        V,
        np.asarray(word_costs, dtype=float),
        np.asarray(user_counts, dtype=float),
        np.asarray(turn_counts, dtype=float),
        np.asarray(fp_values, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    Frole = role_features_for_sessions(question, sessions)
    Ffull = np.column_stack([Fbase, Frole]).astype(np.float32)

    score_map = {
        "baseline": FROZEN_BASELINE.predict_proba(
            Fbase[:, BASE_IDX]
        )[:, 1],

        "role_aware": FROZEN_ROLE_AWARE.predict_proba(
            Ffull[:, FULL_IDX]
        )[:, 1],

        "recency": Fbase[:, 2]
    }

    token_costs = np.asarray([
        len(reader_tokenizer.encode(
            text, add_special_tokens=False
        ))
        for text in rendered_sessions
    ], dtype=int)

    qvec = tfidf.transform(hv.transform([question]))
    similarities = (V @ qvec.T).toarray().ravel()

    whole_meta.append({
        "sample_id": sample_id,
        "conversation_index": int(conv_idx),
        "question_hash": hashlib.sha256(
            f"{conv_idx}|{qa_idx}|{question}".encode("utf-8")
        ).hexdigest()[:20],
        "answer": answers,
        "category": str(qa.get("category", "unknown"))
    })

    for method, scores in score_map.items():

        selected, memory_used, memory_cap = (
            _select_exact_token_budget(scores, token_costs)
        )

        retrieved = sorted(
            selected,
            key=lambda i: (-float(similarities[i]), i)
        )[:WHOLE_TOP_K]

        ranked_texts = [
            rendered_sessions[i] for i in retrieved
        ]

        prompt, packed_sessions, input_tokens = (
            _pack_complete_sessions(question, ranked_texts)
        )

        gold_numbers = set()

        for evidence in qa.get("evidence", []) or []:
            gold_numbers.update(
                int(x)
                for x in re.findall(r"D(\d+):", str(evidence))
            )

        retrieved_numbers = {
            kept_numbers[i] for i in retrieved
        }

        packed_numbers = {
            kept_numbers[i]
            for i in retrieved
            if rendered_sessions[i] in packed_sessions
        }

        available_gold = gold_numbers.intersection(
            set(kept_numbers)
        )

        whole_prompt_rows.append({
            "sample_id": sample_id,
            "method": method,
            "prompt": prompt,
            "memory_tokens_selected": int(memory_used),
            "memory_token_budget": int(memory_cap),
            "retrieved_session_count": int(len(retrieved)),
            "packed_session_count": int(len(packed_sessions)),
            "reader_input_tokens": int(input_tokens),

            "retrieved_gold_recall": (
                len(available_gold.intersection(retrieved_numbers))
                / max(1, len(available_gold))
            ),

            "packed_gold_recall": (
                len(available_gold.intersection(packed_numbers))
                / max(1, len(available_gold))
            )
        })

    # Question-only comparison
    qonly = _whole_prompt(question, "")

    whole_prompt_rows.append({
        "sample_id": sample_id,
        "method": "question_only",
        "prompt": qonly,
        "memory_tokens_selected": 0,
        "memory_token_budget": 0,
        "retrieved_session_count": 0,
        "packed_session_count": 0,
        "reader_input_tokens": len(
            reader_tokenizer.encode(
                qonly, add_special_tokens=True
            )
        ),
        "retrieved_gold_recall": 0.0,
        "packed_gold_recall": 0.0
    })


assert len(whole_prompt_rows) == 800, (
    f"Expected 800 prompts, got {len(whole_prompt_rows)}"
)


# ------------------------------------------------------------
# 2. Generate reader predictions
# ------------------------------------------------------------

batch_size = 8 if device.type == "cuda" else 4
whole_generated = []

for start in range(0, len(whole_prompt_rows), batch_size):

    batch = whole_prompt_rows[start:start + batch_size]

    encoded = reader_tokenizer(
        [x["prompt"] for x in batch],
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to(device)

    assert encoded["input_ids"].shape[1] <= WHOLE_MAX_INPUT

    with torch.inference_mode():
        output_ids = reader_model.generate(
            **encoded,
            max_new_tokens=READER_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1
        )

    predictions = reader_tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )

    for row, prediction in zip(batch, predictions):
        whole_generated.append({
            **{k: v for k, v in row.items() if k != "prompt"},
            "prediction": str(prediction).strip()
        })


# ------------------------------------------------------------
# 3. Score predictions
# ------------------------------------------------------------

whole_meta_by_id = {
    x["sample_id"]: x for x in whole_meta
}

whole_scored = []

for row in whole_generated:

    meta = whole_meta_by_id[row["sample_id"]]

    em, f1 = _qa_metrics(
        row["prediction"],
        meta["answer"]
    )

    whole_scored.append({
        "sample_id": row["sample_id"],
        "conversation_index": meta["conversation_index"],
        "question_hash": meta["question_hash"],
        "category": meta["category"],
        "method": row["method"],
        "exact_match": em,
        "token_f1": f1,
        "reader_input_tokens": row["reader_input_tokens"],
        "packed_session_count": row["packed_session_count"],
        "retrieved_gold_recall": row["retrieved_gold_recall"],
        "packed_gold_recall": row["packed_gold_recall"]
    })


whole_qa_results = pd.DataFrame(whole_scored)

whole_summary = (
    whole_qa_results.groupby("method")
    .agg(
        n=("token_f1", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_input_tokens=("reader_input_tokens", "mean"),
        mean_packed_sessions=("packed_session_count", "mean"),
        mean_retrieved_gold_recall=("retrieved_gold_recall", "mean"),
        mean_packed_gold_recall=("packed_gold_recall", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Paired prefix vs whole-session comparison
# ------------------------------------------------------------

prefix = qa_results[
    [
        "conversation_index",
        "question_hash",
        "method",
        "exact_match",
        "token_f1"
    ]
].copy()

whole = whole_qa_results[
    [
        "conversation_index",
        "question_hash",
        "method",
        "exact_match",
        "token_f1"
    ]
].copy()

comparison = prefix.merge(
    whole,
    on=[
        "conversation_index",
        "question_hash",
        "method"
    ],
    suffixes=("_prefix", "_whole"),
    validate="one_to_one"
)

packing_deltas = (
    comparison.groupby("method")
    .agg(
        n=("token_f1_prefix", "size"),
        prefix_token_f1=("token_f1_prefix", "mean"),
        whole_session_token_f1=("token_f1_whole", "mean"),
        token_f1_delta=("token_f1_whole", lambda x: np.nan)
    )
    .reset_index()
)

for idx, method in enumerate(packing_deltas["method"]):

    sub = comparison[comparison["method"] == method]

    packing_deltas.loc[idx, "token_f1_delta"] = (
        sub["token_f1_whole"].mean()
        - sub["token_f1_prefix"].mean()
    )


# ------------------------------------------------------------
# 5. Display and save results
# ------------------------------------------------------------

print("Whole-session packing QA summary")
display(whole_summary)

print("\nPrefix vs whole-session packing")
display(packing_deltas)

whole_qa_results.to_csv(
    WHOLE_OUT_DIR /
    "locomo_reader_qa_whole_session_per_question.csv",
    index=False
)

whole_summary.to_csv(
    WHOLE_OUT_DIR /
    "locomo_reader_qa_whole_session_summary.csv",
    index=False
)

packing_deltas.to_csv(
    WHOLE_OUT_DIR /
    "locomo_reader_qa_packing_comparison.csv",
    index=False
)

with open(
    WHOLE_OUT_DIR /
    "locomo_reader_qa_whole_session_manifest.json",
    "w"
) as f:

    json.dump({
        "experiment": "LoCoMo whole-session context packing",
        "status": "exploratory",
        "reader_model": READER_MODEL_ID,
        "reader_model_revision": reader_revision,
        "sample_seed": SAMPLE_SEED,
        "questions": int(
            whole_qa_results["question_hash"].nunique()
        ),
        "conversation_clusters": int(
            whole_qa_results["conversation_index"].nunique()
        ),
        "reader_input_max_tokens": WHOLE_MAX_INPUT,
        "retrieval_top_k": WHOLE_TOP_K,
        "packing": (
            "Greedy complete sessions in descending query similarity; "
            "skip sessions that do not fit."
        ),
        "generation": "Greedy deterministic decoding",
        "note": (
            "Selector scores, sample, reader, and generation settings "
            "match the prefix-packing experiment."
        )
    }, f, indent=2)


print("\nSaved artifacts to:", WHOLE_OUT_DIR)
print("PASS: whole-session prompts preserve questions and stay within 512 tokens.")
print("Generated prompts:", len(whole_generated))
print("Scored rows:", len(whole_qa_results))


Whole-session packing QA summary


,method,n,exact_match,token_f1,mean_reader_input_tokens,mean_packed_sessions,mean_retrieved_gold_recall,mean_packed_gold_recall
0,baseline,200,0.005,0.042409,230.465,0.445,0.100833,0.014167
1,question_only,200,0.005,0.046679,42.770,0.000,0.000000,0.000000
2,recency,200,0.005,0.036622,196.565,0.385,0.073750,0.001667
3,role_aware,200,0.010,0.048561,131.285,0.210,0.672667,0.019167



Prefix vs whole-session packing


,method,n,prefix_token_f1,whole_session_token_f1,token_f1_delta
0,baseline,200,0.101501,0.042409,-0.059092
1,question_only,200,0.046679,0.046679,0.000000
2,recency,200,0.073569,0.036622,-0.036947
3,role_aware,200,0.205651,0.048561,-0.157090



Saved artifacts to: /kaggle/working/memweave_followup_outputs/locomo_reader_qa_whole_session
PASS: whole-session prompts preserve questions and stay within 512 tokens.
Generated prompts: 800
Scored rows: 800


In [92]:

# ============================================================
# MEMWEAVE: EVIDENCE-AWARE SPAN PACKING
# Reuses frozen selectors, sampled questions and loaded reader.
# No gold evidence or answers are used to select spans.
# ============================================================

import re, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# ---------- 1. Validate existing notebook state ----------

required = [
    "sampled", "locomo", "reader_model", "reader_tokenizer",
    "FROZEN_BASELINE", "FROZEN_ROLE_AWARE",
    "role_features_for_sessions", "extract_session_messages",
    "session_stats", "_render_locomo_session",
    "_select_exact_token_budget", "_qa_metrics",
    "tfidf", "hv", "build_features", "device"
]
missing = [x for x in required if x not in globals()]
assert not missing, f"Missing notebook variables: {missing}"

MAX_INPUT = 512
TOP_K = 3
MAX_NEW = int(globals().get("READER_MAX_NEW_TOKENS", 64))
OUT = Path(OUT_DIR) / "locomo_reader_qa_evidence_span"
OUT.mkdir(parents=True, exist_ok=True)

print("Reader:", globals().get("READER_MODEL_ID", "currently loaded model"))
print("Questions:", len(sampled))
print("Conversation clusters:", len(set(x[0] for x in sampled)))

# ---------- 2. Prompt and span construction ----------

def span_prompt(question, context):
    if context:
        return (
            "Answer the question using only the conversation evidence. "
            "Give a concise answer. If the evidence does not contain the answer, "
            "say: I don't know.\n\n"
            f"Conversation evidence:\n{context}\n\n"
            f"Question: {question}\nAnswer:"
        )
    return (
        "Answer the question. No conversation evidence is available. "
        "If you cannot determine the answer, say: I don't know.\n\n"
        f"Question: {question}\nAnswer:"
    )


def split_message_into_spans(text):
    """Sentence-level spans; split unusually long sentences into token chunks."""
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text)
    output = []

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        ids = reader_tokenizer.encode(
            sentence, add_special_tokens=False
        )

        if len(ids) <= 120:
            output.append(sentence)
        else:
            for start in range(0, len(ids), 100):
                chunk = reader_tokenizer.decode(
                    ids[start:start + 100],
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                ).strip()
                if chunk:
                    output.append(chunk)

    return output


def exact_input_length(question, context):
    return len(reader_tokenizer.encode(
        span_prompt(question, context),
        add_special_tokens=True
    ))


def pack_relevant_spans(question, candidates):
    """
    Greedy relevance-first packing.
    Candidate spans are selected without access to gold annotations.
    Exact full-prompt tokenization checks preserve the 512-token limit.
    """
    ranked = sorted(
        candidates,
        key=lambda x: (-x["similarity"], x["session_order"],
                       x["turn_order"], x["span_order"])
    )

    chosen = []

    for candidate in ranked:
        trial = chosen + [candidate]
        chronological = sorted(
            trial,
            key=lambda x: (
                x["session_order"], x["turn_order"], x["span_order"]
            )
        )
        context = "\n".join(x["text"] for x in chronological)

        if exact_input_length(question, context) <= MAX_INPUT:
            chosen = trial

    chosen = sorted(
        chosen,
        key=lambda x: (
            x["session_order"], x["turn_order"], x["span_order"]
        )
    )

    context = "\n".join(x["text"] for x in chosen)
    prompt = span_prompt(question, context)
    input_tokens = exact_input_length(question, context)

    assert input_tokens <= MAX_INPUT
    assert prompt.endswith(f"Question: {question}\nAnswer:")

    return prompt, chosen, input_tokens


# ---------- 3. Build prompts for all methods ----------

span_prompt_rows = []
span_meta = []

for sample_id, (
    conv_idx, qa_idx, item, qa, gold, session_numbers
) in enumerate(sampled):

    conv = item["conversation"]
    speaker_a = str(conv.get("speaker_a", "Speaker A"))
    speaker_b = str(conv.get("speaker_b", "Speaker B"))
    question = str(qa["question"]).strip()

    sessions = []
    rendered_sessions = []
    words, user_counts, turn_counts = [], [], []
    fp_values, digit_values, kept_numbers = [], [], []

    for number in session_numbers:
        turns = extract_session_messages(conv, number)
        if not turns:
            continue

        full, w, uw, nt, fpv, dgv = session_stats(turns, "user")
        rendered = _render_locomo_session(
            turns, speaker_a, speaker_b
        )

        if not rendered.strip():
            continue

        sessions.append(turns)
        rendered_sessions.append(rendered)
        words.append(w)
        user_counts.append(uw)
        turn_counts.append(nt)
        fp_values.append(fpv)
        digit_values.append(dgv)
        kept_numbers.append(number)

    if not sessions:
        continue

    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    Fbase, _ = build_features(
        V,
        np.asarray(words, dtype=float),
        np.asarray(user_counts, dtype=float),
        np.asarray(turn_counts, dtype=float),
        np.asarray(fp_values, dtype=float),
        np.asarray(digit_values, dtype=float),
        np.arange(len(sessions), dtype=float),
        None
    )

    Frole = role_features_for_sessions(question, sessions)
    Ffull = np.column_stack([Fbase, Frole]).astype(np.float32)

    score_map = {
        "baseline": FROZEN_BASELINE.predict_proba(
            Fbase[:, BASE_IDX]
        )[:, 1],
        "role_aware": FROZEN_ROLE_AWARE.predict_proba(
            Ffull[:, FULL_IDX]
        )[:, 1],
        "recency": Fbase[:, 2]
    }

    token_costs = np.asarray([
        len(reader_tokenizer.encode(
            text, add_special_tokens=False
        ))
        for text in rendered_sessions
    ], dtype=int)

    qvec = tfidf.transform(hv.transform([question]))
    similarities = (V @ qvec.T).toarray().ravel()

    # Convert the annotated evidence IDs into actual session numbers.
    gold_numbers = set()
    for evidence in qa.get("evidence", []) or []:
        gold_numbers.update(
            int(x) for x in re.findall(r"D(\d+):", str(evidence))
        )

    available_gold = gold_numbers.intersection(set(kept_numbers))

    span_meta.append({
        "sample_id": sample_id,
        "conversation_index": int(conv_idx),
        "question_hash": hashlib.sha256(
            f"{conv_idx}|{qa_idx}|{question}".encode()
        ).hexdigest()[:20],
        "answer": _qa_refs(qa.get("answer", "")),
        "category": str(qa.get("category", "unknown")),
        "available_gold_sessions": available_gold
    })

    # Create question-relevant spans for each method's retrieved sessions.
    for method, scores in score_map.items():

        selected, memory_used, memory_cap = (
            _select_exact_token_budget(scores, token_costs)
        )

        retrieved = sorted(
            selected,
            key=lambda i: (-float(similarities[i]), i)
        )[:TOP_K]

        candidates = []

        for session_idx in retrieved:
            session_number = kept_numbers[session_idx]

            for turn_idx, (role, message) in enumerate(sessions[session_idx]):
                speaker = speaker_a if role == "user" else speaker_b

                for span_idx, sentence in enumerate(
                    split_message_into_spans(message)
                ):
                    labelled = (
                        f"[Session D{session_number}] "
                        f"{speaker}: {sentence}"
                    )

                    candidates.append({
                        "text": labelled,
                        "session_number": session_number,
                        "session_order": session_idx,
                        "turn_order": turn_idx,
                        "span_order": span_idx,
                        "raw_text": sentence
                    })

        # Score spans using the already-fitted text representation.
        if candidates:
            span_matrix = tfidf.transform(
                hv.transform([x["raw_text"] for x in candidates])
            )
            span_similarities = (
                span_matrix @ qvec.T
            ).toarray().ravel()

            for candidate, similarity in zip(
                candidates, span_similarities
            ):
                candidate["similarity"] = float(similarity)

        prompt, packed, input_tokens = pack_relevant_spans(
            question, candidates
        )

        retrieved_numbers = {
            kept_numbers[i] for i in retrieved
        }
        packed_numbers = {
            x["session_number"] for x in packed
        }

        span_prompt_rows.append({
            "sample_id": sample_id,
            "method": method,
            "prompt": prompt,
            "memory_tokens_selected": int(memory_used),
            "memory_token_budget": int(memory_cap),
            "retrieved_sessions": len(retrieved),
            "packed_sessions": len(packed_numbers),
            "packed_spans": len(packed),
            "reader_input_tokens": int(input_tokens),
            "retrieved_gold_recall": (
                len(available_gold & retrieved_numbers) /
                max(1, len(available_gold))
            ),
            "packed_gold_recall": (
                len(available_gold & packed_numbers) /
                max(1, len(available_gold))
            )
        })

    # Matched question-only control.
    qprompt = span_prompt(question, "")
    qtokens = exact_input_length(question, "")

    assert qtokens <= MAX_INPUT

    span_prompt_rows.append({
        "sample_id": sample_id,
        "method": "question_only",
        "prompt": qprompt,
        "memory_tokens_selected": 0,
        "memory_token_budget": 0,
        "retrieved_sessions": 0,
        "packed_sessions": 0,
        "packed_spans": 0,
        "reader_input_tokens": qtokens,
        "retrieved_gold_recall": 0.0,
        "packed_gold_recall": 0.0
    })

assert len(span_prompt_rows) == len(sampled) * 4, (
    f"Expected {len(sampled)*4} prompts; got {len(span_prompt_rows)}"
)

print("Constructed prompts:", len(span_prompt_rows))
print("All prompts preserve the complete question and fit 512 tokens.")


# ---------- 4. Generate answers ----------

batch_size = 8 if device.type == "cuda" else 4
span_generated = []

for start in range(0, len(span_prompt_rows), batch_size):
    batch = span_prompt_rows[start:start + batch_size]

    encoded = reader_tokenizer(
        [x["prompt"] for x in batch],
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to(device)

    assert encoded["input_ids"].shape[1] <= MAX_INPUT

    with torch.inference_mode():
        output_ids = reader_model.generate(
            **encoded,
            max_new_tokens=MAX_NEW,
            do_sample=False,
            num_beams=1
        )

    predictions = reader_tokenizer.batch_decode(
        output_ids, skip_special_tokens=True
    )

    for row, prediction in zip(batch, predictions):
        span_generated.append({
            **{k: v for k, v in row.items() if k != "prompt"},
            "prediction": str(prediction).strip()
        })

    if (start // batch_size + 1) % 20 == 0:
        print(
            f"Span-packing generated "
            f"{min(start + batch_size, len(span_prompt_rows))}/"
            f"{len(span_prompt_rows)}"
        )


# ---------- 5. Score and summarize ----------

meta_by_id = {x["sample_id"]: x for x in span_meta}
scored = []

for row in span_generated:
    meta = meta_by_id[row["sample_id"]]
    em, f1 = _qa_metrics(row["prediction"], meta["answer"])

    scored.append({
        "sample_id": row["sample_id"],
        "conversation_index": meta["conversation_index"],
        "question_hash": meta["question_hash"],
        "category": meta["category"],
        "method": row["method"],
        "exact_match": em,
        "token_f1": f1,
        "reader_input_tokens": row["reader_input_tokens"],
        "packed_sessions": row["packed_sessions"],
        "packed_spans": row["packed_spans"],
        "retrieved_gold_recall": row["retrieved_gold_recall"],
        "packed_gold_recall": row["packed_gold_recall"]
    })

span_results = pd.DataFrame(scored)

span_summary = (
    span_results.groupby("method")
    .agg(
        n=("token_f1", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_input_tokens=("reader_input_tokens", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_packed_spans=("packed_spans", "mean"),
        mean_retrieved_gold_recall=("retrieved_gold_recall", "mean"),
        mean_packed_gold_recall=("packed_gold_recall", "mean")
    )
    .reset_index()
)


# ---------- 6. Paired conversation-cluster bootstrap ----------

def span_cluster_bootstrap(df, metric, method_a, method_b,
                           n_boot=20000, seed=20261129):
    wide = df.pivot_table(
        index="conversation_index",
        columns="method",
        values=metric,
        aggfunc="mean"
    )

    paired = (
        wide[method_a] - wide[method_b]
    ).dropna().to_numpy(float)

    if len(paired) < 2:
        return {"delta": np.nan, "ci95": [np.nan, np.nan],
                "clusters": len(paired)}

    rng = np.random.default_rng(seed)
    draws = np.empty(n_boot)

    for start in range(0, n_boot, 1000):
        end = min(start + 1000, n_boot)
        ix = rng.integers(
            0, len(paired), size=(end - start, len(paired))
        )
        draws[start:end] = paired[ix].mean(axis=1)

    return {
        "delta": float(paired.mean()),
        "ci95": [
            float(np.quantile(draws, 0.025)),
            float(np.quantile(draws, 0.975))
        ],
        "clusters": int(len(paired))
    }


span_bootstrap = {
    "role_aware_minus_baseline_f1": span_cluster_bootstrap(
        span_results, "token_f1", "role_aware", "baseline"
    ),
    "role_aware_minus_recency_f1": span_cluster_bootstrap(
        span_results, "token_f1", "role_aware", "recency",
        seed=20261130
    ),
    "role_aware_minus_baseline_exact_match": span_cluster_bootstrap(
        span_results, "exact_match", "role_aware", "baseline",
        seed=20261201
    )
}


# ---------- 7. Save privacy-safe experiment artifacts ----------

span_results.to_csv(
    OUT / "evidence_span_per_question.csv", index=False
)
span_summary.to_csv(
    OUT / "evidence_span_summary.csv", index=False
)

with open(OUT / "evidence_span_bootstrap.json", "w") as f:
    json.dump(span_bootstrap, f, indent=2)

manifest = {
    "experiment": "MemWeave evidence-aware sentence-span packing",
    "status": "exploratory",
    "reader_model": globals().get("READER_MODEL_ID", "unknown"),
    "reader_revision": globals().get("reader_revision", "unknown"),
    "questions": int(span_results["question_hash"].nunique()),
    "conversation_clusters": int(
        span_results["conversation_index"].nunique()
    ),
    "reader_input_limit": MAX_INPUT,
    "retrieval_top_k": TOP_K,
    "packing": (
        "Question-relevance-ranked sentence spans from frozen-selector "
        "retrieved sessions; exact prompt-token checks; chronological "
        "ordering after selection."
    ),
    "gold_annotations_used_for_selection": False,
    "generation": "greedy deterministic",
    "bootstrap": span_bootstrap,
    "limitations": [
        "Only ten LoCoMo conversation clusters.",
        "Gold-session retention is not gold-utterance entailment.",
        "Sentence segmentation may separate context across utterances.",
        "Lexical relevance is not semantic relevance."
    ]
}

with open(OUT / "evidence_span_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n========== EVIDENCE-AWARE SPAN PACKING COMPLETE ==========")
display(span_summary.round(4))

print("\nPaired conversation-cluster bootstrap:")
print(json.dumps(span_bootstrap, indent=2))

print("\nSaved artifacts:", OUT)
print("PASS: exact 512-token input cap and complete question preservation.")


Reader: google/flan-t5-small
Questions: 200
Conversation clusters: 10
Constructed prompts: 800
All prompts preserve the complete question and fit 512 tokens.
Span-packing generated 160/800
Span-packing generated 320/800
Span-packing generated 480/800
Span-packing generated 640/800
Span-packing generated 800/800

========== EVIDENCE-AWARE SPAN PACKING COMPLETE ==========


,method,n,exact_match,token_f1,mean_reader_input_tokens,mean_packed_sessions,mean_packed_spans,mean_retrieved_gold_recall,mean_packed_gold_recall
0,baseline,200,0.025,0.1016,508.245,2.700,23.805,0.1008,0.1008
1,question_only,200,0.010,0.0440,42.770,0.000,0.000,0.0000,0.0000
2,recency,200,0.015,0.0812,508.025,2.620,23.285,0.0738,0.0738
3,role_aware,200,0.120,0.2417,507.685,2.445,23.045,0.6727,0.6727



Paired conversation-cluster bootstrap:
{
  "role_aware_minus_baseline_f1": {
    "delta": 0.14012547522996438,
    "ci95": [
      0.06403755258759129,
      0.22127224621998215
    ],
    "clusters": 10
  },
  "role_aware_minus_recency_f1": {
    "delta": 0.1605804478708703,
    "ci95": [
      0.10159454703792942,
      0.22400604039517608
    ],
    "clusters": 10
  },
  "role_aware_minus_baseline_exact_match": {
    "delta": 0.095,
    "ci95": [
      0.030000000000000006,
      0.175
    ],
    "clusters": 10
  }
}

Saved artifacts: /kaggle/working/memweave_followup_outputs/locomo_reader_qa_evidence_span
PASS: exact 512-token input cap and complete question preservation.


### Experiment: Cross-reader validation of evidence-aware span packing

**Objective:** Test whether the downstream advantage of role-aware memory selection with evidence-aware span packing generalizes from FLAN-T5-small to FLAN-T5-base.

**Fixed conditions:**
- Same 200 LoCoMo QA questions and 10 conversation clusters.
- Same frozen baseline, role-aware and recency selectors.
- Same top-three retrieved sessions.
- Same sentence-span construction and relevance-ranking procedure.
- Exact tokenizer-specific 512-token reader input limit.
- Greedy deterministic generation.
- Gold answers and evidence annotations excluded from span selection.

**Primary outcomes:** Exact match and token F1.

**Secondary outcomes:** Gold-session retention after packing, reader input utilization and paired conversation-cluster bootstrap intervals.

**Interpretation:** This is a cross-reader replication, not independent-dataset validation. The 10 conversation clusters remain the statistical limitation.

In [93]:

# Load FLAN-T5-base for the evidence-span replication.
# Keep the FLAN-T5-small artifacts unchanged.

from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

READER_MODEL_ID = "google/flan-t5-base"
reader_revision = HfApi().model_info(READER_MODEL_ID).sha

print("Loading:", READER_MODEL_ID)
print("Pinned revision:", reader_revision)

reader_tokenizer = AutoTokenizer.from_pretrained(
    READER_MODEL_ID,
    revision=reader_revision
)

reader_model = AutoModelForSeq2SeqLM.from_pretrained(
    READER_MODEL_ID,
    revision=reader_revision
).to(device)

reader_model.eval()

print("Second reader loaded:", READER_MODEL_ID)
print("Device:", device)


Loading: google/flan-t5-base
Pinned revision: 7bcac572ce56db69c1ea7c8af255c5d7c9672fc2


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Second reader loaded: google/flan-t5-base
Device: cuda


In [96]:
OUT = Path(OUT_DIR) / "locomo_reader_qa_evidence_span"

In [97]:

# ============================================================
# EVIDENCE-AWARE SPAN PACKING — FLAN-T5-BASE REPLICATION
#
# Reuses:
#   sampled, locomo, reader_model, reader_tokenizer
#   FROZEN_BASELINE, FROZEN_ROLE_AWARE
#   role_features_for_sessions, extract_session_messages
#   session_stats, _render_locomo_session
#   _select_exact_token_budget, _qa_metrics
#   tfidf, hv, build_features, device
#   BASE_IDX, FULL_IDX, OUT_DIR
#
# Keeps:
#   Same 200 questions
#   Same frozen selectors
#   Same 10% memory budget
#   Same top-3 session retrieval
#   Same 512-token reader input cap
#
# Writes to a separate FLAN-T5-base output directory.
# ============================================================

import os
import re
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------
# 1. PRE-FLIGHT CHECKS
# ----------------------------

required = [
    "sampled", "locomo", "reader_model", "reader_tokenizer",
    "FROZEN_BASELINE", "FROZEN_ROLE_AWARE",
    "role_features_for_sessions", "extract_session_messages",
    "session_stats", "_render_locomo_session",
    "_select_exact_token_budget", "_qa_metrics",
    "tfidf", "hv", "build_features", "BASE_IDX",
    "FULL_IDX", "OUT_DIR"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

assert len(sampled) == 200, f"Expected 200 sampled questions, got {len(sampled)}"
assert len(locomo) == 10, f"Expected 10 LoCoMo conversations, got {len(locomo)}"

READER_INPUT_MAX = 512
SPAN_TOP_K_SESSIONS = 3
SPAN_MAX_NEW_TOKENS = 64
SPAN_BOOTSTRAP_REPS = 20000
SPAN_SEED = 20261126

SPAN_OUT_DIR = Path(OUT_DIR) / "locomo_reader_qa_evidence_span_flan_t5_base"
SPAN_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reader:", getattr(reader_model.config, "_name_or_path", "loaded model"))
print("Questions:", len(sampled))
print("Conversation clusters:", len(locomo))
print("Output:", SPAN_OUT_DIR)

# ----------------------------
# 2. PROMPT AND SPAN HELPERS
# ----------------------------

def span_question_only_prompt(question):
    return (
        "Answer the question. No conversation context is available. "
        "If you cannot determine the answer, say: I don't know.\n\n"
        f"Question: {question}\nAnswer:"
    )


def span_make_prompt(question, context):
    if not context:
        return span_question_only_prompt(question)

    return (
        "Answer the question using only the conversation context. "
        "Give a concise answer. If the context does not contain the answer, "
        "say: I don't know.\n\n"
        f"Conversation context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )


def span_token_count(text):
    return len(
        reader_tokenizer.encode(
            text,
            add_special_tokens=True
        )
    )


def span_split_message(text):
    """Split a message into sentence-like spans without dropping its text."""
    text = str(text).strip()
    if not text:
        return []

    pieces = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in pieces if p.strip()]


def span_make_candidates(turns, speaker_a, speaker_b, session_number):
    candidates = []

    for turn_idx, (role, message) in enumerate(turns):
        speaker = speaker_a if role == "user" else speaker_b

        for span_idx, sentence in enumerate(span_split_message(message)):
            candidates.append({
                "session_number": int(session_number),
                "turn_index": int(turn_idx),
                "span_index": int(span_idx),
                "role": role,
                "speaker": speaker,
                "text": sentence,
                "rendered": f"{speaker}: {sentence}"
            })

    return candidates


def span_rank_candidates(question, candidates):
    """Rank spans using lexical TF-IDF similarity to the question."""
    if not candidates:
        return []

    texts = [x["rendered"] for x in candidates]

    matrix = tfidf.transform(hv.transform(texts))
    qvec = tfidf.transform(hv.transform([question]))

    similarities = (matrix @ qvec.T).toarray().ravel()

    ranked = []
    for candidate, similarity in zip(candidates, similarities):
        item = dict(candidate)
        item["similarity"] = float(similarity)
        ranked.append(item)

    # Stable ordering makes ties reproducible.
    ranked.sort(
        key=lambda x: (
            -x["similarity"],
            x["session_number"],
            x["turn_index"],
            x["span_index"]
        )
    )
    return ranked


def span_pack_to_reader_limit(question, ranked_spans):
    """
    Greedily add ranked spans while checking the complete prompt.
    The full question is preserved in every generated prompt.
    """
    selected = []

    question_prompt = span_make_prompt(question, "")
    question_tokens = span_token_count(question_prompt)

    if question_tokens > READER_INPUT_MAX:
        raise ValueError(
            f"Question/instructions exceed reader limit: "
            f"{question_tokens} > {READER_INPUT_MAX}"
        )

    for span in ranked_spans:
        candidate = selected + [span]
        context = "\n".join(x["rendered"] for x in candidate)
        prompt = span_make_prompt(question, context)

        if span_token_count(prompt) <= READER_INPUT_MAX:
            selected.append(span)

    context = "\n".join(x["rendered"] for x in selected)
    prompt = span_make_prompt(question, context)
    token_count = span_token_count(prompt)

    assert token_count <= READER_INPUT_MAX
    assert prompt.endswith(f"Question: {question}\nAnswer:")

    return prompt, context, selected, token_count


def span_generate_answer(prompt):
    encoded = reader_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False
    )

    input_length = int(encoded["input_ids"].shape[1])
    if input_length > READER_INPUT_MAX:
        raise AssertionError(
            f"Prompt exceeds cap: {input_length} > {READER_INPUT_MAX}"
        )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.inference_mode():
        output = reader_model.generate(
            **encoded,
            max_new_tokens=SPAN_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1
        )

    return reader_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()


# ----------------------------
# 3. BUILD QUESTION-LEVEL DATA
# ----------------------------

span_prompt_rows = []
span_meta = []

for sample_id, (
    conv_idx, qa_idx, item, qa, gold, session_numbers
) in enumerate(sampled):

    conv = item["conversation"]

    speaker_a = str(conv.get("speaker_a", "Speaker A"))
    speaker_b = str(conv.get("speaker_b", "Speaker B"))

    question = str(qa["question"]).strip()
    answer_refs = _qa_refs(qa.get("answer", ""))

    sessions = []
    rendered_sessions = []
    word_costs = []
    user_counts = []
    turn_counts = []
    fp_values = []
    digit_values = []
    kept_numbers = []

    # Reconstruct the candidate sessions in their original order.
    for number in session_numbers:
        turns = extract_session_messages(conv, number)

        if not turns:
            continue

        _, words, user_words, nturns, fpv, dgv = session_stats(
            turns, "user"
        )

        rendered = _render_locomo_session(
            turns, speaker_a, speaker_b
        )

        if not rendered.strip():
            continue

        sessions.append(turns)
        rendered_sessions.append(rendered)
        word_costs.append(words)
        user_counts.append(user_words)
        turn_counts.append(nturns)
        fp_values.append(fpv)
        digit_values.append(dgv)
        kept_numbers.append(number)

    if not sessions:
        raise RuntimeError(
            f"No usable sessions for sample {sample_id}"
        )

    # Reconstruct the same base and role-aware features.
    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    days = np.arange(len(sessions), dtype=float)

    Fbase, _ = build_features(
        V,
        np.asarray(word_costs, dtype=float),
        np.asarray(user_counts, dtype=float),
        np.asarray(turn_counts, dtype=float),
        np.asarray(fp_values, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    Frole = role_features_for_sessions(question, sessions)
    Ffull = np.column_stack([Fbase, Frole]).astype(np.float32)

    score_map = {
        "baseline": FROZEN_BASELINE.predict_proba(
            Fbase[:, BASE_IDX]
        )[:, 1],

        "role_aware": FROZEN_ROLE_AWARE.predict_proba(
            Ffull[:, FULL_IDX]
        )[:, 1],

        "recency": Fbase[:, 2]
    }

    session_token_costs = np.asarray([
        len(reader_tokenizer.encode(
            text,
            add_special_tokens=False
        ))
        for text in rendered_sessions
    ], dtype=int)

    # Candidate spans are produced from all available sessions.
    all_spans = []

    for number, turns in zip(kept_numbers, sessions):
        all_spans.extend(
            span_make_candidates(
                turns,
                speaker_a,
                speaker_b,
                number
            )
        )

    ranked_all_spans = span_rank_candidates(
        question,
        all_spans
    )

    question_hash = hashlib.sha256(
        f"{conv_idx}|{qa_idx}|{question}".encode("utf-8")
    ).hexdigest()[:20]

    span_meta.append({
        "sample_id": int(sample_id),
        "conversation_index": int(conv_idx),
        "question_hash": question_hash,
        "question": question,
        "answer": answer_refs,
        "category": str(qa.get("category", "unknown")),
        "gold_session_indices": sorted(
            locomo_gold_indices(qa, kept_numbers)
        ),
        "session_numbers": [int(x) for x in kept_numbers]
    })

    # Question-only control.
    q_prompt = span_make_prompt(question, "")
    span_prompt_rows.append({
        "sample_id": int(sample_id),
        "conversation_index": int(conv_idx),
        "question_hash": question_hash,
        "method": "question_only",
        "prompt": q_prompt,
        "input_tokens": span_token_count(q_prompt),
        "selected_sessions": [],
        "selected_spans": [],
        "retrieved_gold_recall": 0.0,
        "packed_gold_recall": 0.0,
        "packed_context_tokens": 0
    })

    # Each memory selector first selects sessions under the same budget.
    for method, scores in score_map.items():

        selected_indices, memory_used, memory_cap = (
            _select_exact_token_budget(
                scores,
                session_token_costs
            )
        )

        # Keep the selector's top three selected sessions.
        selected_indices = sorted(
            selected_indices,
            key=lambda idx: (-float(scores[idx]), int(idx))
        )[:SPAN_TOP_K_SESSIONS]

        selected_session_numbers = [
            int(kept_numbers[idx]) for idx in selected_indices
        ]

        selected_session_spans = [
            span for span in ranked_all_spans
            if span["session_number"] in selected_session_numbers
        ]

        # Rank spans within the selected sessions.
        selected_session_spans.sort(
            key=lambda x: (
                -x["similarity"],
                x["session_number"],
                x["turn_index"],
                x["span_index"]
            )
        )

        prompt, context, packed_spans, input_tokens = (
            span_pack_to_reader_limit(
                question,
                selected_session_spans
            )
        )

        gold_indices = set(
            locomo_gold_indices(qa, kept_numbers)
        )

        retrieved_gold = (
            len(set(selected_indices) & gold_indices) /
            len(gold_indices)
            if gold_indices else np.nan
        )

        packed_session_numbers = {
            x["session_number"] for x in packed_spans
        }

        packed_gold_indices = {
            idx for idx, number in enumerate(kept_numbers)
            if number in packed_session_numbers
        }

        packed_gold = (
            len(packed_gold_indices & gold_indices) /
            len(gold_indices)
            if gold_indices else np.nan
        )

        span_prompt_rows.append({
            "sample_id": int(sample_id),
            "conversation_index": int(conv_idx),
            "question_hash": question_hash,
            "method": method,
            "prompt": prompt,
            "input_tokens": int(input_tokens),
            "selected_sessions": selected_session_numbers,
            "selected_spans": packed_spans,
            "retrieved_gold_recall": retrieved_gold,
            "packed_gold_recall": packed_gold,
            "packed_context_tokens": int(
                len(reader_tokenizer.encode(
                    context,
                    add_special_tokens=False
                ))
            ),
            "memory_tokens_used": int(memory_used),
            "memory_token_budget": int(memory_cap)
        })

    if (sample_id + 1) % 25 == 0:
        print(
            f"Constructed prompts for {sample_id + 1}/"
            f"{len(sampled)} questions"
        )


# ----------------------------
# 4. AUDIT PROMPTS BEFORE GENERATION
# ----------------------------

assert len(span_prompt_rows) == 800, (
    f"Expected 800 prompts, got {len(span_prompt_rows)}"
)

span_prompt_df = pd.DataFrame(span_prompt_rows)

assert span_prompt_df["input_tokens"].max() <= READER_INPUT_MAX
assert span_prompt_df["prompt"].notna().all()

for row in span_prompt_rows:
    q = next(
        x["question"] for x in span_meta
        if x["question_hash"] == row["question_hash"]
    )
    assert row["prompt"].endswith(f"Question: {q}\nAnswer:")

print("\nPrompt audit passed.")
print("Constructed prompts:", len(span_prompt_rows))
print("Maximum input tokens:", span_prompt_df["input_tokens"].max())
print("All complete questions preserved:", True)


# ----------------------------
# 5. GENERATE ANSWERS
# ----------------------------

span_generated = []
generation_start = time.time()

for idx, row in enumerate(span_prompt_rows):

    prediction = span_generate_answer(row["prompt"])

    meta = next(
        x for x in span_meta
        if x["question_hash"] == row["question_hash"]
    )

    em, f1 = _qa_metrics(
        prediction,
        meta["answer"]
    )

    selected_spans = row["selected_spans"]
    selected_sessions = row["selected_sessions"]

    span_generated.append({
        "sample_id": row["sample_id"],
        "conversation_index": row["conversation_index"],
        "question_hash": row["question_hash"],
        "method": row["method"],
        "prediction": prediction,
        "exact_match": float(em),
        "token_f1": float(f1),
        "input_tokens": int(row["input_tokens"]),
        "packed_context_tokens": int(row["packed_context_tokens"]),
        "packed_sessions": int(len(selected_sessions)),
        "packed_spans": int(len(selected_spans)),
        "retrieved_gold_recall": row["retrieved_gold_recall"],
        "packed_gold_recall": row["packed_gold_recall"]
    })

    if (idx + 1) % 40 == 0:
        elapsed = time.time() - generation_start
        print(
            f"Generated {idx + 1}/800 | "
            f"elapsed: {elapsed / 60:.1f} min"
        )

span_results = pd.DataFrame(span_generated)

assert len(span_results) == 800
assert span_results["prediction"].notna().all()


# ----------------------------
# 6. SUMMARIZE RESULTS
# ----------------------------

span_summary = (
    span_results
    .groupby("method", dropna=False)
    .agg(
        n=("exact_match", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_input_tokens=("input_tokens", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_packed_spans=("packed_spans", "mean"),
        mean_retrieved_gold_recall=("retrieved_gold_recall", "mean"),
        mean_packed_gold_recall=("packed_gold_recall", "mean"),
        mean_packed_context_tokens=("packed_context_tokens", "mean")
    )
    .reset_index()
)

print("\nEVIDENCE-SPAN READER QA RESULTS")
display(span_summary)


# ----------------------------
# 7. PAIRED CONVERSATION-CLUSTER BOOTSTRAP
# ----------------------------

def span_cluster_bootstrap(
    results,
    metric,
    method_a,
    method_b,
    reps=SPAN_BOOTSTRAP_REPS,
    seed=SPAN_SEED
):
    """
    Paired bootstrap over the 10 conversation clusters.
    Questions within a conversation are not treated as independent clusters.
    """

    pivot = results.pivot_table(
        index=["conversation_index", "sample_id"],
        columns="method",
        values=metric,
        aggfunc="first"
    )

    if method_a not in pivot.columns or method_b not in pivot.columns:
        raise ValueError("Requested methods missing from results.")

    paired = pivot[[method_a, method_b]].dropna()
    paired["delta"] = paired[method_a] - paired[method_b]

    cluster_means = (
        paired.groupby(level="conversation_index")["delta"]
        .mean()
        .to_numpy()
    )

    if len(cluster_means) != 10:
        raise ValueError(
            f"Expected 10 conversation clusters, got {len(cluster_means)}"
        )

    rng = np.random.default_rng(seed)
    sampled_indices = rng.integers(
        0,
        len(cluster_means),
        size=(reps, len(cluster_means))
    )

    bootstrap_means = cluster_means[sampled_indices].mean(axis=1)

    return {
        "metric": metric,
        "method_a": method_a,
        "method_b": method_b,
        "delta": float(cluster_means.mean()),
        "ci_2_5": float(np.quantile(bootstrap_means, 0.025)),
        "ci_97_5": float(np.quantile(bootstrap_means, 0.975)),
        "clusters": int(len(cluster_means)),
        "bootstrap_reps": int(reps)
    }


span_bootstrap = []

for metric in ["token_f1", "exact_match"]:
    for comparison in [
        ("role_aware", "baseline"),
        ("role_aware", "recency")
    ]:
        span_bootstrap.append(
            span_cluster_bootstrap(
                span_results,
                metric,
                comparison[0],
                comparison[1]
            )
        )

span_bootstrap_df = pd.DataFrame(span_bootstrap)

print("\nPAIRED CONVERSATION-CLUSTER BOOTSTRAP")
display(span_bootstrap_df)


# ----------------------------
# 8. SAVE RAW RESULTS AND MANIFEST
# ----------------------------

span_results.to_csv(
    SPAN_OUT_DIR / "evidence_span_per_question.csv",
    index=False
)

span_summary.to_csv(
    SPAN_OUT_DIR / "evidence_span_summary.csv",
    index=False
)

span_bootstrap_df.to_csv(
    SPAN_OUT_DIR / "evidence_span_bootstrap.csv",
    index=False
)

# Save detailed span selections separately for auditability.
span_selection_rows = []

for row in span_prompt_rows:
    span_selection_rows.append({
        "sample_id": row["sample_id"],
        "conversation_index": row["conversation_index"],
        "question_hash": row["question_hash"],
        "method": row["method"],
        "selected_sessions": json.dumps(row["selected_sessions"]),
        "selected_spans": json.dumps(row["selected_spans"]),
        "input_tokens": row["input_tokens"],
        "packed_context_tokens": row["packed_context_tokens"],
        "retrieved_gold_recall": row["retrieved_gold_recall"],
        "packed_gold_recall": row["packed_gold_recall"]
    })

pd.DataFrame(span_selection_rows).to_csv(
    SPAN_OUT_DIR / "evidence_span_selections.csv",
    index=False
)

manifest = {
    "experiment": "evidence_aware_span_packing_flan_t5_base",
    "reader_model": getattr(
        reader_model.config,
        "_name_or_path",
        "google/flan-t5-base"
    ),
    "questions": int(len(sampled)),
    "conversation_clusters": int(len(locomo)),
    "methods": [
        "baseline",
        "question_only",
        "recency",
        "role_aware"
    ],
    "memory_budget_fraction": 0.10,
    "top_k_sessions": SPAN_TOP_K_SESSIONS,
    "reader_input_max_tokens": READER_INPUT_MAX,
    "max_new_tokens": SPAN_MAX_NEW_TOKENS,
    "span_ranking": "TF-IDF lexical similarity",
    "gold_evidence_granularity": "session-level annotations",
    "bootstrap_unit": "conversation cluster",
    "bootstrap_repetitions": SPAN_BOOTSTRAP_REPS,
    "seed": SPAN_SEED,
    "prompts_constructed": int(len(span_prompt_rows)),
    "prompts_generated": int(len(span_results)),
    "all_prompts_within_token_limit": bool(
        span_results["input_tokens"].max() <= READER_INPUT_MAX
    ),
    "output_directory": str(SPAN_OUT_DIR)
}

with open(
    SPAN_OUT_DIR / "evidence_span_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)

print("\nEXPERIMENT COMPLETE")
print("Saved to:", SPAN_OUT_DIR)
print("Generated answers:", len(span_results))
print("Maximum reader input:", span_results["input_tokens"].max())
print("All inputs within 512 tokens:", 
      bool((span_results["input_tokens"] <= 512).all()))


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (680 > 512). Running this sequence through the model will result in indexing errors


Reader: google/flan-t5-base
Questions: 200
Conversation clusters: 10
Output: /kaggle/working/memweave_followup_outputs/locomo_reader_qa_evidence_span_flan_t5_base
Constructed prompts for 25/200 questions
Constructed prompts for 50/200 questions
Constructed prompts for 75/200 questions
Constructed prompts for 100/200 questions
Constructed prompts for 125/200 questions
Constructed prompts for 150/200 questions
Constructed prompts for 175/200 questions
Constructed prompts for 200/200 questions

Prompt audit passed.
Constructed prompts: 800
Maximum input tokens: 512
All complete questions preserved: True
Generated 40/800 | elapsed: 0.1 min
Generated 80/800 | elapsed: 0.2 min
Generated 120/800 | elapsed: 0.3 min
Generated 160/800 | elapsed: 0.4 min
Generated 200/800 | elapsed: 0.5 min
Generated 240/800 | elapsed: 0.6 min
Generated 280/800 | elapsed: 0.7 min
Generated 320/800 | elapsed: 0.8 min
Generated 360/800 | elapsed: 0.9 min
Generated 400/800 | elapsed: 0.9 min
Generated 440/800 | elap

,method,n,exact_match,token_f1,mean_reader_input_tokens,mean_packed_sessions,mean_packed_spans,mean_retrieved_gold_recall,mean_packed_gold_recall,mean_packed_context_tokens
0,baseline,200,0.030,0.091778,510.660,2.80,45.200,0.093333,0.093333,457.890
1,question_only,200,0.005,0.005714,42.770,0.00,0.000,0.000000,0.000000,0.000
2,recency,200,0.010,0.057676,510.755,2.70,45.460,0.073750,0.073750,457.985
3,role_aware,200,0.100,0.237578,510.675,2.46,43.905,0.672667,0.672667,457.905



PAIRED CONVERSATION-CLUSTER BOOTSTRAP


,metric,method_a,method_b,delta,ci_2_5,ci_97_5,clusters,bootstrap_reps
0,token_f1,role_aware,baseline,0.145799,0.081810,0.216988,10,20000
1,token_f1,role_aware,recency,0.179901,0.117619,0.243587,10,20000
2,exact_match,role_aware,baseline,0.070000,0.025000,0.130000,10,20000
3,exact_match,role_aware,recency,0.090000,0.040000,0.150000,10,20000



EXPERIMENT COMPLETE
Saved to: /kaggle/working/memweave_followup_outputs/locomo_reader_qa_evidence_span_flan_t5_base
Generated answers: 800
Maximum reader input: 512
All inputs within 512 tokens: True


In [98]:

# ============================================================
# MEMWEAVE — INDEPENDENT DATASET VALIDATION
# Long-MT-Bench+ | FLAN-T5-base | Evidence-span packing
#
# Frozen LongMemEval-trained selectors.
# No fitting or tuning on Long-MT-Bench+.
# ============================================================

import re
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_dataset

# ------------------------------------------------------------
# 1. PRE-FLIGHT CHECKS
# ------------------------------------------------------------

required = [
    "FROZEN_BASELINE",
    "FROZEN_ROLE_AWARE",
    "BASE_IDX",
    "FULL_IDX",
    "role_features_for_sessions",
    "session_stats",
    "build_features",
    "tfidf",
    "hv",
    "reader_model",
    "reader_tokenizer",
    "device",
    "OUT_DIR",
    "_qa_metrics"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}")

reader_name = str(
    getattr(reader_model.config, "_name_or_path", "")
).lower()

if "flan-t5-base" not in reader_name:
    raise RuntimeError(
        "Load google/flan-t5-base before running this experiment."
    )

MAX_INPUT = 512
MAX_NEW_TOKENS = 64
TOP_K_SESSIONS = 3
MEMORY_FRACTION = 0.10
BOOTSTRAP_REPS = 20000
SEED = 20261127

OUT = Path(OUT_DIR) / "long_mt_bench_plus_flan_t5_base"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD EXTERNAL DATASET
# ------------------------------------------------------------

dataset = load_dataset(
    "panzs19/Long-MT-Bench-Plus",
    split="test"
)

print("Dataset conversations:", len(dataset))

if len(dataset) != 11:
    print(
        "WARNING: Expected 11 conversations. "
        "Check the dataset revision and schema."
    )

# ------------------------------------------------------------
# 3. PARSE HUMAN/ASSISTANT TURNS
# ------------------------------------------------------------

def parse_longmt_turn(raw_turn):

    text = str(raw_turn).strip()

    match = re.match(
        r"^\[(human|assistant)\]\s*:\s*(.*)$",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        raise ValueError(
            f"Unexpected Long-MT-Bench+ turn format: {text[:150]}"
        )

    role = match.group(1).lower()
    content = match.group(2).strip()

    role = "user" if role == "human" else "assistant"

    return role, content


def parse_longmt_sessions(raw_sessions):

    parsed = []

    for session in raw_sessions:

        turns = []

        for raw_turn in session:

            role, content = parse_longmt_turn(raw_turn)

            if content:
                turns.append((role, content))

        if turns:
            parsed.append(turns)

    return parsed


def render_longmt_session(turns):

    lines = []

    for role, content in turns:

        speaker = (
            "Human" if role == "user"
            else "Assistant"
        )

        lines.append(f"{speaker}: {content}")

    return "\n".join(lines)


# ------------------------------------------------------------
# 4. PROMPT AND SPAN HELPERS
# ------------------------------------------------------------

def longmt_prompt(question, context):

    if not context:
        return (
            "Answer the question. No conversation context is available. "
            "If you cannot determine the answer, say: I don't know.\n\n"
            f"Question: {question}\nAnswer:"
        )

    return (
        "Answer the question using only the conversation context. "
        "Give a concise answer. If the context does not contain the answer, "
        "say: I don't know.\n\n"
        f"Conversation context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )


def longmt_token_count(text):

    return len(
        reader_tokenizer.encode(
            text,
            add_special_tokens=True
        )
    )


def longmt_split_spans(text):

    pieces = re.split(
        r"(?<=[.!?])\s+",
        str(text).strip()
    )

    return [
        x.strip()
        for x in pieces
        if x.strip()
    ]


def longmt_candidates(sessions):

    candidates = []

    for session_idx, turns in enumerate(sessions):

        for turn_idx, (role, content) in enumerate(turns):

            speaker = (
                "Human" if role == "user"
                else "Assistant"
            )

            for span_idx, sentence in enumerate(
                longmt_split_spans(content)
            ):

                candidates.append({
                    "session": session_idx,
                    "turn": turn_idx,
                    "span": span_idx,
                    "role": role,
                    "text": sentence,
                    "rendered": f"{speaker}: {sentence}"
                })

    return candidates


def longmt_rank_spans(question, candidates):

    if not candidates:
        return []

    texts = [
        x["rendered"] for x in candidates
    ]

    matrix = tfidf.transform(
        hv.transform(texts)
    )

    qvec = tfidf.transform(
        hv.transform([question])
    )

    similarities = (
        matrix @ qvec.T
    ).toarray().ravel()

    ranked = []

    for candidate, similarity in zip(
        candidates,
        similarities
    ):

        item = dict(candidate)
        item["similarity"] = float(similarity)

        ranked.append(item)

    ranked.sort(
        key=lambda x: (
            -x["similarity"],
            x["session"],
            x["turn"],
            x["span"]
        )
    )

    return ranked


def longmt_pack(question, ranked_spans):

    question_only = longmt_prompt(question, "")

    if longmt_token_count(question_only) > MAX_INPUT:
        raise ValueError(
            "Question and instructions alone exceed 512 tokens."
        )

    chosen = []

    for span in ranked_spans:

        candidate = chosen + [span]

        context = "\n".join(
            x["rendered"] for x in candidate
        )

        prompt = longmt_prompt(question, context)

        if longmt_token_count(prompt) <= MAX_INPUT:
            chosen.append(span)

    context = "\n".join(
        x["rendered"] for x in chosen
    )

    prompt = longmt_prompt(question, context)

    token_count = longmt_token_count(prompt)

    assert token_count <= MAX_INPUT
    assert prompt.endswith(
        f"Question: {question}\nAnswer:"
    )

    return prompt, chosen, token_count


def longmt_generate(prompt):

    encoded = reader_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False
    )

    assert encoded["input_ids"].shape[1] <= MAX_INPUT

    encoded = {
        k: v.to(device)
        for k, v in encoded.items()
    }

    with torch.inference_mode():

        generated = reader_model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1
        )

    return reader_tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    ).strip()


# ------------------------------------------------------------
# 5. EXACT TOKEN-BUDGET SELECTION
# ------------------------------------------------------------

def longmt_select(scores, costs):

    scores = np.asarray(scores, dtype=float)
    costs = np.asarray(costs, dtype=int)

    budget = max(
        1,
        int(np.floor(costs.sum() * MEMORY_FRACTION))
    )

    selected = []
    used = 0

    for idx in np.argsort(-scores, kind="stable"):

        cost = int(costs[idx])

        if used + cost <= budget:

            selected.append(int(idx))
            used += cost

    return selected, used, budget


# ------------------------------------------------------------
# 6. RUN ALL CONVERSATIONS AND QUESTIONS
# ------------------------------------------------------------

results = []
selection_audit = []

start_time = time.time()
total_questions = sum(
    len(row["questions"])
    for row in dataset
)

completed = 0

for conversation_idx, item in enumerate(dataset):

    sessions = parse_longmt_sessions(
        item["sessions"]
    )

    if not sessions:
        raise RuntimeError(
            f"Conversation {conversation_idx} has no usable sessions."
        )

    rendered_sessions = [
        render_longmt_session(x)
        for x in sessions
    ]

    # Reconstruct base features using the frozen
    # LongMemEval text representation.
    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    words = []
    user_words = []
    turns_count = []
    first_person = []
    digit_values = []

    for session in sessions:

        _, w, uw, nt, fpv, dgv = session_stats(
            session,
            "user"
        )

        words.append(w)
        user_words.append(uw)
        turns_count.append(nt)
        first_person.append(fpv)
        digit_values.append(dgv)

    days = np.arange(
        len(sessions),
        dtype=float
    )

    Fbase, _ = build_features(
        V,
        np.asarray(words, dtype=float),
        np.asarray(user_words, dtype=float),
        np.asarray(turns_count, dtype=float),
        np.asarray(first_person, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    # Compute the role-aware feature representation.
    Frole = role_features_for_sessions(
        "",
        sessions
    )

    # Role features are question-dependent.
    # Recompute inside the QA loop below.

    session_costs = np.asarray([
        len(
            reader_tokenizer.encode(
                text,
                add_special_tokens=False
            )
        )
        for text in rendered_sessions
    ], dtype=int)

    questions = item["questions"]
    answers = item["answers"]

    if len(questions) != len(answers):
        raise ValueError(
            f"Question/answer mismatch in conversation {conversation_idx}"
        )

    for question_idx, (question, answer) in enumerate(
        zip(questions, answers)
    ):

        question = str(question).strip()

        if not question:
            continue

        # Question-specific role features.
        Frole = role_features_for_sessions(
            question,
            sessions
        )

        Ffull = np.column_stack([
            Fbase,
            Frole
        ]).astype(np.float32)

        score_map = {
            "baseline": FROZEN_BASELINE.predict_proba(
                Fbase[:, BASE_IDX]
            )[:, 1],

            "role_aware": FROZEN_ROLE_AWARE.predict_proba(
                Ffull[:, FULL_IDX]
            )[:, 1],

            "recency": Fbase[:, 2]
        }

        candidates = longmt_candidates(sessions)

        ranked_all = longmt_rank_spans(
            question,
            candidates
        )

        question_hash = hashlib.sha256(
            f"{conversation_idx}|{question_idx}|{question}".encode()
        ).hexdigest()[:20]

        # Question-only control.
        prompt = longmt_prompt(question, "")

        results.append({
            "conversation_index": conversation_idx,
            "question_index": question_idx,
            "question_hash": question_hash,
            "method": "question_only",
            "prediction": longmt_generate(prompt),
            "input_tokens": longmt_token_count(prompt),
            "packed_sessions": 0,
            "packed_spans": 0,
            "answer": str(answer)
        })

        for method, scores in score_map.items():

            selected, memory_used, memory_budget = (
                longmt_select(
                    scores,
                    session_costs
                )
            )

            # Retain the selector's top three sessions.
            selected = sorted(
                selected,
                key=lambda idx: (
                    -float(scores[idx]),
                    int(idx)
                )
            )[:TOP_K_SESSIONS]

            selected_spans = [
                x for x in ranked_all
                if x["session"] in selected
            ]

            selected_spans.sort(
                key=lambda x: (
                    -x["similarity"],
                    x["session"],
                    x["turn"],
                    x["span"]
                )
            )

            prompt, packed_spans, input_tokens = (
                longmt_pack(
                    question,
                    selected_spans
                )
            )

            prediction = longmt_generate(prompt)

            results.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": method,
                "prediction": prediction,
                "input_tokens": input_tokens,
                "packed_sessions": len(set(
                    x["session"] for x in packed_spans
                )),
                "packed_spans": len(packed_spans),
                "answer": str(answer)
            })

            selection_audit.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": method,
                "selected_sessions": selected,
                "memory_tokens_used": memory_used,
                "memory_token_budget": memory_budget,
                "packed_spans": len(packed_spans),
                "reader_input_tokens": input_tokens
            })

        completed += 1

        if completed % 25 == 0:
            print(
                f"Completed {completed}/{total_questions} "
                f"questions | "
                f"elapsed {(time.time()-start_time)/60:.1f} min"
            )


# ------------------------------------------------------------
# 7. SCORE ANSWERS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

metric_values = results_df.apply(
    lambda row: _qa_metrics(
        row["prediction"],
        row["answer"]
    ),
    axis=1
)

results_df["exact_match"] = [
    x[0] for x in metric_values
]

results_df["token_f1"] = [
    x[1] for x in metric_values
]

summary = (
    results_df
    .groupby("method")
    .agg(
        n=("exact_match", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_input_tokens=("input_tokens", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_packed_spans=("packed_spans", "mean")
    )
    .reset_index()
)

print("\nLONG-MT-BENCH+ RESULTS")
display(summary)


# ------------------------------------------------------------
# 8. CONVERSATION-CLUSTER PAIRED BOOTSTRAP
# ------------------------------------------------------------

def longmt_cluster_bootstrap(
    results_df,
    metric,
    method_a,
    method_b,
    reps=BOOTSTRAP_REPS,
    seed=SEED
):

    pivot = results_df.pivot_table(
        index=[
            "conversation_index",
            "question_index"
        ],
        columns="method",
        values=metric,
        aggfunc="first"
    )

    paired = pivot[
        [method_a, method_b]
    ].dropna()

    paired["delta"] = (
        paired[method_a] - paired[method_b]
    )

    cluster_means = (
        paired.groupby(level="conversation_index")["delta"]
        .mean()
        .to_numpy()
    )

    n_clusters = len(cluster_means)

    if n_clusters < 2:
        raise RuntimeError(
            "Not enough independent conversation clusters."
        )

    rng = np.random.default_rng(seed)

    sampled = rng.integers(
        0,
        n_clusters,
        size=(reps, n_clusters)
    )

    boot = cluster_means[sampled].mean(axis=1)

    return {
        "metric": metric,
        "contrast": f"{method_a} - {method_b}",
        "delta": float(cluster_means.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
        "conversation_clusters": n_clusters,
        "bootstrap_reps": reps
    }


bootstrap_rows = []

for metric in [
    "token_f1",
    "exact_match"
]:

    for comparator in [
        "baseline",
        "recency"
    ]:

        bootstrap_rows.append(
            longmt_cluster_bootstrap(
                results_df,
                metric,
                "role_aware",
                comparator
            )
        )

bootstrap_df = pd.DataFrame(bootstrap_rows)

print("\nPAIRED CONVERSATION-CLUSTER BOOTSTRAP")
display(bootstrap_df)


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

# Save hashed question identifiers and generated outputs.
# No raw conversation history is exported.

results_df.drop(
    columns=["answer"]
).to_csv(
    OUT / "longmt_reader_qa_per_question.csv",
    index=False
)

summary.to_csv(
    OUT / "longmt_reader_qa_summary.csv",
    index=False
)

bootstrap_df.to_csv(
    OUT / "longmt_reader_qa_bootstrap.csv",
    index=False
)

pd.DataFrame(selection_audit).to_csv(
    OUT / "longmt_selection_audit.csv",
    index=False
)

manifest = {
    "experiment": "Long-MT-Bench+ external transfer",
    "dataset": "panzs19/Long-MT-Bench-Plus",
    "reader": reader_name,
    "selector_training": "LongMemEval DEV only",
    "selector_tuning_on_external_dataset": False,
    "memory_budget_fraction": MEMORY_FRACTION,
    "top_k_sessions": TOP_K_SESSIONS,
    "reader_input_max_tokens": MAX_INPUT,
    "max_new_tokens": MAX_NEW_TOKENS,
    "span_ranking": "TF-IDF lexical similarity",
    "conversation_clusters": int(
        results_df["conversation_index"].nunique()
    ),
    "questions": int(
        results_df["question_hash"].nunique()
    ),
    "generated_rows": int(len(results_df)),
    "gold_retrieval_annotations_available": False,
    "metrics": [
        "exact_match",
        "token_f1"
    ],
    "limitations": [
        "Only 11 conversation clusters.",
        "No gold retrieval-session annotations.",
        "Long-MT-Bench+ conversations are shorter than LoCoMo.",
        "TF-IDF span ranking is lexical, not semantic.",
        "Results are external transfer evidence, not broad generalization proof."
    ]
}

with open(
    OUT / "longmt_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=2)

print("\nEXTERNAL VALIDATION COMPLETE")
print("Questions:", manifest["questions"])
print("Conversation clusters:", manifest["conversation_clusters"])
print("Generated rows:", manifest["generated_rows"])
print("Maximum input tokens:", results_df["input_tokens"].max())
print("All inputs within cap:", bool(
    (results_df["input_tokens"] <= MAX_INPUT).all()
))
print("Saved to:", OUT)


README.md:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.06MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/11 [00:00<?, ? examples/s]

Dataset conversations: 11
Completed 25/288 questions | elapsed 0.2 min
Completed 50/288 questions | elapsed 0.5 min
Completed 75/288 questions | elapsed 0.9 min
Completed 100/288 questions | elapsed 1.2 min
Completed 125/288 questions | elapsed 1.4 min
Completed 150/288 questions | elapsed 1.6 min
Completed 175/288 questions | elapsed 1.9 min
Completed 200/288 questions | elapsed 2.1 min
Completed 225/288 questions | elapsed 2.3 min
Completed 250/288 questions | elapsed 2.6 min
Completed 275/288 questions | elapsed 2.8 min

LONG-MT-BENCH+ RESULTS


,method,n,exact_match,token_f1,mean_reader_input_tokens,mean_packed_sessions,mean_packed_spans
0,baseline,288,0.0,0.013826,93.152778,0.083333,1.215278
1,question_only,288,0.0,0.006017,55.444444,0.000000,0.000000
2,recency,288,0.0,0.013826,93.152778,0.083333,1.215278
3,role_aware,288,0.0,0.013826,93.152778,0.083333,1.215278



PAIRED CONVERSATION-CLUSTER BOOTSTRAP


,metric,contrast,delta,ci_low,ci_high,conversation_clusters,bootstrap_reps
0,token_f1,role_aware - baseline,0.0,0.0,0.0,11,20000
1,token_f1,role_aware - recency,0.0,0.0,0.0,11,20000
2,exact_match,role_aware - baseline,0.0,0.0,0.0,11,20000
3,exact_match,role_aware - recency,0.0,0.0,0.0,11,20000



EXTERNAL VALIDATION COMPLETE
Questions: 288
Conversation clusters: 11
Generated rows: 1152
Maximum input tokens: 512
All inputs within cap: True
Saved to: /kaggle/working/memweave_followup_outputs/long_mt_bench_plus_flan_t5_base


In [99]:

import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path(
    "/kaggle/working/memweave_followup_outputs/"
    "long_mt_bench_plus_flan_t5_base"
)

audit = pd.read_csv(
    OUT / "longmt_selection_audit.csv"
)

print("AUDIT ROWS:", len(audit))
print("\nMETHOD SUMMARY")

display(
    audit.groupby("method").agg(
        questions=("question_hash", "nunique"),
        mean_memory_used=("memory_tokens_used", "mean"),
        mean_memory_budget=("memory_token_budget", "mean"),
        mean_packed_spans=("packed_spans", "mean"),
        mean_reader_tokens=("reader_input_tokens", "mean"),
        zero_packed_questions=(
            "packed_spans",
            lambda x: int((x == 0).sum())
        )
    )
)

print("\nSELECTED SESSION COUNT DISTRIBUTION")

for method, group in audit.groupby("method"):

    counts = group["selected_sessions"].apply(
        lambda x: len(eval(x)) if isinstance(x, str) else 0
    )

    print(
        method,
        counts.value_counts().sort_index().to_dict()
    )

print("\nCHECK WHETHER SELECTORS ACTUALLY DIFFER")

baseline = audit[
    audit.method == "baseline"
].set_index("question_hash")

role = audit[
    audit.method == "role_aware"
].set_index("question_hash")

common = baseline.index.intersection(role.index)

same = (
    baseline.loc[common, "selected_sessions"].astype(str)
    ==
    role.loc[common, "selected_sessions"].astype(str)
)

print("Identical selected-session lists:",
      float(same.mean()))

print("Different selected-session lists:",
      int((~same).sum()), "/", len(common))


AUDIT ROWS: 864

METHOD SUMMARY


,questions,mean_memory_used,mean_memory_budget,mean_packed_spans,mean_reader_tokens,zero_packed_questions
method,,,,,,
baseline,288,154.666667,2185.201389,1.215278,93.152778,264
recency,288,154.666667,2185.201389,1.215278,93.152778,264
role_aware,288,154.666667,2185.201389,1.215278,93.152778,264



SELECTED SESSION COUNT DISTRIBUTION
baseline {0: 264, 1: 24}
recency {0: 264, 1: 24}
role_aware {0: 264, 1: 24}

CHECK WHETHER SELECTORS ACTUALLY DIFFER
Identical selected-session lists: 1.0
Different selected-session lists: 0 / 288


In [100]:

# ============================================================
# MEMWEAVE — CORRECTED LONG-MT-BENCH+ EXPERIMENT
# Span-level memory selection under exact 10% token budget
# ============================================================

import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

OUT = Path(
    "/kaggle/working/memweave_followup_outputs/"
    "long_mt_bench_plus_span_budget_flan_t5_base"
)
OUT.mkdir(parents=True, exist_ok=True)

MAX_INPUT = 512
MEMORY_FRACTION = 0.10
BOOTSTRAP_REPS = 20000
SEED = 20261127

# ------------------------------------------------------------
# 1. SPAN-LEVEL BUDGET SELECTOR
# ------------------------------------------------------------

def select_spans_under_budget(
    question,
    candidates,
    session_scores,
    memory_budget
):
    if not candidates:
        return [], 0

    n_sessions = len(session_scores)

    # Convert session scores into descending percentile ranks.
    session_order = np.argsort(
        -np.asarray(session_scores),
        kind="stable"
    )

    session_rank = np.empty(n_sessions, dtype=float)
    session_rank[session_order] = np.linspace(
        1.0, 0.0, n_sessions
    ) if n_sessions > 1 else 1.0

    # Rank lexical similarity across candidate spans.
    lexical = np.asarray([
        x["similarity"] for x in candidates
    ], dtype=float)

    lexical_order = np.argsort(
        -lexical,
        kind="stable"
    )

    lexical_rank = np.empty(len(candidates), dtype=float)
    lexical_rank[lexical_order] = np.linspace(
        1.0, 0.0, len(candidates)
    ) if len(candidates) > 1 else 1.0

    ranked = []

    for i, candidate in enumerate(candidates):

        item = dict(candidate)

        session_idx = int(item["session"])

        # Equal-weight combination of:
        # frozen selector's session score and lexical span relevance.
        item["combined_score"] = (
            0.5 * session_rank[session_idx]
            + 0.5 * lexical_rank[i]
        )

        item["memory_cost"] = len(
            reader_tokenizer.encode(
                item["rendered"],
                add_special_tokens=False
            )
        )

        ranked.append(item)

    ranked.sort(
        key=lambda x: (
            -x["combined_score"],
            -x["similarity"],
            x["session"],
            x["turn"],
            x["span"]
        )
    )

    selected = []
    used = 0

    for item in ranked:

        cost = item["memory_cost"]

        if used + cost <= memory_budget:
            selected.append(item)
            used += cost

    return selected, used


# ------------------------------------------------------------
# 2. RE-RUN EXTERNAL DATASET
# ------------------------------------------------------------

results = []
audit_rows = []

start_time = time.time()
completed = 0

total_questions = sum(
    len(x["questions"]) for x in dataset
)

for conversation_idx, item in enumerate(dataset):

    sessions = parse_longmt_sessions(item["sessions"])

    rendered_sessions = [
        render_longmt_session(x)
        for x in sessions
    ]

    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    words = []
    user_words = []
    turns_count = []
    first_person = []
    digit_values = []

    for session in sessions:

        _, w, uw, nt, fpv, dgv = session_stats(
            session,
            "user"
        )

        words.append(w)
        user_words.append(uw)
        turns_count.append(nt)
        first_person.append(fpv)
        digit_values.append(dgv)

    days = np.arange(len(sessions), dtype=float)

    Fbase, _ = build_features(
        V,
        np.asarray(words, dtype=float),
        np.asarray(user_words, dtype=float),
        np.asarray(turns_count, dtype=float),
        np.asarray(first_person, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    # Total memory cost across all sessions.
    session_costs = np.asarray([
        len(reader_tokenizer.encode(
            text,
            add_special_tokens=False
        ))
        for text in rendered_sessions
    ], dtype=int)

    memory_budget = max(
        1,
        int(np.floor(
            session_costs.sum() * MEMORY_FRACTION
        ))
    )

    for question_idx, (question, answer) in enumerate(
        zip(item["questions"], item["answers"])
    ):

        question = str(question).strip()

        if not question:
            continue

        Frole = role_features_for_sessions(
            question,
            sessions
        )

        Ffull = np.column_stack([
            Fbase,
            Frole
        ]).astype(np.float32)

        score_map = {
            "baseline": FROZEN_BASELINE.predict_proba(
                Fbase[:, BASE_IDX]
            )[:, 1],

            "role_aware": FROZEN_ROLE_AWARE.predict_proba(
                Ffull[:, FULL_IDX]
            )[:, 1],

            "recency": Fbase[:, 2]
        }

        candidates = longmt_candidates(sessions)

        lexical_ranked = longmt_rank_spans(
            question,
            candidates
        )

        question_hash = hashlib.sha256(
            f"{conversation_idx}|{question_idx}|{question}".encode()
        ).hexdigest()[:20]

        # Question-only control.
        prompt = longmt_prompt(question, "")

        results.append({
            "conversation_index": conversation_idx,
            "question_index": question_idx,
            "question_hash": question_hash,
            "method": "question_only",
            "prediction": longmt_generate(prompt),
            "input_tokens": longmt_token_count(prompt),
            "packed_sessions": 0,
            "packed_spans": 0,
            "answer": str(answer)
        })

        for method, scores in score_map.items():

            selected_spans, memory_used = (
                select_spans_under_budget(
                    question,
                    lexical_ranked,
                    scores,
                    memory_budget
                )
            )

            # Preserve ranking order for reader packing.
            prompt, packed_spans, input_tokens = longmt_pack(
                question,
                selected_spans
            )

            prediction = longmt_generate(prompt)

            selected_session_ids = sorted(set(
                x["session"] for x in selected_spans
            ))

            results.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": method,
                "prediction": prediction,
                "input_tokens": input_tokens,
                "packed_sessions": len(set(
                    x["session"] for x in packed_spans
                )),
                "packed_spans": len(packed_spans),
                "answer": str(answer)
            })

            audit_rows.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": method,
                "selected_sessions": json.dumps(
                    selected_session_ids
                ),
                "memory_tokens_used": memory_used,
                "memory_token_budget": memory_budget,
                "selected_spans": len(selected_spans),
                "packed_spans": len(packed_spans),
                "reader_input_tokens": input_tokens
            })

        completed += 1

        if completed % 25 == 0:
            print(
                f"Completed {completed}/{total_questions} "
                f"| elapsed {(time.time()-start_time)/60:.1f} min"
            )


# ------------------------------------------------------------
# 3. SCORE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

metric_values = results_df.apply(
    lambda row: _qa_metrics(
        row["prediction"],
        row["answer"]
    ),
    axis=1
)

results_df["exact_match"] = [
    x[0] for x in metric_values
]

results_df["token_f1"] = [
    x[1] for x in metric_values
]

summary = (
    results_df
    .groupby("method")
    .agg(
        n=("exact_match", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_input_tokens=("input_tokens", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_packed_spans=("packed_spans", "mean")
    )
    .reset_index()
)

print("\nCORRECTED SPAN-BUDGET RESULTS")
display(summary)


# ------------------------------------------------------------
# 4. PAIRED CONVERSATION-CLUSTER BOOTSTRAP
# ------------------------------------------------------------

def cluster_bootstrap(
    df,
    metric,
    method_a,
    method_b,
    reps=BOOTSTRAP_REPS,
    seed=SEED
):

    pivot = df.pivot_table(
        index=["conversation_index", "question_index"],
        columns="method",
        values=metric,
        aggfunc="first"
    )

    paired = pivot[[method_a, method_b]].dropna()

    paired["delta"] = (
        paired[method_a] - paired[method_b]
    )

    cluster_means = (
        paired.groupby(level="conversation_index")["delta"]
        .mean()
        .to_numpy()
    )

    rng = np.random.default_rng(seed)

    sampled = rng.integers(
        0,
        len(cluster_means),
        size=(reps, len(cluster_means))
    )

    boot = cluster_means[sampled].mean(axis=1)

    return {
        "metric": metric,
        "contrast": f"{method_a} - {method_b}",
        "delta": float(cluster_means.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
        "conversation_clusters": len(cluster_means),
        "bootstrap_reps": reps
    }


bootstrap_df = pd.DataFrame([
    cluster_bootstrap(results_df, metric, "role_aware", comparator)
    for metric in ["token_f1", "exact_match"]
    for comparator in ["baseline", "recency"]
])

print("\nBOOTSTRAP RESULTS")
display(bootstrap_df)


# ------------------------------------------------------------
# 5. BUDGET AND SELECTION AUDIT
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

print("\nSELECTION AUDIT")
display(
    audit_df.groupby("method").agg(
        questions=("question_hash", "nunique"),
        mean_memory_used=("memory_tokens_used", "mean"),
        mean_memory_budget=("memory_token_budget", "mean"),
        mean_selected_spans=("selected_spans", "mean"),
        mean_packed_spans=("packed_spans", "mean"),
        mean_reader_tokens=("reader_input_tokens", "mean"),
        zero_packed_questions=(
            "packed_spans",
            lambda x: int((x == 0).sum())
        )
    )
)

assert (
    audit_df["memory_tokens_used"]
    <= audit_df["memory_token_budget"]
).all()

assert (
    results_df["input_tokens"] <= MAX_INPUT
).all()


# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

results_df.drop(
    columns=["answer"]
).to_csv(
    OUT / "longmt_span_budget_per_question.csv",
    index=False
)

summary.to_csv(
    OUT / "longmt_span_budget_summary.csv",
    index=False
)

bootstrap_df.to_csv(
    OUT / "longmt_span_budget_bootstrap.csv",
    index=False
)

audit_df.to_csv(
    OUT / "longmt_span_budget_audit.csv",
    index=False
)

print("\nCORRECTED EXPERIMENT COMPLETE")
print("Questions:", results_df["question_hash"].nunique())
print("Rows:", len(results_df))
print("Maximum reader input:", results_df["input_tokens"].max())
print("All memory budgets respected:", bool(
    (audit_df["memory_tokens_used"]
     <= audit_df["memory_token_budget"]).all()
))
print("All reader inputs within 512:", bool(
    (results_df["input_tokens"] <= 512).all()
))
print("Saved to:", OUT)


Completed 25/288 | elapsed 0.7 min
Completed 50/288 | elapsed 1.5 min
Completed 75/288 | elapsed 2.2 min
Completed 100/288 | elapsed 2.8 min
Completed 125/288 | elapsed 3.5 min
Completed 150/288 | elapsed 4.1 min
Completed 175/288 | elapsed 4.8 min
Completed 200/288 | elapsed 5.5 min
Completed 225/288 | elapsed 6.1 min
Completed 250/288 | elapsed 6.8 min
Completed 275/288 | elapsed 7.6 min

CORRECTED SPAN-BUDGET RESULTS


,method,n,exact_match,token_f1,mean_reader_input_tokens,mean_packed_sessions,mean_packed_spans
0,baseline,288,0.000000,0.082333,510.913194,1.656250,24.277778
1,question_only,288,0.000000,0.006017,55.444444,0.000000,0.000000
2,recency,288,0.003472,0.080957,510.420139,2.204861,21.364583
3,role_aware,288,0.003472,0.144299,510.972222,1.343750,20.395833



BOOTSTRAP RESULTS


,metric,contrast,delta,ci_low,ci_high,conversation_clusters,bootstrap_reps
0,token_f1,role_aware - baseline,0.062667,0.044136,0.082037,11,20000
1,token_f1,role_aware - recency,0.063443,0.045635,0.081690,11,20000
2,exact_match,role_aware - baseline,0.003497,0.000000,0.010490,11,20000
3,exact_match,role_aware - recency,0.000000,0.000000,0.000000,11,20000



SELECTION AUDIT


,questions,mean_memory_used,mean_memory_budget,mean_selected_spans,mean_packed_spans,mean_reader_tokens,zero_packed_questions
method,,,,,,,
baseline,288,2184.295139,2185.201389,93.041667,24.277778,510.913194,0
recency,288,2184.295139,2185.201389,83.489583,21.364583,510.420139,0
role_aware,288,2184.385417,2185.201389,89.902778,20.395833,510.972222,0



CORRECTED EXPERIMENT COMPLETE
Questions: 288
Rows: 1152
Maximum reader input: 512
All memory budgets respected: True
All reader inputs within 512: True
Saved to: /kaggle/working/memweave_followup_outputs/long_mt_bench_plus_span_budget_flan_t5_base


In [101]:

# ============================================================
# MEMWEAVE — ROLE-FEATURE ABLATION
# Long-MT-Bench+ | FLAN-T5-base | Span-level memory budget
#
# Tests:
#   1. Full role-aware model
#   2. Remove each role feature individually
#   3. Remove all role features (baseline)
#
# Reuses completed dataset, tokenizers, span helpers,
# frozen model, and generation functions.
# ============================================================

import numpy as np
import pandas as pd
import json
import time
import hashlib
from pathlib import Path

OUT = Path(
    "/kaggle/working/memweave_followup_outputs/"
    "longmt_role_feature_ablation"
)
OUT.mkdir(parents=True, exist_ok=True)

MAX_INPUT = 512
MEMORY_FRACTION = 0.10
BOOTSTRAP_REPS = 20000
SEED = 20261128

# ------------------------------------------------------------
# 1. DEFINE FEATURE VARIANTS
# ------------------------------------------------------------

# Uses the same role feature names and frozen feature indices
# from the completed Long-MT-Bench+ experiment.

print("Role features:", ROLE_FEATURE_NAMES)

ABLATIONS = {
    "full_role_aware": list(range(len(ROLE_FEATURE_NAMES))),
    **{
        f"without_{name}": [
            j for j, feature_name in enumerate(ROLE_FEATURE_NAMES)
            if feature_name != name
        ]
        for name in ROLE_FEATURE_NAMES
    },
    "baseline_no_role": []
}

# ------------------------------------------------------------
# 2. FEATURE-SPECIFIC ROLE-AWARE SCORING
# ------------------------------------------------------------

def score_role_variant(Fbase, Frole, retained_role_indices):

    if len(retained_role_indices) == 0:
        return FROZEN_BASELINE.predict_proba(
            Fbase[:, BASE_IDX]
        )[:, 1]

    selected_columns = [
        len(BASE_IDX) + j
        for j in retained_role_indices
    ]

    # The full model was trained with all role features.
    # Therefore, feature removal must use a model trained
    # on the corresponding reduced feature set.
    model_key = tuple(retained_role_indices)

    if model_key not in ABLATION_MODELS:
        raise RuntimeError(
            f"Missing trained ablation model for {model_key}. "
            "Do not pass zero-filled features to the full model."
        )

    X = np.column_stack([
        Fbase[:, BASE_IDX],
        Frole[:, retained_role_indices]
    ]).astype(np.float32)

    return ABLATION_MODELS[model_key].predict_proba(X)[:, 1]


# ------------------------------------------------------------
# 3. TRAIN ABLATION MODELS
# ------------------------------------------------------------

# IMPORTANT:
# Ablation models must be trained using the original training
# data and the same preprocessing/fold protocol as the frozen
# full role-aware model. Do not fit on Long-MT-Bench+ test data.
#
# If you saved the original training matrices and labels,
# populate ABLATION_MODELS using those artifacts.
#
# Expected:
#   X_train_base: baseline training matrix
#   X_train_role: role-feature training matrix
#   y_train: training labels
#   same scaler/model pipeline and C as original training.

from sklearn.base import clone

ABLATION_MODELS = {}

for retained in ABLATIONS.values():

    key = tuple(retained)

    if key in ABLATION_MODELS:
        continue

    if len(retained) == 0:
        continue

    X_train_ablation = np.column_stack([
        X_train_base[:, BASE_IDX],
        X_train_role[:, retained]
    ]).astype(np.float32)

    model = clone(FROZEN_ROLE_AWARE)

    model.fit(X_train_ablation, y_train)

    ABLATION_MODELS[key] = model

print("Trained ablation models:", len(ABLATION_MODELS))


# ------------------------------------------------------------
# 4. REUSE THE COMPLETED LONG-MT-BENCH+ SAMPLE
# ------------------------------------------------------------

ablation_results = []
ablation_audit = []

start_time = time.time()
completed = 0

for conversation_idx, item in enumerate(dataset):

    sessions = parse_longmt_sessions(item["sessions"])

    rendered_sessions = [
        render_longmt_session(x)
        for x in sessions
    ]

    V = tfidf.transform(
        hv.transform(rendered_sessions)
    ).astype(np.float32).tocsr()

    words = []
    user_words = []
    turns_count = []
    first_person = []
    digit_values = []

    for session in sessions:
        _, w, uw, nt, fpv, dgv = session_stats(
            session, "user"
        )
        words.append(w)
        user_words.append(uw)
        turns_count.append(nt)
        first_person.append(fpv)
        digit_values.append(dgv)

    days = np.arange(len(sessions), dtype=float)

    Fbase, _ = build_features(
        V,
        np.asarray(words, dtype=float),
        np.asarray(user_words, dtype=float),
        np.asarray(turns_count, dtype=float),
        np.asarray(first_person, dtype=float),
        np.asarray(digit_values, dtype=float),
        days,
        None
    )

    Frole_cache = {}

    session_costs = np.asarray([
        len(reader_tokenizer.encode(
            text,
            add_special_tokens=False
        ))
        for text in rendered_sessions
    ])

    memory_budget = max(
        1,
        int(np.floor(
            session_costs.sum() * MEMORY_FRACTION
        ))
    )

    for question_idx, (question, answer) in enumerate(
        zip(item["questions"], item["answers"])
    ):

        question = str(question).strip()

        if not question:
            continue

        Frole = role_features_for_sessions(
            question,
            sessions
        )

        candidates = longmt_rank_spans(
            question,
            longmt_candidates(sessions)
        )

        question_hash = hashlib.sha256(
            f"{conversation_idx}|{question_idx}|{question}".encode()
        ).hexdigest()[:20]

        for variant_name, retained in ABLATIONS.items():

            if len(retained) == 0:
                scores = FROZEN_BASELINE.predict_proba(
                    Fbase[:, BASE_IDX]
                )[:, 1]
            else:
                scores = score_role_variant(
                    Fbase,
                    Frole,
                    retained
                )

            selected_spans, memory_used = (
                select_spans_under_budget(
                    question,
                    candidates,
                    scores,
                    memory_budget
                )
            )

            prompt, packed_spans, input_tokens = longmt_pack(
                question,
                selected_spans
            )

            prediction = longmt_generate(prompt)

            ablation_results.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": variant_name,
                "prediction": prediction,
                "answer": str(answer),
                "input_tokens": input_tokens,
                "packed_sessions": len(set(
                    x["session"] for x in packed_spans
                )),
                "packed_spans": len(packed_spans)
            })

            ablation_audit.append({
                "conversation_index": conversation_idx,
                "question_index": question_idx,
                "question_hash": question_hash,
                "method": variant_name,
                "memory_used": memory_used,
                "memory_budget": memory_budget,
                "selected_spans": len(selected_spans),
                "packed_spans": len(packed_spans),
                "reader_tokens": input_tokens
            })

        completed += 1

        if completed % 25 == 0:
            print(
                f"Completed {completed}/288 "
                f"| elapsed {(time.time()-start_time)/60:.1f} min"
            )


# ------------------------------------------------------------
# 5. EVALUATE
# ------------------------------------------------------------

ablation_df = pd.DataFrame(ablation_results)

metrics = ablation_df.apply(
    lambda row: _qa_metrics(
        row["prediction"],
        row["answer"]
    ),
    axis=1
)

ablation_df["exact_match"] = [x[0] for x in metrics]
ablation_df["token_f1"] = [x[1] for x in metrics]

ablation_summary = (
    ablation_df
    .groupby("method")
    .agg(
        n=("exact_match", "size"),
        exact_match=("exact_match", "mean"),
        token_f1=("token_f1", "mean"),
        mean_reader_tokens=("input_tokens", "mean"),
        mean_packed_sessions=("packed_sessions", "mean"),
        mean_packed_spans=("packed_spans", "mean")
    )
    .reset_index()
)

print("\nROLE FEATURE ABLATION RESULTS")
display(
    ablation_summary.sort_values(
        "token_f1",
        ascending=False
    )
)


# ------------------------------------------------------------
# 6. PAIRED CLUSTER BOOTSTRAP
# ------------------------------------------------------------

def ablation_bootstrap(
    df,
    variant,
    reference="full_role_aware",
    metric="token_f1",
    reps=20000,
    seed=SEED
):

    pivot = df.pivot_table(
        index=["conversation_index", "question_index"],
        columns="method",
        values=metric
    )

    paired = pivot[[variant, reference]].dropna()
    paired["delta"] = paired[variant] - paired[reference]

    cluster_means = (
        paired.groupby(level="conversation_index")["delta"]
        .mean()
        .to_numpy()
    )

    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        len(cluster_means),
        size=(reps, len(cluster_means))
    )

    boot = cluster_means[indices].mean(axis=1)

    return {
        "variant": variant,
        "reference": reference,
        "metric": metric,
        "delta": float(cluster_means.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
        "clusters": len(cluster_means)
    }


bootstrap_rows = []

for variant in ABLATIONS:

    if variant == "full_role_aware":
        continue

    bootstrap_rows.append(
        ablation_bootstrap(
            ablation_df,
            variant=variant,
            reference="full_role_aware",
            metric="token_f1"
        )
    )

ablation_bootstrap_df = pd.DataFrame(bootstrap_rows)

print("\nABLATION DELTAS VS FULL ROLE-AWARE")
display(ablation_bootstrap_df)


# ------------------------------------------------------------
# 7. AUDIT AND SAVE
# ------------------------------------------------------------

audit_df = pd.DataFrame(ablation_audit)

assert (audit_df["memory_used"] <= audit_df["memory_budget"]).all()
assert (ablation_df["input_tokens"] <= MAX_INPUT).all()

ablation_df.drop(
    columns=["answer"]
).to_csv(
    OUT / "role_ablation_per_question.csv",
    index=False
)

ablation_summary.to_csv(
    OUT / "role_ablation_summary.csv",
    index=False
)

ablation_bootstrap_df.to_csv(
    OUT / "role_ablation_bootstrap.csv",
    index=False
)

audit_df.to_csv(
    OUT / "role_ablation_budget_audit.csv",
    index=False
)

print("\nABLATION COMPLETE")
print("Rows:", len(ablation_df))
print("Variants:", ablation_df["method"].nunique())
print("Conversation clusters:", ablation_df["conversation_index"].nunique())
print("All budgets respected:", bool(
    (audit_df["memory_used"] <= audit_df["memory_budget"]).all()
))
print("All inputs within 512:", bool(
    (ablation_df["input_tokens"] <= 512).all()
))
print("Saved:", OUT)


Role features: ['user_max_question_similarity', 'user_top3_mean_question_similarity', 'assistant_max_question_similarity', 'assistant_top3_mean_question_similarity', 'log_user_message_count', 'log_assistant_message_count']


NameError: name 'X_train_base' is not defined

In [102]:
# MEMWEAVE — inspect existing training objects for a valid role-feature ablation

import numpy as np

required_names = [
    "F_aug",
    "F",
    "BASE_IDX",
    "ROLE_IDX",
    "FULL_IDX",
    "ROLE_FEATURE_NAMES",
    "POOL",
    "VARIANTS",
    "fit_variant",
    "stack_aug",
    "evaluate_scores",
    "select_budgeted",
    "C_FROZEN",
]

for name in required_names:
    exists = name in globals()
    value = globals().get(name)
    if isinstance(value, np.ndarray):
        detail = f"shape={value.shape}, dtype={value.dtype}"
    elif hasattr(value, "shape"):
        detail = f"shape={value.shape}"
    else:
        detail = repr(value)[:180] if exists else "NOT FOUND"
    print(f"{name}: {'FOUND' if exists else 'MISSING'} | {detail}")

print("\nPossible existing training matrices:")
for name, value in list(globals().items()):
    if name.startswith(("X_train", "y_train", "F_train", "train_")):
        shape = getattr(value, "shape", None)
        print(f"{name}: {shape if shape is not None else type(value).__name__}")

F_aug: MISSING | NOT FOUND
F: FOUND | shape=(32, 10), dtype=float32
BASE_IDX: FOUND | [0, 1, 7, 8, 9]
ROLE_IDX: FOUND | [10, 11, 12, 13, 14, 15]
FULL_IDX: FOUND | [0, 1, 7, 8, 9, 10, 11, 12, 13, 14, 15]
ROLE_FEATURE_NAMES: FOUND | ['user_max_question_similarity', 'user_top3_mean_question_similarity', 'assistant_max_question_similarity', 'assistant_top3_mean_question_similarity', 'log_user_message_count', 'lo
POOL: FOUND | 'same_pipeline'
VARIANTS: FOUND | {'baseline': [0, 1, 7, 8, 9], 'full_role_aware': [0, 1, 7, 8, 9, 10, 11, 12, 13, 14, 15], 'leave_out__user_max_question_similarity': [0, 1, 7, 8, 9, 11, 12, 13, 14, 15], 'leave_out
fit_variant: FOUND | <function fit_variant at 0x7aff32f04ea0>
stack_aug: FOUND | <function stack_aug at 0x7aff32f079c0>
evaluate_scores: FOUND | <function evaluate_scores at 0x7aff32f04220>
select_budgeted: FOUND | <function select_budgeted at 0x7aff32f07e20>
C_FROZEN: FOUND | 0.3

Possible existing training matrices:
train_test_split: function
train_and_sc

In [103]:
# MEMWEAVE — inspect the original fold-safe training pipeline

import inspect

objects_to_inspect = [
    "fit_variant",
    "stack_aug",
    "evaluate_scores",
    "select_budgeted",
    "train_baseline",
    "train_and_score",
    "train_test_split",
]

for name in objects_to_inspect:
    obj = globals().get(name)
    print(f"\n{'=' * 20} {name} {'=' * 20}")

    if obj is None:
        print("MISSING")
        continue

    try:
        print("Signature:", inspect.signature(obj))
        print(inspect.getsource(obj)[:5000])
    except (TypeError, OSError) as e:
        print("Source unavailable:", e)

print("\nTRAINING DATA")
for name in ["train_questions", "train_x", "train_y", "train_idx",
             "train_groups", "y_train", "F"]:
    obj = globals().get(name)
    if obj is None:
        print(name, ": MISSING")
    else:
        print(name, ": type =", type(obj).__name__,
              "| shape =", getattr(obj, "shape", None),
              "| length =", len(obj) if hasattr(obj, "__len__") else "N/A")

print("\nFEATURE VARIANTS")
for name, indices in VARIANTS.items():
    print(name, "->", indices)


==================== fit_variant ====================
Signature: (feature_idx)
Source unavailable: source code not available

==================== stack_aug ====================
Signature: (qis, feature_idx)
Source unavailable: source code not available

==================== evaluate_scores ====================
Signature: (score_map)
Source unavailable: source code not available

==================== select_budgeted ====================
Signature: (scores, cost, budget=0.1)
Source unavailable: source code not available

==================== train_baseline ====================
Signature: ()
def train_baseline():
    xs, ys, hashes = [], [], []
    for qi in DEV:
        xs.append(BASE_MATRICES[qi])
        ys.append(np.asarray(Q[POOL][qi]["gold"], dtype=int))
        hashes.append(np.asarray(Q[POOL][qi]["h"], dtype=np.int64))
    x_train, y_train, h_train = np.vstack(xs), np.concatenate(ys), np.concatenate(hashes)
    keep = ~np.isin(h_train, np.fromiter(TEST_HASHES, dtype=np.int64))
 

In [105]:
# MEMWEAVE — inspect the original role-feature construction inputs

import inspect

print("POOL:", POOL)
print("Q entry keys:")
print(list(Q[POOL][DEV[0]].keys()))

print("\nOriginal feature constructor:")
print(inspect.signature(construct_role_features_lme))

print("\nAvailable role-feature objects:")
for name, obj in globals().copy().items():
    if any(term in name.lower() for term in [
        "role", "session", "message", "feature_map", "aug"
    ]):
        if not name.startswith("_"):
            print(
                name,
                "| type:", type(obj).__name__,
                "| shape:", getattr(obj, "shape", None)
            )

print("\nSample question record:")
print({
    k: (type(v).__name__, getattr(v, "shape", None))
    for k, v in Q[POOL][DEV[0]].items()
})

POOL: same_pipeline
Q entry keys:
['h', 'F', 'S', 'gold', 'cost', 'F_aug']

Original feature constructor:
(qi)

Available role-feature objects:
session_stats | type: function | shape: None
ROLEFREE_IDX | type: list | shape: None
ROLE_IDX | type: list | shape: None
ROLE_FEATURE_NAMES | type: list | shape: None
construct_role_features | type: function | shape: None
ROLE_FEATURES | type: dict | shape: None
feature_map | type: dict | shape: None
sessions | type: list | shape: None
session_id | type: str | shape: None
session | type: list | shape: None
canonical_session | type: str | shape: None
sessions_audited | type: int | shape: None
session_ids | type: set | shape: None
message | type: str | shape: None
raw_role | type: str | shape: None
normalized_role | type: str | shape: None
session_owner | type: dict | shape: None
session_questions | type: defaultdict | shape: None
validation_sessions | type: set | shape: None
answer_sessions | type: dict | shape: None
session_counts | type: Count

In [106]:
# MEMWEAVE — corrected role-feature ablation

import numpy as np
import pandas as pd

assert all("F_aug" in Q[POOL][qi] for qi in list(DEV) + list(TEST))

ablation_results = []

variants = {
    "baseline": [],
    "full_role_aware": list(range(len(ROLE_FEATURE_NAMES))),
}

for j, feature in enumerate(ROLE_FEATURE_NAMES):
    variants[f"leave_out__{feature}"] = [
        k for k in range(len(ROLE_FEATURE_NAMES)) if k != j
    ]

for variant, retained in variants.items():

    if variant == "baseline":
        _, scores, n_train, n_total = train_baseline()
    else:
        feature_map = {
            qi: Q[POOL][qi]["F_aug"][:, BASE_IDX + [
                ROLE_IDX[k] - 10 for k in retained
            ]]
            for qi in list(DEV) + list(TEST)
        }

        _, scores, n_train, n_total = train_and_score(feature_map)

    metrics = evaluate_scores(scores)

    row = {
        "variant": variant,
        "retained_role_features": len(retained),
        "training_candidates_used": n_train,
        "training_candidates_total": n_total,
    }

    if isinstance(metrics, dict):
        row.update(metrics)

    ablation_results.append(row)

ablation_df = pd.DataFrame(ablation_results)

display(ablation_df)

ablation_df.to_csv(
    "memweave_role_feature_ablation.csv",
    index=False
)

print("Saved memweave_role_feature_ablation.csv")

,variant,retained_role_features,training_candidates_used,training_candidates_total,0,1,2,5,10,17,...,479,480,483,484,487,490,493,495,496,497
0,baseline,0,4216,5523,"{'rank': 6, 'recall': 0.0, 'utilization': 0.77...","{'rank': 12, 'recall': 0.0, 'utilization': 0.8...","{'rank': 10, 'recall': 0.0, 'utilization': 0.7...","{'rank': 20, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 0.0, 'utilization': 0.73...","{'rank': 15, 'recall': 0.0, 'utilization': 0.9...",...,"{'rank': 23, 'recall': 0.0, 'utilization': 0.7...","{'rank': 1, 'recall': 1.0, 'utilization': 0.92...","{'rank': 4, 'recall': 1.0, 'utilization': 0.71...","{'rank': 15, 'recall': 0.0, 'utilization': 0.8...","{'rank': 20, 'recall': 0.0, 'utilization': 0.9...","{'rank': 24, 'recall': 0.0, 'utilization': 0.7...","{'rank': 2, 'recall': 1.0, 'utilization': 0.81...","{'rank': 23, 'recall': 0.0, 'utilization': 0.9...","{'rank': 1, 'recall': 1.0, 'utilization': 0.96...","{'rank': 4, 'recall': 0.0, 'utilization': 0.87..."
1,full_role_aware,6,4216,5523,"{'rank': 7, 'recall': 0.0, 'utilization': 0.77...","{'rank': 11, 'recall': 0.0, 'utilization': 0.8...","{'rank': 9, 'recall': 0.0, 'utilization': 0.70...","{'rank': 21, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 0.0, 'utilization': 0.73...","{'rank': 17, 'recall': 0.0, 'utilization': 0.9...",...,"{'rank': 22, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 1.0, 'utilization': 0.95...","{'rank': 2, 'recall': 1.0, 'utilization': 0.71...","{'rank': 14, 'recall': 0.0, 'utilization': 0.8...","{'rank': 19, 'recall': 0.0, 'utilization': 0.9...","{'rank': 24, 'recall': 0.0, 'utilization': 0.7...","{'rank': 1, 'recall': 1.0, 'utilization': 0.88...","{'rank': 23, 'recall': 0.0, 'utilization': 0.9...","{'rank': 1, 'recall': 1.0, 'utilization': 0.96...","{'rank': 4, 'recall': 0.0, 'utilization': 0.87..."
2,leave_out__user_max_question_similarity,5,4216,5523,"{'rank': 7, 'recall': 0.0, 'utilization': 0.77...","{'rank': 11, 'recall': 0.0, 'utilization': 0.8...","{'rank': 9, 'recall': 0.0, 'utilization': 0.70...","{'rank': 21, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 0.0, 'utilization': 0.73...","{'rank': 17, 'recall': 0.0, 'utilization': 0.9...",...,"{'rank': 22, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 1.0, 'utilization': 0.95...","{'rank': 2, 'recall': 1.0, 'utilization': 0.71...","{'rank': 14, 'recall': 0.0, 'utilization': 0.8...","{'rank': 19, 'recall': 0.0, 'utilization': 0.9...","{'rank': 24, 'recall': 0.0, 'utilization': 0.7...","{'rank': 1, 'recall': 1.0, 'utilization': 0.88...","{'rank': 23, 'recall': 0.0, 'utilization': 0.9...","{'rank': 1, 'recall': 1.0, 'utilization': 0.96...","{'rank': 4, 'recall': 0.0, 'utilization': 0.87..."
3,leave_out__user_top3_mean_question_similarity,5,4216,5523,"{'rank': 7, 'recall': 0.0, 'utilization': 0.77...","{'rank': 11, 'recall': 0.0, 'utilization': 0.8...","{'rank': 9, 'recall': 0.0, 'utilization': 0.70...","{'rank': 21, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 0.0, 'utilization': 0.73...","{'rank': 17, 'recall': 0.0, 'utilization': 0.9...",...,"{'rank': 22, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 1.0, 'utilization': 0.95...","{'rank': 2, 'recall': 1.0, 'utilization': 0.71...","{'rank': 14, 'recall': 0.0, 'utilization': 0.8...","{'rank': 19, 'recall': 0.0, 'utilization': 0.9...","{'rank': 24, 'recall': 0.0, 'utilization': 0.7...","{'rank': 1, 'recall': 1.0, 'utilization': 0.88...","{'rank': 23, 'recall': 0.0, 'utilization': 0.9...","{'rank': 1, 'recall': 1.0, 'utilization': 0.96...","{'rank': 4, 'recall': 0.0, 'utilization': 0.87..."
4,leave_out__assistant_max_question_similarity,5,4216,5523,"{'rank': 7, 'recall': 0.0, 'utilization': 0.77...","{'rank': 11, 'recall': 0.0, 'utilization': 0.8...","{'rank': 9, 'recall': 0.0, 'utilization': 0.70...","{'rank': 21, 'recall': 0.0, 'utilization': 0.8...","{'rank': 2, 'recall': 0.0, 'utilization': 0.73...","{'rank': 15, 'recall': 0.0, 

Saved memweave_role_feature_ablation.csv


In [107]:
# MEMWEAVE — aggregate question-level ablation metrics

import pandas as pd
import numpy as np

metric_rows = []

for _, row in ablation_df.iterrows():
    variant = row["variant"]

    question_results = [
        value for value in row.values
        if isinstance(value, dict)
        and {"rank", "recall", "utilization"}.issubset(value.keys())
    ]

    if not question_results:
        continue

    df = pd.DataFrame(question_results)

    metric_rows.append({
        "variant": variant,
        "n_questions": len(df),
        "mean_recall": df["recall"].mean(),
        "mean_rank": df["rank"].mean(),
        "mean_utilization": df["utilization"].mean(),
        "median_rank": df["rank"].median(),
    })

ablation_summary = pd.DataFrame(metric_rows)

baseline_recall = ablation_summary.loc[
    ablation_summary["variant"] == "baseline", "mean_recall"
].iloc[0]

full_recall = ablation_summary.loc[
    ablation_summary["variant"] == "full_role_aware", "mean_recall"
].iloc[0]

ablation_summary["recall_delta_vs_full"] = (
    ablation_summary["mean_recall"] - full_recall
)

display(
    ablation_summary.sort_values(
        "mean_recall", ascending=False
    ).reset_index(drop=True)
)

ablation_summary.to_csv(
    "memweave_role_feature_ablation_summary.csv",
    index=False
)

print("Baseline recall:", baseline_recall)
print("Full role-aware recall:", full_recall)
print("Saved aggregated ablation summary.")

,variant,n_questions,mean_recall,mean_rank,mean_utilization,median_rank,recall_delta_vs_full
0,baseline,235,0.404184,5.855319,0.872741,2.0,0.029362
1,leave_out__log_assistant_message_count,235,0.401206,5.689362,0.874742,2.0,0.026383
2,leave_out__log_user_message_count,235,0.389716,5.668085,0.871669,2.0,0.014894
3,leave_out__assistant_top3_mean_question_simila...,235,0.381915,5.723404,0.873161,2.0,0.007092
4,leave_out__user_top3_mean_question_similarity,235,0.374823,5.642553,0.873155,2.0,0.000000
5,leave_out__user_max_question_similarity,235,0.374823,5.642553,0.873155,2.0,0.000000
6,full_role_aware,235,0.374823,5.642553,0.873155,2.0,0.000000
7,leave_out__assistant_max_question_similarity,235,0.374113,5.646809,0.872560,2.0,-0.000709


Baseline recall: 0.40418439716312055
Full role-aware recall: 0.37482269503546095
Saved aggregated ablation summary.


In [108]:
# MEMWEAVE — corrected role-feature ablation (actual columns 10–15)

import numpy as np
import pandas as pd

assert list(ROLE_IDX) == [10, 11, 12, 13, 14, 15]
assert len(ROLE_FEATURE_NAMES) == 6

all_qis = list(DEV) + list(TEST)

role_feature_map = {
    qi: Q[POOL][qi]["F_aug"][:, ROLE_IDX].astype(np.float32)
    for qi in all_qis
}

# Validate that the selected columns really are the six role features.
for qi in all_qis:
    assert role_feature_map[qi].shape == (
        len(Q[POOL][qi]["gold"]), 6
    )

variants = {
    "baseline": [],
    "full_role_aware": list(range(6)),
}

for j, feature in enumerate(ROLE_FEATURE_NAMES):
    variants[f"leave_out__{feature}"] = [
        k for k in range(6) if k != j
    ]

results = []

for variant, retained in variants.items():

    if variant == "baseline":
        model, scores, n_train, n_total = train_baseline()
    else:
        feature_map = {
            qi: role_feature_map[qi][:, retained]
            for qi in all_qis
        }
        model, scores, n_train, n_total = train_and_score(feature_map)

    question_metrics = evaluate_scores(scores)

    metric_df = pd.DataFrame.from_dict(
        question_metrics, orient="index"
    )

    results.append({
        "variant": variant,
        "retained_role_features": len(retained),
        "training_candidates_used": n_train,
        "training_candidates_total": n_total,
        "n_questions": len(metric_df),
        "mean_recall": metric_df["recall"].mean(),
        "mean_rank": metric_df["rank"].mean(),
        "mean_utilization": metric_df["utilization"].mean(),
        "median_rank": metric_df["rank"].median(),
    })

corrected_ablation = pd.DataFrame(results)

full_recall = corrected_ablation.loc[
    corrected_ablation.variant == "full_role_aware",
    "mean_recall"
].iloc[0]

corrected_ablation["recall_delta_vs_full"] = (
    corrected_ablation["mean_recall"] - full_recall
)

display(
    corrected_ablation.sort_values(
        "mean_recall", ascending=False
    ).reset_index(drop=True)
)

corrected_ablation.to_csv(
    "memweave_role_feature_ablation_corrected.csv",
    index=False
)

print("Corrected ablation saved.")

,variant,retained_role_features,training_candidates_used,training_candidates_total,n_questions,mean_recall,mean_rank,mean_utilization,median_rank,recall_delta_vs_full
0,leave_out__user_top3_mean_question_similarity,5,4216,5523,235,0.827021,1.302128,0.896714,1.0,0.006383
1,leave_out__assistant_top3_mean_question_simila...,5,4216,5523,235,0.827021,1.314894,0.898535,1.0,0.006383
2,leave_out__assistant_max_question_similarity,5,4216,5523,235,0.822057,1.314894,0.897832,1.0,0.001418
3,full_role_aware,6,4216,5523,235,0.820638,1.306383,0.899461,1.0,0.000000
4,leave_out__log_user_message_count,5,4216,5523,235,0.820638,1.306383,0.899461,1.0,0.000000
5,leave_out__log_assistant_message_count,5,4216,5523,235,0.820638,1.306383,0.899461,1.0,0.000000
6,leave_out__user_max_question_similarity,5,4216,5523,235,0.819574,1.319149,0.897026,1.0,-0.001064
7,baseline,0,4216,5523,235,0.404184,5.855319,0.872741,2.0,-0.416454


Corrected ablation saved.


In [109]:
# Inspect the actual evaluation output structure — no model retraining

sample_scores = {
    qi: np.zeros(len(Q[POOL][qi]["gold"]), dtype=np.float32)
    for qi in TEST
}

sample_eval = evaluate_scores(sample_scores)

print("Evaluation return type:", type(sample_eval))
print("Number of test questions:", len(TEST))
print("Number of score entries:", len(sample_scores))

if isinstance(sample_eval, dict):
    print("Number of returned keys:", len(sample_eval))
    for key in list(sample_eval.keys())[:3]:
        print("Key:", key)
        print("Value:", repr(sample_eval[key])[:500])

elif isinstance(sample_eval, pd.DataFrame):
    print("Shape:", sample_eval.shape)
    display(sample_eval.head())

elif isinstance(sample_eval, (list, tuple)):
    print("Length:", len(sample_eval))
    print("First item:", repr(sample_eval[0])[:500])

print("\nOne actual question's score shape:")
qi = TEST[0]
print("Question ID:", qi)
print("Candidates:", len(Q[POOL][qi]["gold"]))
print("Gold positives:", int(np.sum(Q[POOL][qi]["gold"])))

Evaluation return type: <class 'dict'>
Number of test questions: 235
Number of score entries: 235
Number of returned keys: 235
Key: 0
Value: {'rank': 23, 'recall': 0.0, 'utilization': 0.8372582920964248}
Key: 1
Value: {'rank': 4, 'recall': 0.0, 'utilization': 0.8209024196380709}
Key: 2
Value: {'rank': 20, 'recall': 0.0, 'utilization': 0.9257571696596086}

One actual question's score shape:
Question ID: 0
Candidates: 24
Gold positives: 1


In [110]:

from pathlib import Path
import zipfile
import hashlib
import json
from datetime import datetime

# ============================================================
# MEMWEAVE: AUTOMATIC OUTPUT ZIP EXPORT
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/memweave_followup_outputs")
EXPORT_DIR = Path("/kaggle/working")
ZIP_PATH = EXPORT_DIR / "MemWeave_Research_Artifacts.zip"

# Include the main experiment output directory.
# Also include any corrected ablation CSV saved elsewhere
# under /kaggle/working.
files_to_add = []

if OUTPUT_DIR.exists():
    files_to_add.extend(
        p for p in OUTPUT_DIR.rglob("*")
        if p.is_file()
    )
else:
    raise FileNotFoundError(
        f"Output directory not found: {OUTPUT_DIR}"
    )

# Include corrected ablation exports if present.
for pattern in [
    "*ablation*corrected*.csv",
    "memweave_role_feature_ablation_corrected.csv"
]:
    files_to_add.extend(
        p for p in EXPORT_DIR.rglob(pattern)
        if p.is_file() and ZIP_PATH not in p.parents
    )

# Remove duplicate paths.
files_to_add = sorted(set(files_to_add))

if not files_to_add:
    raise RuntimeError("No experiment output files found.")

# Create archive and checksum inventory.
manifest = {
    "project": "MemWeave",
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "file_count": len(files_to_add),
    "files": []
}

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:

    for file_path in files_to_add:
        relative_path = file_path.relative_to(EXPORT_DIR)

        digest = hashlib.sha256(
            file_path.read_bytes()
        ).hexdigest()

        archive.write(file_path, arcname=str(relative_path))

        manifest["files"].append({
            "path": str(relative_path),
            "size_bytes": file_path.stat().st_size,
            "sha256": digest
        })

    archive.writestr(
        "SHA256_MANIFEST.json",
        json.dumps(manifest, indent=2)
    )

# Verify ZIP integrity.
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()

if bad_file:
    raise RuntimeError(f"ZIP verification failed: {bad_file}")

zip_hash = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()

print("=" * 55)
print("MEMWEAVE EXPORT COMPLETE")
print("=" * 55)
print(f"Files archived: {len(files_to_add)}")
print(f"ZIP location: {ZIP_PATH}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"ZIP SHA-256: {zip_hash}")
print("ZIP integrity: PASSED")
print("=" * 55)


MEMWEAVE EXPORT COMPLETE
Files archived: 52
ZIP location: /kaggle/working/MemWeave_Research_Artifacts.zip
ZIP size: 1.51 MB
ZIP SHA-256: 32bc3f886f569a8fe788ca62b37032fe51d44178184b432539a7b224b7575634
ZIP integrity: PASSED
